In [1]:
# ======================================================================
# STEP 1 — ENVIRONMENT / DATASET DISCOVERY
# ======================================================================

from pathlib import Path
import sys
import platform
import importlib.util
import hashlib

print("=" * 70)
print("STEP 1 — ENVIRONMENT / DATASET DISCOVERY")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Python environment
# ----------------------------------------------------------------------

print("\n[1] Python environment")
print("-" * 70)

print(f"Python version : {sys.version}")
print(f"Platform       : {platform.platform()}")
print(f"Python path    : {sys.executable}")

# ----------------------------------------------------------------------
# 2. Check important packages
# ----------------------------------------------------------------------

print("\n[2] Required / relevant package availability")
print("-" * 70)

packages_to_check = [
    "pandas",
    "numpy",
    "sklearn",
    "imblearn",
    "lightgbm",
    "torch",
    "rtdl_revisiting_models",
    "shap",
]

package_status = {}

for package_name in packages_to_check:
    available = importlib.util.find_spec(package_name) is not None
    package_status[package_name] = available
    status = "AVAILABLE" if available else "NOT FOUND"
    print(f"{package_name:<28} : {status}")

# ----------------------------------------------------------------------
# 3. Print versions where available
# ----------------------------------------------------------------------

print("\n[3] Package versions")
print("-" * 70)

def get_version(package_name):
    try:
        module = __import__(package_name)
        return getattr(module, "__version__", "version attribute unavailable")
    except Exception as e:
        return f"could not import ({type(e).__name__})"

version_name_map = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "imblearn": "imbalanced-learn",
    "lightgbm": "lightgbm",
    "torch": "torch",
    "rtdl_revisiting_models": "rtdl_revisiting_models",
    "shap": "shap",
}

for import_name, display_name in version_name_map.items():
    if package_status.get(import_name, False):
        print(f"{display_name:<28} : {get_version(import_name)}")
    else:
        print(f"{display_name:<28} : NOT INSTALLED")

# ----------------------------------------------------------------------
# 4. Current working directory
# ----------------------------------------------------------------------

print("\n[4] Notebook working directory")
print("-" * 70)

cwd = Path.cwd()
print(f"Current directory: {cwd}")

# ----------------------------------------------------------------------
# 5. Search for candidate dataset files
# ----------------------------------------------------------------------

print("\n[5] Dataset/file discovery")
print("-" * 70)

# Search the current directory and a few common data directories.
search_roots = [
    cwd,
    cwd / "data",
    cwd / "dataset",
    cwd / "datasets",
    cwd / "Data",
    cwd / "Dataset",
    Path("/mnt/data"),
]

# Keep only existing directories and remove duplicates.
existing_roots = []
seen_roots = set()

for root in search_roots:
    try:
        resolved = root.resolve()
    except Exception:
        continue

    if resolved.exists() and resolved.is_dir() and resolved not in seen_roots:
        existing_roots.append(resolved)
        seen_roots.add(resolved)

supported_extensions = {
    ".csv",
    ".xlsx",
    ".xls",
    ".parquet",
    ".pq",
}

candidate_files = []

for root in existing_roots:
    try:
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in supported_extensions:
                candidate_files.append(path.resolve())
    except Exception as e:
        print(f"Could not fully scan {root}: {e}")

# Remove duplicates while preserving order.
candidate_files = list(dict.fromkeys(candidate_files))

if not candidate_files:
    print("No CSV / Excel / Parquet files were discovered.")
else:
    print(f"Discovered {len(candidate_files)} candidate data file(s):\n")

    for i, path in enumerate(candidate_files, start=1):
        try:
            size_mb = path.stat().st_size / (1024 ** 2)
        except Exception:
            size_mb = float("nan")

        print(f"{i:>3}. {path}")
        print(f"     Extension : {path.suffix.lower()}")
        print(f"     Size      : {size_mb:.3f} MB")

# ----------------------------------------------------------------------
# 6. Create immutable file inventory
# ----------------------------------------------------------------------

print("\n[6] File inventory")
print("-" * 70)

file_inventory = []

for path in candidate_files:
    try:
        stat = path.stat()

        # SHA-256 gives us a way to verify later that the original file
        # itself was not silently modified.
        sha256 = hashlib.sha256()

        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                sha256.update(chunk)

        file_inventory.append({
            "path": str(path),
            "name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "sha256": sha256.hexdigest(),
        })

    except Exception as e:
        print(f"Could not inventory {path}: {e}")

if file_inventory:
    for item in file_inventory:
        print(f"\nFile: {item['name']}")
        print(f"Path: {item['path']}")
        print(f"Size: {item['size_bytes']:,} bytes")
        print(f"SHA256: {item['sha256']}")
else:
    print("No files available for inventory.")

# ----------------------------------------------------------------------
# 7. Basic environment assertions
# ----------------------------------------------------------------------

print("\n[7] Assertions")
print("-" * 70)

assert isinstance(cwd, Path), "Working directory was not resolved correctly."
assert cwd.exists(), "Current working directory does not exist."

assert isinstance(candidate_files, list), \
    "candidate_files must be a list."

assert isinstance(file_inventory, list), \
    "file_inventory must be a list."

print("✓ Working directory assertion passed")
print("✓ Candidate-file inventory assertion passed")
print("✓ File-inventory assertion passed")

print("\n" + "=" * 70)
print("STEP 1 DISCOVERY COMPLETE")
print("=" * 70)

print("\nIMPORTANT:")
print("No dataset was loaded or modified in this step.")
print("No train/validation/test split was performed.")
print("No preprocessing was performed.")
print("No SMOTENC was performed.")
print("No model was created.")

STEP 1 — ENVIRONMENT / DATASET DISCOVERY

[1] Python environment
----------------------------------------------------------------------
Python version : 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform       : Windows-11-10.0.26100-SP0
Python path    : c:\ProgramData\anaconda3\python.exe

[2] Required / relevant package availability
----------------------------------------------------------------------
pandas                       : AVAILABLE
numpy                        : AVAILABLE
sklearn                      : AVAILABLE
imblearn                     : AVAILABLE
lightgbm                     : AVAILABLE
torch                        : AVAILABLE
rtdl_revisiting_models       : AVAILABLE
shap                         : AVAILABLE

[3] Package versions
----------------------------------------------------------------------
pandas                       : 2.2.3
numpy                        : 2.1.3
scikit-learn                 : 1.6.1
imbala

In [3]:
# ======================================================================
# STEP 2 — RAW DATA SCHEMA INSPECTION
# ======================================================================

from pathlib import Path
import pandas as pd
import csv

print("=" * 70)
print("STEP 2 — RAW DATA SCHEMA INSPECTION")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Required-object verification
# ----------------------------------------------------------------------

print("\n[0] Required-object verification")
print("-" * 70)

assert "candidate_files" in globals(), (
    "Missing object: candidate_files. "
    "This should have been created by STEP 1. "
    "Rerun STEP 1 before continuing."
)

assert "file_inventory" in globals(), (
    "Missing object: file_inventory. "
    "This should have been created by STEP 1. "
    "Rerun STEP 1 before continuing."
)

assert isinstance(candidate_files, list), "candidate_files must be a list."
assert isinstance(file_inventory, list), "file_inventory must be a list."

print("✓ candidate_files exists")
print("✓ file_inventory exists")

# ----------------------------------------------------------------------
# 1. Build a clean file lookup
# ----------------------------------------------------------------------

print("\n[1] Building file lookup")
print("-" * 70)

file_lookup = {
    Path(path).name: Path(path)
    for path in candidate_files
}

assert len(file_lookup) == len(candidate_files), (
    "Duplicate filenames detected. "
    "File selection must be resolved before proceeding."
)

print(f"✓ {len(file_lookup)} unique dataset filenames available")

# ----------------------------------------------------------------------
# 2. Select files for schema inspection
# ----------------------------------------------------------------------

files_to_inspect = [
    "microbiology_cultures_cohort.csv",
    "microbiology_cultures_microbial_resistance.csv",
    "microbiology_cultures_demographics.csv",
    "microbiology_cultures_labs.csv",
    "microbiology_cultures_ward_info.csv",
    "microbiology_cultures_implied_susceptibility.csv",
    "microbiology_cultures_prior_infecting_organism.csv",
    "microbiology_cultures_adi_scores.csv",
    "microbiology_cultures_nursing_home_visits.csv",
]

print("\n[2] Selected files")
print("-" * 70)

for filename in files_to_inspect:
    if filename in file_lookup:
        print(f"✓ {filename}")
    else:
        print(f"⚠ NOT FOUND: {filename}")

# Do not silently continue if one of the core files is absent.
core_files = [
    "microbiology_cultures_cohort.csv",
    "microbiology_cultures_microbial_resistance.csv",
]

for filename in core_files:
    assert filename in file_lookup, (
        f"Required core file not found: {filename}"
    )

# ----------------------------------------------------------------------
# 3. Safe CSV header reader
# ----------------------------------------------------------------------

def read_csv_header(path):
    """
    Read only the CSV header.

    This does NOT load the dataset into memory.
    """
    path = Path(path)

    with path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
        errors="replace"
    ) as f:

        reader = csv.reader(f)
        header = next(reader)

    return header


# ----------------------------------------------------------------------
# 4. Schema inspection
# ----------------------------------------------------------------------

print("\n[3] RAW CSV SCHEMAS")
print("-" * 70)

raw_schema = {}

for filename in files_to_inspect:

    if filename not in file_lookup:
        continue

    path = file_lookup[filename]

    print(f"\n{'=' * 70}")
    print(filename)
    print(f"{'=' * 70}")

    header = read_csv_header(path)

    assert isinstance(header, list), (
        f"Header for {filename} was not read as a list."
    )

    assert len(header) > 0, (
        f"{filename} contains no columns."
    )

    assert len(header) == len(set(header)), (
        f"{filename} contains duplicate column names."
    )

    raw_schema[filename] = header

    print(f"Column count: {len(header)}")

    for i, column in enumerate(header, start=1):
        print(f"{i:>4}. {column}")

# ----------------------------------------------------------------------
# 5. Search specifically for important AMR-related fields
# ----------------------------------------------------------------------

print("\n[4] TARGET / FEATURE / LEAKAGE CANDIDATE SEARCH")
print("-" * 70)

search_terms = [
    "was_positive",
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
    "resistance",
    "resistant",
    "suscept",
    "sensitivity",
    "result",
    "outcome",
    "target",
    "positive",
    "negative",
    "culture",
    "specimen",
    "patient",
    "encounter",
    "order",
    "time",
    "date",
]

important_columns = {}

for filename, header in raw_schema.items():

    matches = []

    for column in header:
        column_lower = column.lower()

        if any(term in column_lower for term in search_terms):
            matches.append(column)

    important_columns[filename] = matches

    print(f"\n{filename}")

    if matches:
        for column in matches:
            print(f"  → {column}")
    else:
        print("  → No matching candidate columns")

# ----------------------------------------------------------------------
# 6. Explicit required-feature check
# ----------------------------------------------------------------------

print("\n[5] INTENDED FEATURE CHECK")
print("-" * 70)

intended_features = [
    "was_positive",
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

all_columns = {
    column
    for header in raw_schema.values()
    for column in header
}

for feature in intended_features:

    exact_matches = [
        filename
        for filename, header in raw_schema.items()
        if feature in header
    ]

    if exact_matches:
        print(f"✓ {feature} → {exact_matches}")
    else:
        print(f"⚠ {feature} → NOT FOUND as an exact column name")

# ----------------------------------------------------------------------
# 7. Potential identifier detection
# ----------------------------------------------------------------------

print("\n[6] POTENTIAL IDENTIFIER COLUMNS")
print("-" * 70)

identifier_terms = [
    "patient_id",
    "patientid",
    "person_id",
    "personid",
    "encounter_id",
    "encounterid",
    "visit_id",
    "visitid",
    "order_id",
    "orderid",
    "specimen_id",
    "specimenid",
    "culture_id",
    "cultureid",
    "record_id",
    "recordid",
]

potential_identifier_columns = {}

for filename, header in raw_schema.items():

    matches = []

    for column in header:

        normalized = (
            column.lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        if any(term in normalized for term in identifier_terms):
            matches.append(column)

    potential_identifier_columns[filename] = matches

    if matches:
        print(f"\n{filename}")
        for column in matches:
            print(f"  → {column}")

# ----------------------------------------------------------------------
# 8. Potential temporal columns
# ----------------------------------------------------------------------

print("\n[7] POTENTIAL TEMPORAL COLUMNS")
print("-" * 70)

temporal_terms = [
    "date",
    "time",
    "timestamp",
    "datetime",
    "collected",
    "collection",
    "ordered",
    "order_time",
    "result_time",
    "reported",
]

potential_temporal_columns = {}

for filename, header in raw_schema.items():

    matches = []

    for column in header:

        column_lower = column.lower()

        if any(term in column_lower for term in temporal_terms):
            matches.append(column)

    potential_temporal_columns[filename] = matches

    if matches:
        print(f"\n{filename}")
        for column in matches:
            print(f"  → {column}")

# ----------------------------------------------------------------------
# 9. Schema assertions
# ----------------------------------------------------------------------

print("\n[8] Schema assertions")
print("-" * 70)

assert "microbiology_cultures_cohort.csv" in raw_schema
assert "microbiology_cultures_microbial_resistance.csv" in raw_schema

assert len(raw_schema["microbiology_cultures_cohort.csv"]) > 0
assert len(raw_schema["microbiology_cultures_microbial_resistance.csv"]) > 0

for filename, header in raw_schema.items():
    assert len(header) == len(set(header)), (
        f"Duplicate column names remain in {filename}"
    )

print("✓ Core schemas successfully read")
print("✓ No full dataset was loaded")
print("✓ No data were modified")
print("✓ Duplicate-column assertions passed")
print("✓ Candidate feature search completed")
print("✓ Identifier search completed")
print("✓ Temporal-column search completed")

print("\n" + "=" * 70)
print("STEP 2 SCHEMA INSPECTION COMPLETE")
print("=" * 70)

print("\nIMPORTANT:")
print("No cleaning was performed.")
print("No target was constructed.")
print("No train/validation/test split was performed.")
print("No categorical encoding was performed.")
print("No SMOTENC was performed.")
print("No model was created.")

STEP 2 — RAW DATA SCHEMA INSPECTION

[0] Required-object verification
----------------------------------------------------------------------
✓ candidate_files exists
✓ file_inventory exists

[1] Building file lookup
----------------------------------------------------------------------
✓ 16 unique dataset filenames available

[2] Selected files
----------------------------------------------------------------------
✓ microbiology_cultures_cohort.csv
✓ microbiology_cultures_microbial_resistance.csv
✓ microbiology_cultures_demographics.csv
✓ microbiology_cultures_labs.csv
✓ microbiology_cultures_ward_info.csv
✓ microbiology_cultures_implied_susceptibility.csv
⚠ NOT FOUND: microbiology_cultures_prior_infecting_organism.csv
✓ microbiology_cultures_adi_scores.csv
✓ microbiology_cultures_nursing_home_visits.csv

[3] RAW CSV SCHEMAS
----------------------------------------------------------------------

microbiology_cultures_cohort.csv
Column count: 10
   1. anon_id
   2. pat_enc_csn_id_coded


In [5]:
# ======================================================================
# STEP 3 — RAW DATA CONTENT, TARGET, AND PREDICTION-TIME AUDIT
# ======================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 3 — RAW DATA CONTENT, TARGET, AND PREDICTION-TIME AUDIT")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Required-object verification
# ----------------------------------------------------------------------

print("\n[0] Required-object verification")
print("-" * 70)

required_objects = [
    "candidate_files",
    "file_inventory",
    "file_lookup",
    "raw_schema",
]

for obj_name in required_objects:
    assert obj_name in globals(), (
        f"Missing object: {obj_name}. "
        f"This object should have been created by STEP 1 or STEP 2. "
        f"Rerun the relevant previous step."
    )
    print(f"✓ {obj_name} exists")

# ----------------------------------------------------------------------
# 1. Resolve core files
# ----------------------------------------------------------------------

print("\n[1] Resolving core files")
print("-" * 70)

cohort_filename = "microbiology_cultures_cohort.csv"
resistance_filename = "microbiology_cultures_microbial_resistance.csv"

assert cohort_filename in file_lookup, (
    f"Missing core file: {cohort_filename}. Rerun STEP 1."
)

assert resistance_filename in file_lookup, (
    f"Missing core file: {resistance_filename}. Rerun STEP 1."
)

cohort_path = file_lookup[cohort_filename]
resistance_path = file_lookup[resistance_filename]

print(f"✓ Cohort file    : {cohort_path}")
print(f"✓ Resistance file: {resistance_path}")

# ----------------------------------------------------------------------
# 2. Load a bounded raw sample
# ----------------------------------------------------------------------

print("\n[2] Loading bounded raw samples")
print("-" * 70)

# We intentionally do NOT load the complete 200+ MB files here.
# The sample is used only to understand value semantics and data quality.

SAMPLE_ROWS = 25_000

cohort_sample = pd.read_csv(
    cohort_path,
    nrows=SAMPLE_ROWS,
    low_memory=False
)

resistance_sample = pd.read_csv(
    resistance_path,
    nrows=SAMPLE_ROWS,
    low_memory=False
)

assert len(cohort_sample) > 0, "Cohort sample is empty."
assert len(resistance_sample) > 0, "Resistance sample is empty."

print(f"Cohort sample rows    : {len(cohort_sample):,}")
print(f"Resistance sample rows: {len(resistance_sample):,}")

# ----------------------------------------------------------------------
# 3. Preserve original sample snapshots
# ----------------------------------------------------------------------

print("\n[3] Raw-sample integrity snapshots")
print("-" * 70)

cohort_sample_columns_before = cohort_sample.columns.tolist()
resistance_sample_columns_before = resistance_sample.columns.tolist()

cohort_sample_shape_before = cohort_sample.shape
resistance_sample_shape_before = resistance_sample.shape

assert cohort_sample_columns_before == raw_schema[cohort_filename]
assert resistance_sample_columns_before == raw_schema[resistance_filename]

print("✓ Cohort schema matches STEP 2")
print("✓ Resistance schema matches STEP 2")

# ----------------------------------------------------------------------
# 4. Display dtypes
# ----------------------------------------------------------------------

print("\n[4] Raw dtypes — cohort")
print("-" * 70)

print(cohort_sample.dtypes.to_string())

print("\n[4b] Raw dtypes — resistance")
print("-" * 70)

print(resistance_sample.dtypes.to_string())

# ----------------------------------------------------------------------
# 5. Missing-value audit
# ----------------------------------------------------------------------

print("\n[5] Missing-value audit — cohort")
print("-" * 70)

cohort_missing = (
    cohort_sample.isna()
    .sum()
    .sort_values(ascending=False)
)

cohort_missing_pct = (
    cohort_sample.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

cohort_missing_table = pd.DataFrame({
    "missing_count": cohort_missing,
    "missing_percent": cohort_missing_pct
})

print(cohort_missing_table.to_string())

print("\n[5b] Missing-value audit — resistance")
print("-" * 70)

resistance_missing = (
    resistance_sample.isna()
    .sum()
    .sort_values(ascending=False)
)

resistance_missing_pct = (
    resistance_sample.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

resistance_missing_table = pd.DataFrame({
    "missing_count": resistance_missing,
    "missing_percent": resistance_missing_pct
})

print(resistance_missing_table.to_string())

# ----------------------------------------------------------------------
# 6. Examine intended target candidates
# ----------------------------------------------------------------------

print("\n[6] TARGET CANDIDATE VALUE AUDIT")
print("-" * 70)

target_candidates = [
    "susceptibility",
    "resistant_time_to_culturetime",
    "was_positive",
]

for column in target_candidates:

    if column in cohort_sample.columns:
        print(f"\nCOHORT → {column}")
        print("-" * 50)
        print(f"dtype: {cohort_sample[column].dtype}")
        print(f"non-null: {cohort_sample[column].notna().sum():,}")
        print(f"unique: {cohort_sample[column].nunique(dropna=False):,}")
        print(
            cohort_sample[column]
            .value_counts(dropna=False)
            .head(30)
            .to_string()
        )

    if column in resistance_sample.columns:
        print(f"\nRESISTANCE → {column}")
        print("-" * 50)
        print(f"dtype: {resistance_sample[column].dtype}")
        print(f"non-null: {resistance_sample[column].notna().sum():,}")
        print(f"unique: {resistance_sample[column].nunique(dropna=False):,}")
        print(
            resistance_sample[column]
            .value_counts(dropna=False)
            .head(30)
            .to_string()
        )

# ----------------------------------------------------------------------
# 7. Examine intended predictor values
# ----------------------------------------------------------------------

print("\n[7] INTENDED PREDICTOR VALUE AUDIT")
print("-" * 70)

intended_predictors = [
    "was_positive",
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

for column in intended_predictors:

    if column not in cohort_sample.columns:
        print(f"\n⚠ {column} NOT PRESENT IN COHORT SAMPLE")
        continue

    series = cohort_sample[column]

    print(f"\nCOHORT → {column}")
    print("-" * 50)
    print(f"dtype          : {series.dtype}")
    print(f"non-null       : {series.notna().sum():,}")
    print(f"missing        : {series.isna().sum():,}")
    print(f"unique values  : {series.nunique(dropna=True):,}")

    # For high-cardinality fields, show only the most frequent values.
    print("Top values:")
    print(
        series
        .value_counts(dropna=False)
        .head(20)
        .to_string()
    )

# ----------------------------------------------------------------------
# 8. Inspect timestamps
# ----------------------------------------------------------------------

print("\n[8] TIMESTAMP AUDIT")
print("-" * 70)

timestamp_column = "order_time_jittered_utc"

assert timestamp_column in cohort_sample.columns
assert timestamp_column in resistance_sample.columns

print("COHORT timestamp:")
print(cohort_sample[timestamp_column].head(10).to_string(index=False))

print("\nRESISTANCE timestamp:")
print(resistance_sample[timestamp_column].head(10).to_string(index=False))

# Parse copies only. Original raw columns remain untouched.
cohort_order_time_parsed = pd.to_datetime(
    cohort_sample[timestamp_column],
    errors="coerce",
    utc=True
)

resistance_order_time_parsed = pd.to_datetime(
    resistance_sample[timestamp_column],
    errors="coerce",
    utc=True
)

print("\nParsed cohort timestamp range:")
print(f"min: {cohort_order_time_parsed.min()}")
print(f"max: {cohort_order_time_parsed.max()}")
print(f"invalid: {cohort_order_time_parsed.isna().sum():,}")

print("\nParsed resistance timestamp range:")
print(f"min: {resistance_order_time_parsed.min()}")
print(f"max: {resistance_order_time_parsed.max()}")
print(f"invalid: {resistance_order_time_parsed.isna().sum():,}")

# ----------------------------------------------------------------------
# 9. Identifier / row-granularity audit
# ----------------------------------------------------------------------

print("\n[9] ROW-GRANULARITY / IDENTIFIER AUDIT")
print("-" * 70)

key_columns = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
]

for filename, df in [
    (cohort_filename, cohort_sample),
    (resistance_filename, resistance_sample),
]:

    print(f"\n{filename}")
    print("-" * 50)

    for column in key_columns:

        assert column in df.columns, (
            f"{column} missing from {filename}"
        )

        print(
            f"{column:<28} "
            f"non-null={df[column].notna().sum():,} | "
            f"unique={df[column].nunique(dropna=True):,}"
        )

    full_key_unique = (
        df[key_columns]
        .drop_duplicates()
        .shape[0]
    )

    print(
        f"Unique 3-column key combinations: "
        f"{full_key_unique:,} / {len(df):,}"
    )

    duplicate_key_rows = len(df) - full_key_unique

    print(f"Duplicate key rows: {duplicate_key_rows:,}")

# ----------------------------------------------------------------------
# 10. Check cohort susceptibility versus resistance records
# ----------------------------------------------------------------------

print("\n[10] COHORT ↔ RESISTANCE KEY RELATIONSHIP")
print("-" * 70)

join_keys = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "organism",
    "antibiotic",
]

for column in join_keys:

    assert column in cohort_sample.columns, (
        f"{column} missing from cohort sample."
    )

    assert column in resistance_sample.columns, (
        f"{column} missing from resistance sample."
    )

cohort_keys = cohort_sample[join_keys].drop_duplicates()
resistance_keys = resistance_sample[join_keys].drop_duplicates()

matched_keys = cohort_keys.merge(
    resistance_keys,
    on=join_keys,
    how="inner"
)

print(f"Unique cohort keys     : {len(cohort_keys):,}")
print(f"Unique resistance keys : {len(resistance_keys):,}")
print(f"Matched keys            : {len(matched_keys):,}")

if len(cohort_keys) > 0:
    match_rate = len(matched_keys) / len(cohort_keys) * 100
    print(f"Cohort→resistance match rate: {match_rate:.2f}%")

# ----------------------------------------------------------------------
# 11. Search for exact target-like fields across cohort
# ----------------------------------------------------------------------

print("\n[11] POTENTIAL OUTCOME / POST-OUTCOME COLUMNS")
print("-" * 70)

outcome_terms = [
    "suscept",
    "resist",
    "result",
    "sensitivity",
    "amr",
    "mic",
    "zone",
    "interpret",
    "phenotype",
]

outcome_like_columns = []

for column in cohort_sample.columns:

    column_lower = column.lower()

    if any(term in column_lower for term in outcome_terms):
        outcome_like_columns.append(column)

print("Cohort outcome-like columns:")

for column in outcome_like_columns:
    print(f"  → {column}")

# ----------------------------------------------------------------------
# 12. Explicit integrity assertions
# ----------------------------------------------------------------------

print("\n[12] Assertions")
print("-" * 70)

# Schema unchanged.
assert cohort_sample.columns.tolist() == cohort_sample_columns_before
assert resistance_sample.columns.tolist() == resistance_sample_columns_before

# Shape unchanged.
assert cohort_sample.shape == cohort_sample_shape_before
assert resistance_sample.shape == resistance_sample_shape_before

# Required columns.
for column in intended_predictors:
    assert column in cohort_sample.columns, (
        f"Required intended predictor missing: {column}"
    )

assert "susceptibility" in cohort_sample.columns
assert "resistant_time_to_culturetime" in resistance_sample.columns

# Key columns.
for column in key_columns:
    assert column in cohort_sample.columns
    assert column in resistance_sample.columns

# Parsed timestamps must not contain impossible object types.
assert pd.api.types.is_datetime64_any_dtype(
    cohort_order_time_parsed
)

assert pd.api.types.is_datetime64_any_dtype(
    resistance_order_time_parsed
)

print("✓ Required objects exist")
print("✓ Core files resolved")
print("✓ Raw schemas unchanged")
print("✓ Intended predictors exist")
print("✓ Outcome candidates exist")
print("✓ Key columns exist")
print("✓ Timestamp parsing completed")
print("✓ No source DataFrame was modified")

print("\n" + "=" * 70)
print("STEP 3 RAW CONTENT / TARGET AUDIT COMPLETE")
print("=" * 70)

print("\nNO MODELING DECISION HAS BEEN MADE IN THIS STEP.")
print("NO TARGET HAS BEEN CONSTRUCTED.")
print("NO FEATURE HAS BEEN REMOVED.")
print("NO DATA SPLIT HAS BEEN PERFORMED.")
print("NO SMOTENC HAS BEEN PERFORMED.")

STEP 3 — RAW DATA CONTENT, TARGET, AND PREDICTION-TIME AUDIT

[0] Required-object verification
----------------------------------------------------------------------
✓ candidate_files exists
✓ file_inventory exists
✓ file_lookup exists
✓ raw_schema exists

[1] Resolving core files
----------------------------------------------------------------------
✓ Cohort file    : C:\Users\HPCLAB\SIH_LightGBM\microbiology_cultures_cohort.csv
✓ Resistance file: C:\Users\HPCLAB\SIH_LightGBM\microbiology_cultures_microbial_resistance.csv

[2] Loading bounded raw samples
----------------------------------------------------------------------
Cohort sample rows    : 25,000
Resistance sample rows: 25,000

[3] Raw-sample integrity snapshots
----------------------------------------------------------------------
✓ Cohort schema matches STEP 2
✓ Resistance schema matches STEP 2

[4] Raw dtypes — cohort
----------------------------------------------------------------------
anon_id                    object
pa

In [7]:
# ======================================================================
# STEP 4 — BINARY AMR TARGET CONSTRUCTION
# ======================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4 — BINARY AMR TARGET CONSTRUCTION")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Required-object verification
# ----------------------------------------------------------------------

print("\n[0] Required-object verification")
print("-" * 70)

required_objects = [
    "file_lookup",
    "cohort_filename",
    "cohort_path",
]

for obj_name in required_objects:
    assert obj_name in globals(), (
        f"Missing object: {obj_name}. "
        f"This should have been created by an earlier step. "
        f"Rerun the relevant previous step."
    )
    print(f"✓ {obj_name} exists")

assert cohort_filename == "microbiology_cultures_cohort.csv"
assert cohort_path.exists()

# ----------------------------------------------------------------------
# 1. Load the complete cohort table
# ----------------------------------------------------------------------

print("\n[1] Loading cohort dataset")
print("-" * 70)

# The cohort is ~245 MB, which is substantially smaller than the
# 18.8 GB comorbidity table. We load ONLY the cohort here.
#
# No source file is modified.

cohort_raw = pd.read_csv(
    cohort_path,
    low_memory=False
)

assert isinstance(cohort_raw, pd.DataFrame)
assert len(cohort_raw) > 0

print(f"Rows    : {len(cohort_raw):,}")
print(f"Columns : {len(cohort_raw.columns):,}")
print(f"Memory  : {cohort_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# ----------------------------------------------------------------------
# 2. Verify expected raw schema
# ----------------------------------------------------------------------

print("\n[2] Raw cohort schema verification")
print("-" * 70)

expected_cohort_columns = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "order_time_jittered_utc",
    "ordering_mode",
    "culture_description",
    "was_positive",
    "organism",
    "antibiotic",
    "susceptibility",
]

assert cohort_raw.columns.tolist() == expected_cohort_columns, (
    "The current cohort schema differs from the schema observed in "
    "STEP 2. Do not continue until this discrepancy is investigated."
)

print("✓ Exact expected cohort schema confirmed")

# ----------------------------------------------------------------------
# 3. Verify raw target has no missing values before construction
# ----------------------------------------------------------------------

print("\n[3] Raw susceptibility audit")
print("-" * 70)

assert "susceptibility" in cohort_raw.columns

susceptibility_missing = cohort_raw["susceptibility"].isna().sum()

print(f"Total rows                 : {len(cohort_raw):,}")
print(f"Missing susceptibility    : {susceptibility_missing:,}")

assert susceptibility_missing == 0, (
    "Unexpected missing susceptibility values detected in the full "
    "cohort. The target construction must account for them explicitly."
)

full_susceptibility_counts = (
    cohort_raw["susceptibility"]
    .value_counts(dropna=False)
)

print("\nFull-dataset susceptibility distribution:")
print(full_susceptibility_counts.to_string())

# ----------------------------------------------------------------------
# 4. Verify the observed susceptibility categories
# ----------------------------------------------------------------------

print("\n[4] Susceptibility category verification")
print("-" * 70)

expected_categories = {
    "Susceptible",
    "Resistant",
    "Intermediate",
    "Inconclusive",
    "Null",
    "Synergism",
}

observed_categories = set(
    cohort_raw["susceptibility"].dropna().unique()
)

print(f"Observed categories : {sorted(observed_categories)}")
print(f"Expected categories : {sorted(expected_categories)}")

unexpected_categories = observed_categories - expected_categories

assert not unexpected_categories, (
    f"Unexpected susceptibility categories detected: "
    f"{sorted(unexpected_categories)}"
)

print("✓ No unexpected susceptibility categories detected")

# ----------------------------------------------------------------------
# 5. Construct binary AMR target
# ----------------------------------------------------------------------

print("\n[5] Constructing binary AMR target")
print("-" * 70)

# Defensible binary definition:
#
# Resistant   -> AMR target = 1
# Susceptible -> AMR target = 0
#
# Intermediate, Inconclusive, Null, and Synergism are excluded from
# the binary modeling cohort rather than being arbitrarily relabeled.

binary_target_values = {
    "Susceptible": 0,
    "Resistant": 1,
}

binary_mask = cohort_raw["susceptibility"].isin(
    binary_target_values.keys()
)

assert binary_mask.any(), (
    "No rows contain a valid binary susceptibility category."
)

# Create a NEW DataFrame. Do not modify cohort_raw in-place.
cohort_binary = cohort_raw.loc[binary_mask].copy()

cohort_binary["amr_target"] = (
    cohort_binary["susceptibility"]
    .map(binary_target_values)
    .astype("int8")
)

# ----------------------------------------------------------------------
# 6. Verify excluded categories
# ----------------------------------------------------------------------

print("\n[6] Binary cohort composition")
print("-" * 70)

excluded_mask = ~binary_mask

excluded_counts = (
    cohort_raw.loc[excluded_mask, "susceptibility"]
    .value_counts(dropna=False)
)

print(f"Original cohort rows      : {len(cohort_raw):,}")
print(f"Binary modeling rows      : {len(cohort_binary):,}")
print(f"Excluded rows             : {excluded_mask.sum():,}")
print(
    f"Retention percentage      : "
    f"{len(cohort_binary) / len(cohort_raw) * 100:.2f}%"
)

print("\nExcluded categories:")
if len(excluded_counts) > 0:
    print(excluded_counts.to_string())
else:
    print("None")

# ----------------------------------------------------------------------
# 7. Target distribution
# ----------------------------------------------------------------------

print("\n[7] Binary AMR target distribution")
print("-" * 70)

target_counts = (
    cohort_binary["amr_target"]
    .value_counts()
    .sort_index()
)

target_percentages = (
    cohort_binary["amr_target"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

print(target_distribution.to_string())

assert set(target_counts.index.tolist()) <= {0, 1}
assert len(target_counts) == 2, (
    "The binary target contains only one class. "
    "A binary classifier cannot be trained until both classes exist."
)

# ----------------------------------------------------------------------
# 8. Verify target mapping against susceptibility
# ----------------------------------------------------------------------

print("\n[8] Target mapping verification")
print("-" * 70)

susceptible_targets = cohort_binary.loc[
    cohort_binary["susceptibility"] == "Susceptible",
    "amr_target"
]

resistant_targets = cohort_binary.loc[
    cohort_binary["susceptibility"] == "Resistant",
    "amr_target"
]

assert len(susceptible_targets) > 0
assert len(resistant_targets) > 0

assert (susceptible_targets == 0).all(), (
    "Susceptible rows were not mapped entirely to target 0."
)

assert (resistant_targets == 1).all(), (
    "Resistant rows were not mapped entirely to target 1."
)

print("✓ Susceptible → 0 verified")
print("✓ Resistant   → 1 verified")

# ----------------------------------------------------------------------
# 9. Target length / index integrity
# ----------------------------------------------------------------------

print("\n[9] Target integrity")
print("-" * 70)

assert len(cohort_binary) == len(cohort_binary["amr_target"])
assert cohort_binary["amr_target"].notna().all()
assert cohort_binary["amr_target"].dtype == np.int8

assert cohort_binary.index.is_unique, (
    "Binary cohort index is not unique."
)

print(f"Binary cohort rows : {len(cohort_binary):,}")
print(f"Target rows        : {len(cohort_binary['amr_target']):,}")
print(f"Target dtype       : {cohort_binary['amr_target'].dtype}")
print("✓ Target length matches feature-table length")
print("✓ No missing target values")
print("✓ Target contains only 0 and 1")

# ----------------------------------------------------------------------
# 10. Verify original cohort was not modified
# ----------------------------------------------------------------------

print("\n[10] Raw-data integrity verification")
print("-" * 70)

assert "amr_target" not in cohort_raw.columns, (
    "cohort_raw was modified in-place. This violates raw-data "
    "integrity requirements."
)

assert cohort_raw.columns.tolist() == expected_cohort_columns

assert cohort_raw.shape[0] == len(cohort_binary) + excluded_mask.sum()

print("✓ cohort_raw contains no derived target column")
print("✓ Original cohort schema unchanged")
print("✓ Row accounting is exact")

# ----------------------------------------------------------------------
# 11. Explicit methodological record
# ----------------------------------------------------------------------

print("\n[11] Target-definition record")
print("-" * 70)

target_definition = {
    "target_name": "amr_target",
    "source_column": "susceptibility",
    "positive_class": "Resistant",
    "positive_value": 1,
    "negative_class": "Susceptible",
    "negative_value": 0,
    "excluded_categories": [
        "Intermediate",
        "Inconclusive",
        "Null",
        "Synergism",
    ],
}

for key, value in target_definition.items():
    print(f"{key:<24}: {value}")

# ----------------------------------------------------------------------
# 12. Final assertions
# ----------------------------------------------------------------------

print("\n[12] Final assertions")
print("-" * 70)

assert isinstance(cohort_raw, pd.DataFrame)
assert isinstance(cohort_binary, pd.DataFrame)

assert "susceptibility" in cohort_raw.columns
assert "amr_target" in cohort_binary.columns

assert cohort_binary["amr_target"].isin([0, 1]).all()
assert cohort_binary["amr_target"].notna().all()

assert (
    cohort_binary.loc[
        cohort_binary["susceptibility"] == "Resistant",
        "amr_target"
    ] == 1
).all()

assert (
    cohort_binary.loc[
        cohort_binary["susceptibility"] == "Susceptible",
        "amr_target"
    ] == 0
).all()

print("✓ All target-construction assertions passed")

print("\n" + "=" * 70)
print("STEP 4 TARGET CONSTRUCTION COMPLETE")
print("=" * 70)

print("\nTARGET DEFINITION:")
print("  Resistant   → 1")
print("  Susceptible → 0")
print(
    "  Intermediate / Inconclusive / Null / Synergism "
    "→ excluded from binary cohort"
)

print("\nNO TRAIN/VALIDATION/TEST SPLIT WAS PERFORMED.")
print("NO SMOTENC WAS PERFORMED.")
print("NO MODEL WAS CREATED.")

STEP 4 — BINARY AMR TARGET CONSTRUCTION

[0] Required-object verification
----------------------------------------------------------------------
✓ file_lookup exists
✓ cohort_filename exists
✓ cohort_path exists

[1] Loading cohort dataset
----------------------------------------------------------------------
Rows    : 2,241,050
Columns : 10
Memory  : 961.45 MB

[2] Raw cohort schema verification
----------------------------------------------------------------------
✓ Exact expected cohort schema confirmed

[3] Raw susceptibility audit
----------------------------------------------------------------------
Total rows                 : 2,241,050
Missing susceptibility    : 0

Full-dataset susceptibility distribution:
susceptibility
Susceptible     1289258
Null             634468
Resistant        265071
Intermediate      47651
Inconclusive       2670
Synergism          1932

[4] Susceptibility category verification
----------------------------------------------------------------------
Obs

In [9]:
# ======================================================================
# STEP 5 — FEATURE, LEAKAGE, AND PREDICTION-TIME AUDIT
# ======================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5 — FEATURE, LEAKAGE, AND PREDICTION-TIME AUDIT")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Required-object verification
# ----------------------------------------------------------------------

print("\n[0] Required-object verification")
print("-" * 70)

required_objects = [
    "cohort_raw",
    "cohort_binary",
    "target_definition",
]

for obj_name in required_objects:
    assert obj_name in globals(), (
        f"Missing object: {obj_name}. "
        f"This object should have been created by STEP 4. "
        f"Rerun STEP 4 rather than reconstructing it here."
    )
    print(f"✓ {obj_name} exists")

assert isinstance(cohort_raw, pd.DataFrame)
assert isinstance(cohort_binary, pd.DataFrame)

# ----------------------------------------------------------------------
# 1. Required schema verification
# ----------------------------------------------------------------------

print("\n[1] Schema verification")
print("-" * 70)

required_columns = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "order_time_jittered_utc",
    "ordering_mode",
    "culture_description",
    "was_positive",
    "organism",
    "antibiotic",
    "susceptibility",
    "amr_target",
]

missing_required = [
    c for c in required_columns
    if c not in cohort_binary.columns
]

assert not missing_required, (
    f"Missing required columns: {missing_required}"
)

print("✓ Required columns present")

# ----------------------------------------------------------------------
# 2. Define candidate predictor set
# ----------------------------------------------------------------------

print("\n[2] Candidate predictor audit")
print("-" * 70)

candidate_features = [
    "was_positive",
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

target_column = "amr_target"
target_source_column = "susceptibility"

print("Candidate predictors:")
for feature in candidate_features:
    print(f"  → {feature}")

print(f"\nTarget                : {target_column}")
print(f"Target source column  : {target_source_column}")

assert target_column not in candidate_features
assert target_source_column not in candidate_features

print("✓ Target is not included in candidate predictors")
print("✓ Raw susceptibility column is not included in candidate predictors")

# ----------------------------------------------------------------------
# 3. Explicit identifier / timestamp audit
# ----------------------------------------------------------------------

print("\n[3] Identifier and temporal-column audit")
print("-" * 70)

identifier_columns = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
]

temporal_columns = [
    "order_time_jittered_utc",
]

print("Identifier columns:")
for col in identifier_columns:
    nunique = cohort_binary[col].nunique(dropna=False)
    print(
        f"  {col:<28} "
        f"unique={nunique:,} / rows={len(cohort_binary):,}"
    )

print("\nTemporal columns:")
for col in temporal_columns:
    parsed = pd.to_datetime(
        cohort_binary[col],
        errors="coerce",
        utc=True
    )

    invalid = parsed.isna().sum()

    print(
        f"  {col:<28} "
        f"invalid={invalid:,}"
    )

    assert invalid == 0, (
        f"Invalid timestamps detected in {col}."
    )

print("✓ Identifier and timestamp columns identified explicitly")

# ----------------------------------------------------------------------
# 4. Full-dataset was_positive audit
# ----------------------------------------------------------------------

print("\n[4] Full binary-cohort audit of was_positive")
print("-" * 70)

was_positive_counts = (
    cohort_binary["was_positive"]
    .value_counts(dropna=False)
    .sort_index()
)

print(was_positive_counts.to_string())

was_positive_unique = (
    cohort_binary["was_positive"]
    .nunique(dropna=False)
)

was_positive_missing = (
    cohort_binary["was_positive"]
    .isna()
    .sum()
)

print(f"\nUnique values : {was_positive_unique}")
print(f"Missing       : {was_positive_missing}")

assert was_positive_missing == 0

if was_positive_unique == 1:
    print(
        "⚠ was_positive is constant in the complete binary cohort."
    )
    print(
        "  It has no predictive variance and will be formally excluded"
        " from the model feature matrix."
    )
else:
    print(
        "✓ was_positive contains more than one value and requires"
        " further prediction-time assessment."
    )

# ----------------------------------------------------------------------
# 5. Missing-value audit for candidate predictors
# ----------------------------------------------------------------------

print("\n[5] Candidate predictor missing-value audit")
print("-" * 70)

predictor_missing = pd.DataFrame({
    "missing_count": cohort_binary[candidate_features].isna().sum(),
    "missing_percent": (
        cohort_binary[candidate_features].isna().mean() * 100
    ),
})

print(predictor_missing.to_string())

print(
    "\nNote: literal strings such as 'Null' are not pandas NaN values."
)
print(
    "They will be treated as observed category values unless a later"
    " methodological decision explicitly changes that."
)

# ----------------------------------------------------------------------
# 6. Target leakage audit
# ----------------------------------------------------------------------

print("\n[6] Direct target-leakage audit")
print("-" * 70)

for col in cohort_binary.columns:
    if col == target_column:
        continue

    # Check exact equality with the target after converting to strings.
    # This is only a screening test, not the sole leakage criterion.
    try:
        exact_match = (
            cohort_binary[col].astype(str).values
            == cohort_binary[target_column].astype(str).values
        ).mean()
    except Exception:
        exact_match = np.nan

    if pd.notna(exact_match) and exact_match == 1.0:
        print(
            f"⚠ {col} exactly matches the target for every row."
        )

print(
    "✓ Direct exact-match screening completed"
)

# Explicit assertion: susceptibility must never become a predictor.
assert target_source_column not in candidate_features

# ----------------------------------------------------------------------
# 7. Post-outcome / prediction-time assessment
# ----------------------------------------------------------------------

print("\n[7] Prediction-time availability assessment")
print("-" * 70)

prediction_time_audit = pd.DataFrame([
    {
        "feature": "was_positive",
        "status": "AUDIT_REQUIRED",
        "reason": (
            "Microbiology positivity indicator. It may be available "
            "only after culture processing and is constant in the "
            "current cohort."
        ),
    },
    {
        "feature": "ordering_mode",
        "status": "POTENTIALLY_AVAILABLE",
        "reason": (
            "Ordering context can plausibly be available at the "
            "time the culture/order is placed."
        ),
    },
    {
        "feature": "culture_description",
        "status": "POTENTIALLY_AVAILABLE",
        "reason": (
            "The ordered culture type can plausibly be known at "
            "order time."
        ),
    },
    {
        "feature": "organism",
        "status": "CRITICAL_DECISION",
        "reason": (
            "Organism identification is commonly obtained from "
            "culture results. If AMR is predicted before organism "
            "identification, this is post-outcome leakage. If the "
            "prediction task is explicitly organism-conditioned "
            "after identification, it may be valid."
        ),
    },
    {
        "feature": "antibiotic",
        "status": "CRITICAL_DECISION",
        "reason": (
            "Must be verified as the antibiotic being evaluated/"
            "predicted and as information available at the declared "
            "prediction point. It may be valid for antibiotic-specific "
            "susceptibility prediction."
        ),
    },
])

print(
    prediction_time_audit[
        ["feature", "status", "reason"]
    ].to_string(index=False)
)

# ----------------------------------------------------------------------
# 8. Row-granularity audit
# ----------------------------------------------------------------------

print("\n[8] Row-granularity audit")
print("-" * 70)

row_count = len(cohort_binary)

order_unique = cohort_binary["order_proc_id_coded"].nunique()
encounter_unique = cohort_binary["pat_enc_csn_id_coded"].nunique()
patient_unique = cohort_binary["anon_id"].nunique()

print(f"Rows                    : {row_count:,}")
print(f"Unique orders           : {order_unique:,}")
print(f"Unique encounters       : {encounter_unique:,}")
print(f"Unique patients         : {patient_unique:,}")

order_duplicate_rows = (
    row_count - order_unique
)

encounter_duplicate_rows = (
    row_count - encounter_unique
)

print(f"\nRows beyond unique orders      : {order_duplicate_rows:,}")
print(f"Rows beyond unique encounters : {encounter_duplicate_rows:,}")

assert row_count >= order_unique
assert row_count >= encounter_unique
assert row_count >= patient_unique

# ----------------------------------------------------------------------
# 9. Order-level target consistency audit
# ----------------------------------------------------------------------

print("\n[9] Order-level target consistency")
print("-" * 70)

order_target_nunique = (
    cohort_binary
    .groupby("order_proc_id_coded")["amr_target"]
    .nunique()
)

conflicting_orders = (
    order_target_nunique[order_target_nunique > 1]
)

print(f"Unique orders                 : {len(order_target_nunique):,}")
print(f"Orders with both target classes: {len(conflicting_orders):,}")

if len(conflicting_orders) > 0:
    print(
        "\n⚠ Some order IDs contain both resistant and susceptible"
        " observations."
    )
    print(
        "This means order_proc_id_coded alone does not define a unique"
        " binary outcome at the row level."
    )
else:
    print(
        "✓ Every order has a single AMR target across its rows."
    )

# ----------------------------------------------------------------------
# 10. Encounter-level target consistency audit
# ----------------------------------------------------------------------

print("\n[10] Encounter-level target consistency")
print("-" * 70)

encounter_target_nunique = (
    cohort_binary
    .groupby("pat_enc_csn_id_coded")["amr_target"]
    .nunique()
)

conflicting_encounters = (
    encounter_target_nunique[encounter_target_nunique > 1]
)

print(f"Unique encounters                  : {len(encounter_target_nunique):,}")
print(
    f"Encounters with both target classes: "
    f"{len(conflicting_encounters):,}"
)

# This is an audit only. We do not remove anything here.

# ----------------------------------------------------------------------
# 11. Patient-level target consistency audit
# ----------------------------------------------------------------------

print("\n[11] Patient-level target consistency")
print("-" * 70)

patient_target_nunique = (
    cohort_binary
    .groupby("anon_id")["amr_target"]
    .nunique()
)

conflicting_patients = (
    patient_target_nunique[patient_target_nunique > 1]
)

print(f"Unique patients                  : {len(patient_target_nunique):,}")
print(
    f"Patients with both target classes: "
    f"{len(conflicting_patients):,}"
)

print(
    "\nInterpretation:"
)
print(
    "Patient-level target variation is expected in longitudinal data."
)
print(
    "However, it means a naive row-level random split can place the"
    " same patient's observations into train, validation, and test."
)

# ----------------------------------------------------------------------
# 12. Duplicate feature-row audit
# ----------------------------------------------------------------------

print("\n[12] Duplicate predictor-pattern audit")
print("-" * 70)

duplicate_feature_columns = candidate_features + [target_column]

duplicate_mask = cohort_binary.duplicated(
    subset=duplicate_feature_columns,
    keep=False
)

duplicate_rows = duplicate_mask.sum()

print(
    f"Rows participating in exact duplicate "
    f"feature+target patterns : {duplicate_rows:,}"
)

if duplicate_rows > 0:
    print(
        "⚠ Exact duplicate predictor/target patterns exist."
    )
else:
    print(
        "✓ No exact duplicate predictor/target patterns."
    )

# ----------------------------------------------------------------------
# 13. Feature cardinality audit
# ----------------------------------------------------------------------

print("\n[13] Candidate feature cardinality")
print("-" * 70)

feature_cardinality = []

for feature in candidate_features:
    feature_cardinality.append({
        "feature": feature,
        "dtype": str(cohort_binary[feature].dtype),
        "unique": cohort_binary[feature].nunique(dropna=False),
        "missing": cohort_binary[feature].isna().sum(),
    })

feature_cardinality = pd.DataFrame(feature_cardinality)

print(feature_cardinality.to_string(index=False))

# ----------------------------------------------------------------------
# 14. Preliminary feature-status record
# ----------------------------------------------------------------------

print("\n[14] Preliminary feature-status record")
print("-" * 70)

feature_status = {
    "was_positive": (
        "EXCLUDE_IF_CONSTANT"
        if was_positive_unique == 1
        else "REQUIRES_PREDICTION_TIME_REVIEW"
    ),
    "ordering_mode": "PREDICTION_TIME_REVIEW",
    "culture_description": "PREDICTION_TIME_REVIEW",
    "organism": "CRITICAL_PREDICTION_TIME_DECISION",
    "antibiotic": "CRITICAL_PREDICTION_TIME_DECISION",
}

for feature, status in feature_status.items():
    print(f"{feature:<24}: {status}")

# ----------------------------------------------------------------------
# 15. Explicit leakage exclusions that require no prediction-time debate
# ----------------------------------------------------------------------

print("\n[15] Columns prohibited from model predictors")
print("-" * 70)

always_excluded = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "order_time_jittered_utc",
    "susceptibility",
    "amr_target",
]

for col in always_excluded:
    assert col in cohort_binary.columns
    assert col not in candidate_features
    print(f"✓ {col:<28} excluded from candidate predictor matrix")

print(
    "\nReason:"
)
print(
    "Identifiers are excluded because they identify records rather"
    " than represent clinical predictors and can encode memorization/"
    " dataset structure."
)
print(
    "The timestamp is excluded from the current five-feature design"
    " because it is temporal metadata rather than one of the declared"
    " clinical/order predictors."
)
print(
    "susceptibility is the source of the target and therefore cannot"
    " be a predictor."
)
print(
    "amr_target is the outcome itself and cannot be a predictor."
)

# ----------------------------------------------------------------------
# 16. Final integrity assertions
# ----------------------------------------------------------------------

print("\n[16] Final assertions")
print("-" * 70)

assert len(cohort_binary) == 1_554_329, (
    "The binary cohort row count changed unexpectedly from STEP 4."
)

assert cohort_binary["amr_target"].isin([0, 1]).all()
assert cohort_binary["amr_target"].notna().all()

assert target_source_column not in candidate_features

for col in always_excluded:
    assert col not in candidate_features

assert set(candidate_features) == {
    "was_positive",
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
}

print("✓ Binary cohort row count unchanged")
print("✓ Target remains binary")
print("✓ Target source excluded from predictors")
print("✓ Identifier/target/timestamp exclusions recorded")
print("✓ Candidate feature list verified")

# ----------------------------------------------------------------------
# 17. Important methodological stop condition
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 AUDIT COMPLETE")
print("=" * 70)

print(
    "\nIMPORTANT:"
)
print(
    "No train/validation/test split was performed."
)
print(
    "No preprocessing was performed."
)
print(
    "No category mappings were fitted."
)
print(
    "No SMOTENC was performed."
)
print(
    "No model was created."
)

print(
    "\nCRITICAL DECISION REQUIRED BEFORE SPLITTING:"
)
print(
    "The prediction point must determine whether ORGANISM is legally"
    " available as a predictor."
)
print(
    "ANTIBIOTIC also requires confirmation that it represents the"
    " antibiotic-specific prediction task and is known at prediction."
)
print(
    "The duplicate order/encounter structure must also be considered"
    " when designing the train/validation/test split."
)

STEP 5 — FEATURE, LEAKAGE, AND PREDICTION-TIME AUDIT

[0] Required-object verification
----------------------------------------------------------------------
✓ cohort_raw exists
✓ cohort_binary exists
✓ target_definition exists

[1] Schema verification
----------------------------------------------------------------------
✓ Required columns present

[2] Candidate predictor audit
----------------------------------------------------------------------
Candidate predictors:
  → was_positive
  → ordering_mode
  → culture_description
  → organism
  → antibiotic

Target                : amr_target
Target source column  : susceptibility
✓ Target is not included in candidate predictors
✓ Raw susceptibility column is not included in candidate predictors

[3] Identifier and temporal-column audit
----------------------------------------------------------------------
Identifier columns:
  anon_id                      unique=67,007 / rows=1,554,329
  pat_enc_csn_id_coded         unique=116,994 / row

In [11]:
# ======================================================================
# STEP 6 — GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

print("=" * 70)
print("STEP 6 — GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

# ----------------------------------------------------------------------
# 0. Required-object verification
# ----------------------------------------------------------------------

print("\n[0] Required-object verification")
print("-" * 70)

required_objects = [
    "cohort_binary",
    "candidate_features",
    "target_column",
]

for obj_name in required_objects:
    assert obj_name in globals(), (
        f"Missing object: {obj_name}. "
        f"This should have been created by STEP 4/5. "
        f"Rerun the appropriate previous step instead of reconstructing it."
    )
    print(f"✓ {obj_name} exists")

assert isinstance(cohort_binary, pd.DataFrame)

# ----------------------------------------------------------------------
# 1. Required column verification
# ----------------------------------------------------------------------

print("\n[1] Required-column verification")
print("-" * 70)

required_columns = (
    list(candidate_features)
    + [
        target_column,
        "anon_id",
        "pat_enc_csn_id_coded",
        "order_proc_id_coded",
    ]
)

missing_columns = [
    col for col in required_columns
    if col not in cohort_binary.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print("✓ All split-required columns exist")

# ----------------------------------------------------------------------
# 2. Dataset integrity snapshot
# ----------------------------------------------------------------------

print("\n[2] Pre-split integrity snapshot")
print("-" * 70)

split_source_row_count = len(cohort_binary)
split_source_index = cohort_binary.index.copy()

assert split_source_row_count == 1_554_329
assert split_source_index.is_unique

print(f"Rows                  : {split_source_row_count:,}")
print(f"Unique patients       : {cohort_binary['anon_id'].nunique():,}")
print(f"Unique encounters     : {cohort_binary['pat_enc_csn_id_coded'].nunique():,}")
print(f"Unique orders         : {cohort_binary['order_proc_id_coded'].nunique():,}")

# ----------------------------------------------------------------------
# 3. Target integrity before split
# ----------------------------------------------------------------------

print("\n[3] Target integrity before split")
print("-" * 70)

assert cohort_binary[target_column].notna().all()
assert cohort_binary[target_column].isin([0, 1]).all()

source_target_counts = (
    cohort_binary[target_column]
    .value_counts()
    .sort_index()
)

source_target_percent = (
    cohort_binary[target_column]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

source_target_summary = pd.DataFrame({
    "count": source_target_counts,
    "percentage": source_target_percent
})

print(source_target_summary.to_string())

assert len(source_target_counts) == 2

# ----------------------------------------------------------------------
# 4. Group integrity verification
# ----------------------------------------------------------------------

print("\n[4] Group definition")
print("-" * 70)

group_column = "anon_id"

print(f"Grouping unit : {group_column}")
print(
    "All rows belonging to the same patient will remain in exactly "
    "one partition."
)

assert cohort_binary[group_column].notna().all()

n_groups = cohort_binary[group_column].nunique()

assert n_groups == 67_007

print(f"Unique patient groups : {n_groups:,}")
print("✓ Patient grouping column verified")

# ----------------------------------------------------------------------
# 5. Create stratified group folds
# ----------------------------------------------------------------------

print("\n[5] Creating stratified patient-group folds")
print("-" * 70)

# Five folds gives approximately:
#
#   TRAIN      = 3 folds = ~60%
#   VALIDATION = 1 fold  = ~20%
#   TEST       = 1 fold  = ~20%
#
# Stratification attempts to maintain the binary target distribution
# while keeping every patient entirely within one fold.

X_split = cohort_binary[candidate_features].copy()
y_split = cohort_binary[target_column].copy()
groups_split = cohort_binary[group_column].copy()

assert len(X_split) == len(y_split)
assert len(y_split) == len(groups_split)

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold_assignments = np.full(
    len(cohort_binary),
    fill_value=-1,
    dtype=np.int8,
)

fold_sizes = {}

for fold_number, (_, fold_index) in enumerate(
    sgkf.split(
        X_split,
        y_split,
        groups=groups_split,
    )
):
    assert len(fold_index) > 0

    fold_assignments[fold_index] = fold_number

    fold_sizes[fold_number] = len(fold_index)

assert (fold_assignments >= 0).all()
assert set(np.unique(fold_assignments)) == {0, 1, 2, 3, 4}

print("✓ Five patient-group folds created")

print("\nFold row counts:")
for fold_number in range(5):
    print(
        f"  Fold {fold_number}: "
        f"{fold_sizes[fold_number]:,} rows"
    )

# ----------------------------------------------------------------------
# 6. Assign folds to TRAIN / VALIDATION / TEST
# ----------------------------------------------------------------------

print("\n[6] Assigning TRAIN / VALIDATION / TEST")
print("-" * 70)

# Three folds -> TRAIN
# One fold   -> VALIDATION
# One fold   -> TEST
#
# The specific fold assignment is deterministic because random_state
# is fixed above.

train_folds = {0, 1, 2}
validation_fold = 3
test_fold = 4

assert train_folds.isdisjoint({validation_fold})
assert train_folds.isdisjoint({test_fold})
assert validation_fold != test_fold

train_mask = np.isin(
    fold_assignments,
    list(train_folds)
)

validation_mask = (
    fold_assignments == validation_fold
)

test_mask = (
    fold_assignments == test_fold
)

assert train_mask.sum() > 0
assert validation_mask.sum() > 0
assert test_mask.sum() > 0

assert (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
    == 1
).all()

print(
    f"TRAIN rows      : {train_mask.sum():,}"
)
print(
    f"VALIDATION rows : {validation_mask.sum():,}"
)
print(
    f"TEST rows       : {test_mask.sum():,}"
)

# ----------------------------------------------------------------------
# 7. Construct partition DataFrames
# ----------------------------------------------------------------------

print("\n[7] Constructing partition DataFrames")
print("-" * 70)

# Preserve all currently available columns.
# Feature selection / preprocessing happens later.

X_train_raw = cohort_binary.loc[
    train_mask
].copy()

X_validation_raw = cohort_binary.loc[
    validation_mask
].copy()

X_test_raw = cohort_binary.loc[
    test_mask
].copy()

assert len(X_train_raw) + len(X_validation_raw) + len(X_test_raw) == len(
    cohort_binary
)

print(f"X_train_raw      : {X_train_raw.shape}")
print(f"X_validation_raw : {X_validation_raw.shape}")
print(f"X_test_raw       : {X_test_raw.shape}")

# ----------------------------------------------------------------------
# 8. Extract target vectors
# ----------------------------------------------------------------------

print("\n[8] Constructing target vectors")
print("-" * 70)

y_train_raw = X_train_raw[target_column].copy()
y_validation_raw = X_validation_raw[target_column].copy()
y_test_raw = X_test_raw[target_column].copy()

assert len(X_train_raw) == len(y_train_raw)
assert len(X_validation_raw) == len(y_validation_raw)
assert len(X_test_raw) == len(y_test_raw)

assert y_train_raw.isin([0, 1]).all()
assert y_validation_raw.isin([0, 1]).all()
assert y_test_raw.isin([0, 1]).all()

print(f"y_train_raw      : {y_train_raw.shape}")
print(f"y_validation_raw : {y_validation_raw.shape}")
print(f"y_test_raw       : {y_test_raw.shape}")

# ----------------------------------------------------------------------
# 9. Target distribution by partition
# ----------------------------------------------------------------------

print("\n[9] Target distribution by partition")
print("-" * 70)

def target_summary(y, partition_name):
    counts = y.value_counts().sort_index()
    percentages = (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    result = pd.DataFrame({
        "partition": partition_name,
        "class": counts.index,
        "count": counts.values,
        "percentage": percentages.values,
    })

    return result


train_target_summary = target_summary(
    y_train_raw,
    "TRAIN",
)

validation_target_summary = target_summary(
    y_validation_raw,
    "VALIDATION",
)

test_target_summary = target_summary(
    y_test_raw,
    "TEST",
)

partition_target_summary = pd.concat(
    [
        train_target_summary,
        validation_target_summary,
        test_target_summary,
    ],
    ignore_index=True,
)

print(
    partition_target_summary.to_string(index=False)
)

# Every partition must contain both classes.
assert y_train_raw.nunique() == 2
assert y_validation_raw.nunique() == 2
assert y_test_raw.nunique() == 2

# ----------------------------------------------------------------------
# 10. Patient-overlap audit
# ----------------------------------------------------------------------

print("\n[10] Patient-overlap verification")
print("-" * 70)

train_patients = set(
    X_train_raw[group_column].unique()
)

validation_patients = set(
    X_validation_raw[group_column].unique()
)

test_patients = set(
    X_test_raw[group_column].unique()
)

train_validation_overlap = (
    train_patients & validation_patients
)

train_test_overlap = (
    train_patients & test_patients
)

validation_test_overlap = (
    validation_patients & test_patients
)

print(
    f"TRAIN ∩ VALIDATION patients : "
    f"{len(train_validation_overlap):,}"
)

print(
    f"TRAIN ∩ TEST patients        : "
    f"{len(train_test_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST patients   : "
    f"{len(validation_test_overlap):,}"
)

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

assert (
    len(
        train_patients
        | validation_patients
        | test_patients
    )
    == n_groups
)

print("✓ ZERO patient overlap across partitions")

# ----------------------------------------------------------------------
# 11. Order-overlap audit
# ----------------------------------------------------------------------

print("\n[11] Order-overlap verification")
print("-" * 70)

train_orders = set(
    X_train_raw["order_proc_id_coded"].unique()
)

validation_orders = set(
    X_validation_raw["order_proc_id_coded"].unique()
)

test_orders = set(
    X_test_raw["order_proc_id_coded"].unique()
)

tv_order_overlap = train_orders & validation_orders
tt_order_overlap = train_orders & test_orders
vt_order_overlap = validation_orders & test_orders

print(
    f"TRAIN ∩ VALIDATION orders : "
    f"{len(tv_order_overlap):,}"
)

print(
    f"TRAIN ∩ TEST orders        : "
    f"{len(tt_order_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST orders   : "
    f"{len(vt_order_overlap):,}"
)

# Because the split is patient-grouped, order overlap should also be
# zero if an order belongs to one patient.

assert len(tv_order_overlap) == 0
assert len(tt_order_overlap) == 0
assert len(vt_order_overlap) == 0

print("✓ ZERO order overlap across partitions")

# ----------------------------------------------------------------------
# 12. Encounter-overlap audit
# ----------------------------------------------------------------------

print("\n[12] Encounter-overlap verification")
print("-" * 70)

train_encounters = set(
    X_train_raw["pat_enc_csn_id_coded"].unique()
)

validation_encounters = set(
    X_validation_raw["pat_enc_csn_id_coded"].unique()
)

test_encounters = set(
    X_test_raw["pat_enc_csn_id_coded"].unique()
)

tv_encounter_overlap = (
    train_encounters & validation_encounters
)

tt_encounter_overlap = (
    train_encounters & test_encounters
)

vt_encounter_overlap = (
    validation_encounters & test_encounters
)

print(
    f"TRAIN ∩ VALIDATION encounters : "
    f"{len(tv_encounter_overlap):,}"
)

print(
    f"TRAIN ∩ TEST encounters        : "
    f"{len(tt_encounter_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST encounters   : "
    f"{len(vt_encounter_overlap):,}"
)

assert len(tv_encounter_overlap) == 0
assert len(tt_encounter_overlap) == 0
assert len(vt_encounter_overlap) == 0

print("✓ ZERO encounter overlap across partitions")

# ----------------------------------------------------------------------
# 13. Source-data row accounting
# ----------------------------------------------------------------------

print("\n[13] Row-accounting verification")
print("-" * 70)

assert (
    len(X_train_raw)
    + len(X_validation_raw)
    + len(X_test_raw)
    == len(cohort_binary)
)

print(
    f"TRAIN + VALIDATION + TEST = "
    f"{len(X_train_raw) + len(X_validation_raw) + len(X_test_raw):,}"
)

print(
    f"Original binary cohort    = "
    f"{len(cohort_binary):,}"
)

assert (
    len(X_train_raw)
    + len(X_validation_raw)
    + len(X_test_raw)
    == 1_554_329
)

print("✓ Exact row accounting")

# ----------------------------------------------------------------------
# 14. Source cohort integrity
# ----------------------------------------------------------------------

print("\n[14] Source cohort integrity verification")
print("-" * 70)

assert len(cohort_binary) == split_source_row_count
assert cohort_binary.index.equals(split_source_index)

assert "X_train_raw" not in []  # explicit no-op for readability

print("✓ cohort_binary row count unchanged")
print("✓ cohort_binary index unchanged")
print("✓ Source cohort was not modified in-place")

# ----------------------------------------------------------------------
# 15. Partition size summary
# ----------------------------------------------------------------------

print("\n[15] Final partition summary")
print("-" * 70)

partition_summary = pd.DataFrame({
    "partition": [
        "TRAIN",
        "VALIDATION",
        "TEST",
    ],
    "rows": [
        len(X_train_raw),
        len(X_validation_raw),
        len(X_test_raw),
    ],
    "patients": [
        X_train_raw[group_column].nunique(),
        X_validation_raw[group_column].nunique(),
        X_test_raw[group_column].nunique(),
    ],
    "positive": [
        int(y_train_raw.sum()),
        int(y_validation_raw.sum()),
        int(y_test_raw.sum()),
    ],
    "positive_percent": [
        float(y_train_raw.mean() * 100),
        float(y_validation_raw.mean() * 100),
        float(y_test_raw.mean() * 100),
    ],
})

print(
    partition_summary.to_string(index=False)
)

# ----------------------------------------------------------------------
# 16. Final assertions
# ----------------------------------------------------------------------

print("\n[16] Final assertions")
print("-" * 70)

assert isinstance(X_train_raw, pd.DataFrame)
assert isinstance(X_validation_raw, pd.DataFrame)
assert isinstance(X_test_raw, pd.DataFrame)

assert isinstance(y_train_raw, pd.Series)
assert isinstance(y_validation_raw, pd.Series)
assert isinstance(y_test_raw, pd.Series)

assert len(X_train_raw) == len(y_train_raw)
assert len(X_validation_raw) == len(y_validation_raw)
assert len(X_test_raw) == len(y_test_raw)

assert y_train_raw.isin([0, 1]).all()
assert y_validation_raw.isin([0, 1]).all()
assert y_test_raw.isin([0, 1]).all()

assert y_train_raw.nunique() == 2
assert y_validation_raw.nunique() == 2
assert y_test_raw.nunique() == 2

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

assert len(tv_order_overlap) == 0
assert len(tt_order_overlap) == 0
assert len(vt_order_overlap) == 0

assert len(tv_encounter_overlap) == 0
assert len(tt_encounter_overlap) == 0
assert len(vt_encounter_overlap) == 0

assert (
    len(X_train_raw)
    + len(X_validation_raw)
    + len(X_test_raw)
    == len(cohort_binary)
)

print("✓ All split assertions passed")

# ----------------------------------------------------------------------
# 17. Final methodological record
# ----------------------------------------------------------------------

split_methodology = {
    "split_method": "StratifiedGroupKFold",
    "n_folds": 5,
    "train_folds": sorted(train_folds),
    "validation_fold": validation_fold,
    "test_fold": test_fold,
    "group_column": group_column,
    "random_state": 42,
    "approximate_ratio": "60% / 20% / 20%",
    "group_overlap_allowed": False,
}

print("\n[17] Split methodology")
print("-" * 70)

for key, value in split_methodology.items():
    print(f"{key:<28}: {value}")

print("\n" + "=" * 70)
print("STEP 6 GROUP-AWARE SPLIT COMPLETE")
print("=" * 70)

print(
    "\nIMPORTANT:"
)
print(
    "TRAIN, VALIDATION, and TEST are now patient-disjoint."
)
print(
    "No preprocessing was fitted."
)
print(
    "No categorical mapping was fitted."
)
print(
    "No SMOTENC was performed."
)
print(
    "No model was trained."
)

STEP 6 — GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT

[0] Required-object verification
----------------------------------------------------------------------
✓ cohort_binary exists
✓ candidate_features exists
✓ target_column exists

[1] Required-column verification
----------------------------------------------------------------------
✓ All split-required columns exist

[2] Pre-split integrity snapshot
----------------------------------------------------------------------
Rows                  : 1,554,329
Unique patients       : 67,007
Unique encounters     : 116,994
Unique orders         : 118,737

[3] Target integrity before split
----------------------------------------------------------------------
              count  percentage
amr_target                     
0           1289258   82.946275
1            265071   17.053725

[4] Group definition
----------------------------------------------------------------------
Grouping unit : anon_id
All rows belonging to the same patient will

In [17]:
# ======================================================================
# STEP 7 — PREDICTION-TIME DECISION + FINAL FEATURE SET
# ======================================================================

print("=" * 70)
print("STEP 7 — PREDICTION-TIME DECISION")
print("=" * 70)

# ----------------------------------------------------------------------
# [0] Required-object verification
# ----------------------------------------------------------------------

required_objects = [
    "cohort_binary",
    "candidate_features",
    "target_column",
]

for obj_name in required_objects:
    assert obj_name in globals(), f"Required object missing: {obj_name}"
    print(f"✓ {obj_name} exists")

# ----------------------------------------------------------------------
# [1] Declare the prediction-time definition
# ----------------------------------------------------------------------
#
# DECISION:
# Organism is assumed to be known/available at the prediction point.
#
# Therefore this project is explicitly an:
# organism-conditioned, antibiotic-specific AMR prediction task.
#
# This is NOT a pre-culture / pre-organism-identification prediction task.
# ----------------------------------------------------------------------

prediction_time_definition = "ORGANISM_AVAILABLE"

prediction_time_statement = (
    "Organism is known/available at prediction time. "
    "The model predicts susceptibility/resistance for a specified "
    "organism-antibiotic combination."
)

print("\nPrediction-time definition:")
print("  ORGANISM_AVAILABLE")
print("\nClinical task definition:")
print(f"  {prediction_time_statement}")

# ----------------------------------------------------------------------
# [2] Verify candidate features
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("Candidate features before final exclusion")
print("-" * 70)

for feature in candidate_features:
    print(f"  → {feature}")

# ----------------------------------------------------------------------
# [3] was_positive audit
# ----------------------------------------------------------------------

assert "was_positive" in cohort_binary.columns

was_positive_unique = cohort_binary["was_positive"].nunique(dropna=False)

print("\n" + "-" * 70)
print("was_positive decision")
print("-" * 70)

print(f"Unique values: {was_positive_unique}")

if was_positive_unique == 1:
    print("✓ was_positive is constant and will be excluded.")
else:
    raise RuntimeError(
        "was_positive is no longer constant. "
        "Re-audit prediction-time validity before proceeding."
    )

# ----------------------------------------------------------------------
# [4] Final prediction-time feature decision
# ----------------------------------------------------------------------

final_model_features = [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

excluded_model_features = {
    "was_positive": "Constant feature; no predictive variance.",
    "anon_id": "Patient identifier; excluded to prevent memorization.",
    "pat_enc_csn_id_coded": "Encounter identifier; excluded.",
    "order_proc_id_coded": "Order identifier; excluded.",
    "order_time_jittered_utc": "Temporal metadata; excluded from current declared feature design.",
    "susceptibility": "Source column for the target; direct target leakage.",
    "amr_target": "Outcome variable; cannot be a predictor.",
}

# ----------------------------------------------------------------------
# [5] Verify final feature set
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL MODEL FEATURES")
print("-" * 70)

for feature in final_model_features:
    assert feature in cohort_binary.columns, (
        f"Final feature missing from cohort_binary: {feature}"
    )
    assert feature != target_column, (
        f"Target accidentally included as predictor: {feature}"
    )
    assert feature != "susceptibility", (
        "Raw target-source column accidentally included."
    )

    print(f"  ✓ {feature}")

# ----------------------------------------------------------------------
# [6] Explicit leakage verification
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("LEAKAGE / IDENTIFIER VERIFICATION")
print("-" * 70)

for prohibited_feature in [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "order_time_jittered_utc",
    "susceptibility",
    "amr_target",
    "was_positive",
]:
    assert prohibited_feature not in final_model_features, (
        f"Prohibited feature found in final model features: "
        f"{prohibited_feature}"
    )

print("✓ Patient identifier excluded")
print("✓ Encounter identifier excluded")
print("✓ Order identifier excluded")
print("✓ Timestamp excluded")
print("✓ Raw susceptibility excluded")
print("✓ Target excluded")
print("✓ Constant was_positive excluded")

# ----------------------------------------------------------------------
# [7] Organism decision verification
# ----------------------------------------------------------------------

assert prediction_time_definition == "ORGANISM_AVAILABLE"
assert "organism" in final_model_features

print("\n" + "-" * 70)
print("ORGANISM PREDICTION-TIME DECISION")
print("-" * 70)
print("✓ ORGANISM is declared available at prediction time.")
print("✓ ORGANISM is therefore retained as a predictor.")
print("✓ This is an organism-conditioned AMR prediction task.")

# ----------------------------------------------------------------------
# [8] Antibiotic decision verification
# ----------------------------------------------------------------------

assert "antibiotic" in final_model_features

print("\n" + "-" * 70)
print("ANTIBIOTIC PREDICTION-TIME DECISION")
print("-" * 70)
print("✓ ANTIBIOTIC is retained as the antibiotic being evaluated.")
print("✓ The model predicts AMR status for the specified antibiotic.")

# ----------------------------------------------------------------------
# [9] Create explicit feature-status record
# ----------------------------------------------------------------------

feature_status = {
    "was_positive": "EXCLUDED_CONSTANT",
    "ordering_mode": "ALLOWED",
    "culture_description": "ALLOWED",
    "organism": "ALLOWED_ORGANISM_AVAILABLE",
    "antibiotic": "ALLOWED_ANTIBIOTIC_SPECIFIC",
}

# ----------------------------------------------------------------------
# [10] Create explicit prediction-time record
# ----------------------------------------------------------------------

prediction_time_record = {
    "prediction_time_definition": prediction_time_definition,
    "organism_available": True,
    "antibiotic_available": True,
    "task_type": "organism_conditioned_antibiotic_specific_AMR_prediction",
    "pre_culture_prediction": False,
}

# ----------------------------------------------------------------------
# [11] Final feature matrix preview
# ----------------------------------------------------------------------

X_model_raw = cohort_binary[final_model_features].copy()
y_model = cohort_binary[target_column].copy()

print("\n" + "-" * 70)
print("FINAL RAW MODEL TABLE")
print("-" * 70)

print(f"X_model_raw shape : {X_model_raw.shape}")
print(f"y_model shape     : {y_model.shape}")

print("\nFinal features:")
for feature in final_model_features:
    print(f"  → {feature}")

# ----------------------------------------------------------------------
# [12] Split-specific feature tables
# ----------------------------------------------------------------------
#
# IMPORTANT:
# These are still RAW categorical values.
# No encoder has been fitted.
# No imputation has been fitted.
# No SMOTENC has been performed.
# ----------------------------------------------------------------------

X_train_model_raw = X_train_raw[final_model_features].copy()
X_validation_model_raw = X_validation_raw[final_model_features].copy()
X_test_model_raw = X_test_raw[final_model_features].copy()

print("\nSplit-specific raw feature shapes:")
print(f"  X_train_model_raw      : {X_train_model_raw.shape}")
print(f"  X_validation_model_raw: {X_validation_model_raw.shape}")
print(f"  X_test_model_raw      : {X_test_model_raw.shape}")

# ----------------------------------------------------------------------
# [13] Final target verification
# ----------------------------------------------------------------------

assert len(X_train_model_raw) == len(y_train_raw)
assert len(X_validation_model_raw) == len(y_validation_raw)
assert len(X_test_model_raw) == len(y_test_raw)

assert set(y_train_raw.unique()).issubset({0, 1})
assert set(y_validation_raw.unique()).issubset({0, 1})
assert set(y_test_raw.unique()).issubset({0, 1})

# ----------------------------------------------------------------------
# [14] Final assertions
# ----------------------------------------------------------------------

assert len(final_model_features) == 4

assert final_model_features == [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

assert feature_status["organism"] == "ALLOWED_ORGANISM_AVAILABLE"
assert feature_status["antibiotic"] == "ALLOWED_ANTIBIOTIC_SPECIFIC"
assert feature_status["was_positive"] == "EXCLUDED_CONSTANT"

print("\n" + "=" * 70)
print("STEP 7 PREDICTION-TIME DECISION COMPLETE")
print("=" * 70)

print("\nDECLARED PREDICTION TASK:")
print("  Organism-conditioned, antibiotic-specific binary AMR prediction.")

print("\nTARGET:")
print("  Susceptible → 0")
print("  Resistant   → 1")

print("\nFINAL MODEL FEATURES:")
for feature in final_model_features:
    print(f"  ✓ {feature}")

print("\nEXCLUDED:")
print("  ✗ was_positive — constant")
print("  ✗ anon_id — identifier")
print("  ✗ pat_enc_csn_id_coded — identifier")
print("  ✗ order_proc_id_coded — identifier")
print("  ✗ order_time_jittered_utc — temporal metadata")
print("  ✗ susceptibility — target source")
print("  ✗ amr_target — target")

print("\nIMPORTANT:")
print("  No categorical encoding was performed.")
print("  No preprocessing was performed.")
print("  No SMOTENC was performed.")
print("  No model was trained.")

print("\n✓ All STEP 7 assertions passed.")

STEP 7 — PREDICTION-TIME DECISION
✓ cohort_binary exists
✓ candidate_features exists
✓ target_column exists

Prediction-time definition:
  ORGANISM_AVAILABLE

Clinical task definition:
  Organism is known/available at prediction time. The model predicts susceptibility/resistance for a specified organism-antibiotic combination.

----------------------------------------------------------------------
Candidate features before final exclusion
----------------------------------------------------------------------
  → was_positive
  → ordering_mode
  → culture_description
  → organism
  → antibiotic

----------------------------------------------------------------------
was_positive decision
----------------------------------------------------------------------
Unique values: 1
✓ was_positive is constant and will be excluded.

----------------------------------------------------------------------
FINAL MODEL FEATURES
----------------------------------------------------------------------
  ✓ 

In [19]:
# ======================================================================
# STEP 8 — TRAIN-ONLY CATEGORICAL PREPROCESSING
# ======================================================================

print("=" * 70)
print("STEP 8 — TRAIN-ONLY CATEGORICAL PREPROCESSING")
print("=" * 70)

# ----------------------------------------------------------------------
# [0] Required-object verification
# ----------------------------------------------------------------------

required_objects = [
    "X_train_model_raw",
    "X_validation_model_raw",
    "X_test_model_raw",
    "y_train_raw",
    "y_validation_raw",
    "y_test_raw",
    "final_model_features",
    "target_column",
    "prediction_time_record",
]

for obj_name in required_objects:
    assert obj_name in globals(), f"Required object missing: {obj_name}"
    print(f"✓ {obj_name} exists")

# ----------------------------------------------------------------------
# [1] Feature-order verification
# ----------------------------------------------------------------------

expected_features = [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic",
]

assert final_model_features == expected_features

print("\n" + "-" * 70)
print("FINAL FEATURE ORDER")
print("-" * 70)

for i, feature in enumerate(final_model_features):
    print(f"  [{i}] {feature}")

# ----------------------------------------------------------------------
# [2] Raw input integrity verification
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("RAW INPUT SHAPES")
print("-" * 70)

print(f"X_train_model_raw       : {X_train_model_raw.shape}")
print(f"X_validation_model_raw  : {X_validation_model_raw.shape}")
print(f"X_test_model_raw       : {X_test_model_raw.shape}")

assert list(X_train_model_raw.columns) == expected_features
assert list(X_validation_model_raw.columns) == expected_features
assert list(X_test_model_raw.columns) == expected_features

assert len(X_train_model_raw) == len(y_train_raw)
assert len(X_validation_model_raw) == len(y_validation_raw)
assert len(X_test_model_raw) == len(y_test_raw)

print("✓ Feature ordering identical across all partitions")
print("✓ Row counts match target vectors")

# ----------------------------------------------------------------------
# [3] Confirm all predictors are categorical
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("RAW FEATURE TYPES")
print("-" * 70)

for feature in expected_features:
    print(
        f"{feature:22s} | "
        f"TRAIN dtype={X_train_model_raw[feature].dtype} | "
        f"TRAIN unique={X_train_model_raw[feature].nunique(dropna=False)}"
    )

# ----------------------------------------------------------------------
# [4] Verify no pandas NaN values are present
# ----------------------------------------------------------------------
#
# Literal values such as "Null" are retained as legitimate categories.
# We do not reinterpret them as missing values in this step.
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("MISSING-VALUE VERIFICATION")
print("-" * 70)

for feature in expected_features:
    train_missing = X_train_model_raw[feature].isna().sum()
    val_missing = X_validation_model_raw[feature].isna().sum()
    test_missing = X_test_model_raw[feature].isna().sum()

    print(
        f"{feature:22s} | "
        f"TRAIN={train_missing} | "
        f"VAL={val_missing} | "
        f"TEST={test_missing}"
    )

    assert train_missing == 0
    assert val_missing == 0
    assert test_missing == 0

print("✓ No pandas NaN values detected")
print("✓ Literal category values remain unchanged")

# ----------------------------------------------------------------------
# [5] Snapshot raw category sets BEFORE preprocessing
# ----------------------------------------------------------------------

train_raw_categories = {}
validation_raw_categories = {}
test_raw_categories = {}

for feature in expected_features:
    train_raw_categories[feature] = set(
        X_train_model_raw[feature].astype(str).unique()
    )
    validation_raw_categories[feature] = set(
        X_validation_model_raw[feature].astype(str).unique()
    )
    test_raw_categories[feature] = set(
        X_test_model_raw[feature].astype(str).unique()
    )

# ----------------------------------------------------------------------
# [6] Fit OrdinalEncoder on TRAIN ONLY
# ----------------------------------------------------------------------
#
# IMPORTANT:
# Validation and TEST are NEVER used during fitting.
#
# Unknown categories receive -1 from OrdinalEncoder.
# We subsequently shift all encoded values by +1:
#
#     known categories : 1, 2, 3, ...
#     unknown category  : 0
#
# This produces non-negative integer categorical codes suitable for
# downstream categorical processing and SMOTENC.
# ----------------------------------------------------------------------

from sklearn.preprocessing import OrdinalEncoder
import numpy as np
import pandas as pd

print("\n" + "-" * 70)
print("FITTING TRAIN-ONLY ORDINAL ENCODER")
print("-" * 70)

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int64,
)

# FIT ONLY ON TRAIN
categorical_encoder.fit(
    X_train_model_raw[expected_features].astype(str)
)

print("✓ Encoder fitted using TRAIN only")
print("✓ Validation data was not used during fitting")
print("✓ Test data was not used during fitting")

# ----------------------------------------------------------------------
# [7] Record TRAIN-fitted category mappings
# ----------------------------------------------------------------------

train_category_mapping = {}

print("\n" + "-" * 70)
print("TRAIN-FITTED CATEGORY MAPPINGS")
print("-" * 70)

for feature, categories in zip(
    expected_features,
    categorical_encoder.categories_
):
    train_category_mapping[feature] = list(categories)

    print(
        f"{feature:22s} : "
        f"{len(categories)} TRAIN categories"
    )

# ----------------------------------------------------------------------
# [8] Identify unseen categories in validation/test
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("UNSEEN CATEGORY AUDIT")
print("-" * 70)

unseen_category_audit = {}

for feature in expected_features:

    train_categories = train_raw_categories[feature]

    unseen_validation = (
        validation_raw_categories[feature] - train_categories
    )

    unseen_test = (
        test_raw_categories[feature] - train_categories
    )

    unseen_category_audit[feature] = {
        "validation_unseen_categories": sorted(unseen_validation),
        "test_unseen_categories": sorted(unseen_test),
        "validation_unseen_category_count": len(unseen_validation),
        "test_unseen_category_count": len(unseen_test),
    }

    print(f"\n{feature}")
    print(
        f"  Validation unseen categories : "
        f"{len(unseen_validation)}"
    )
    print(
        f"  Test unseen categories       : "
        f"{len(unseen_test)}"
    )

print("\n✓ Unseen categories will be handled without refitting the encoder")

# ----------------------------------------------------------------------
# [9] Transform TRAIN / VALIDATION / TEST
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TRANSFORMING PARTITIONS")
print("-" * 70)

X_train_encoded_temp = categorical_encoder.transform(
    X_train_model_raw[expected_features].astype(str)
)

X_validation_encoded_temp = categorical_encoder.transform(
    X_validation_model_raw[expected_features].astype(str)
)

X_test_encoded_temp = categorical_encoder.transform(
    X_test_model_raw[expected_features].astype(str)
)

# ----------------------------------------------------------------------
# [10] Convert unknown code -1 → 0
# ----------------------------------------------------------------------
#
# Known categories are shifted:
#
# original encoder:
#     0, 1, 2, ...
#
# final representation:
#     1, 2, 3, ...
#
# Unknown categories:
#     -1 → 0
#
# Therefore every categorical feature has non-negative integer codes.
# ----------------------------------------------------------------------

X_train_encoded = X_train_encoded_temp + 1
X_validation_encoded = X_validation_encoded_temp + 1
X_test_encoded = X_test_encoded_temp + 1

# ----------------------------------------------------------------------
# [11] Build DataFrames
# ----------------------------------------------------------------------

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=expected_features,
    index=X_train_model_raw.index,
).astype(np.int64)

X_validation_encoded = pd.DataFrame(
    X_validation_encoded,
    columns=expected_features,
    index=X_validation_model_raw.index,
).astype(np.int64)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=expected_features,
    index=X_test_model_raw.index,
).astype(np.int64)

# ----------------------------------------------------------------------
# [12] Verify encoded values
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("ENCODED FEATURE AUDIT")
print("-" * 70)

for feature in expected_features:

    train_min = X_train_encoded[feature].min()
    train_max = X_train_encoded[feature].max()

    val_min = X_validation_encoded[feature].min()
    val_max = X_validation_encoded[feature].max()

    test_min = X_test_encoded[feature].min()
    test_max = X_test_encoded[feature].max()

    print(
        f"{feature:22s} | "
        f"TRAIN [{train_min}, {train_max}] | "
        f"VAL [{val_min}, {val_max}] | "
        f"TEST [{test_min}, {test_max}]"
    )

    assert train_min >= 1
    assert val_min >= 0
    assert test_min >= 0

    assert pd.api.types.is_integer_dtype(
        X_train_encoded[feature]
    )
    assert pd.api.types.is_integer_dtype(
        X_validation_encoded[feature]
    )
    assert pd.api.types.is_integer_dtype(
        X_test_encoded[feature]
    )

print("\n✓ All encoded predictors are integer-valued")
print("✓ TRAIN known categories start at code 1")
print("✓ Unknown validation/test categories are represented by code 0")
print("✓ No negative categorical codes remain")

# ----------------------------------------------------------------------
# [13] Target integrity verification
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("TARGET INTEGRITY")
print("-" * 70)

assert y_train_raw.dtype == np.int8
assert y_validation_raw.dtype == np.int8
assert y_test_raw.dtype == np.int8

assert set(y_train_raw.unique()).issubset({0, 1})
assert set(y_validation_raw.unique()).issubset({0, 1})
assert set(y_test_raw.unique()).issubset({0, 1})

print("✓ TRAIN target remains binary")
print("✓ VALIDATION target remains binary")
print("✓ TEST target remains binary")

# ----------------------------------------------------------------------
# [14] Row-accounting verification
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("ROW ACCOUNTING")
print("-" * 70)

print(f"TRAIN       : {len(X_train_encoded):,}")
print(f"VALIDATION  : {len(X_validation_encoded):,}")
print(f"TEST        : {len(X_test_encoded):,}")

assert len(X_train_encoded) == 934500
assert len(X_validation_encoded) == 308772
assert len(X_test_encoded) == 311057

assert len(X_train_encoded) == len(y_train_raw)
assert len(X_validation_encoded) == len(y_validation_raw)
assert len(X_test_encoded) == len(y_test_raw)

print("✓ Exact row counts preserved")
print("✓ No rows were created or deleted")

# ----------------------------------------------------------------------
# [15] Index integrity verification
# ----------------------------------------------------------------------

assert X_train_encoded.index.equals(X_train_model_raw.index)
assert X_validation_encoded.index.equals(
    X_validation_model_raw.index
)
assert X_test_encoded.index.equals(X_test_model_raw.index)

print("✓ Original partition indices preserved")

# ----------------------------------------------------------------------
# [16] Confirm validation/test did NOT influence encoder
# ----------------------------------------------------------------------

# Reconstruct category sets from the fitted encoder.
fitted_train_category_sets = {
    feature: set(categories)
    for feature, categories in zip(
        expected_features,
        categorical_encoder.categories_
    )
}

for feature in expected_features:
    assert fitted_train_category_sets[feature] == (
        train_raw_categories[feature]
    )

print("\n✓ Encoder category sets exactly match TRAIN categories")
print("✓ Validation/test categories did not alter fitted mappings")

# ----------------------------------------------------------------------
# [17] Preserve target vectors under explicit names
# ----------------------------------------------------------------------

y_train = y_train_raw.copy()
y_validation = y_validation_raw.copy()
y_test = y_test_raw.copy()

# ----------------------------------------------------------------------
# [18] Explicit SMOTENC status
# ----------------------------------------------------------------------
#
# SMOTENC MUST NOT occur in STEP 8.
# It will be performed in the next dedicated step on TRAIN ONLY.
# ----------------------------------------------------------------------

smotenc_performed = False

assert smotenc_performed is False

# ----------------------------------------------------------------------
# [19] Preprocessing record
# ----------------------------------------------------------------------

preprocessing_record = {
    "method": "OrdinalEncoder",
    "fit_partition": "TRAIN_ONLY",
    "validation_used_for_fit": False,
    "test_used_for_fit": False,
    "handle_unknown": "use_encoded_value",
    "unknown_initial_code": -1,
    "unknown_final_code": 0,
    "known_category_codes_start": 1,
    "features": expected_features.copy(),
    "smotenc_performed": False,
}

# ----------------------------------------------------------------------
# [20] Final assertions
# ----------------------------------------------------------------------

assert preprocessing_record["fit_partition"] == "TRAIN_ONLY"
assert preprocessing_record["validation_used_for_fit"] is False
assert preprocessing_record["test_used_for_fit"] is False
assert preprocessing_record["smotenc_performed"] is False

assert list(X_train_encoded.columns) == expected_features
assert list(X_validation_encoded.columns) == expected_features
assert list(X_test_encoded.columns) == expected_features

assert X_train_encoded.shape == (934500, 4)
assert X_validation_encoded.shape == (308772, 4)
assert X_test_encoded.shape == (311057, 4)

print("\n" + "=" * 70)
print("STEP 8 TRAIN-ONLY PREPROCESSING COMPLETE")
print("=" * 70)

print("\nENCODING METHOD:")
print("  OrdinalEncoder fitted on TRAIN only")

print("\nFINAL ENCODED MATRICES:")
print(f"  X_train_encoded       : {X_train_encoded.shape}")
print(f"  X_validation_encoded : {X_validation_encoded.shape}")
print(f"  X_test_encoded       : {X_test_encoded.shape}")

print("\nFEATURES:")
for i, feature in enumerate(expected_features):
    print(f"  [{i}] {feature}")

print("\nENCODING RULE:")
print("  Known TRAIN categories → integer codes starting at 1")
print("  Unseen VAL/TEST categories → code 0")

print("\nDATA USAGE:")
print("  ✓ TRAIN used to fit encoder")
print("  ✓ VALIDATION transformed only")
print("  ✓ TEST transformed only")
print("  ✓ No validation/test fitting")
print("  ✓ Target untouched")
print("  ✓ Row counts unchanged")
print("  ✓ Indices preserved")
print("  ✓ SMOTENC NOT performed")

print("\n✓ All STEP 8 assertions passed.")

STEP 8 — TRAIN-ONLY CATEGORICAL PREPROCESSING
✓ X_train_model_raw exists
✓ X_validation_model_raw exists
✓ X_test_model_raw exists
✓ y_train_raw exists
✓ y_validation_raw exists
✓ y_test_raw exists
✓ final_model_features exists
✓ target_column exists
✓ prediction_time_record exists

----------------------------------------------------------------------
FINAL FEATURE ORDER
----------------------------------------------------------------------
  [0] ordering_mode
  [1] culture_description
  [2] organism
  [3] antibiotic

----------------------------------------------------------------------
RAW INPUT SHAPES
----------------------------------------------------------------------
X_train_model_raw       : (934500, 4)
X_validation_model_raw  : (308772, 4)
X_test_model_raw       : (311057, 4)
✓ Feature ordering identical across all partitions
✓ Row counts match target vectors

----------------------------------------------------------------------
RAW FEATURE TYPES
----------------------------

In [26]:
# ================================================================
# STEP 9 — TRAIN-ONLY MEMORY-CONTROLLED SMOTEN
# ================================================================
#
# PURPOSE:
#   Apply SMOTEN to TRAIN only without attempting the enormous
#   158,388 x 158,388 distance matrix.
#
# IMPORTANT:
#   - Complete original TRAIN data is retained.
#   - Only a controlled subset of minority TRAIN observations
#     is used as the SMOTEN fitting population.
#   - Synthetic observations are then added to the complete TRAIN.
#   - VALIDATION and TEST are NEVER used.
# ================================================================

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTEN

print("=" * 70)
print("STEP 9 — TRAIN-ONLY MEMORY-CONTROLLED SMOTEN")
print("=" * 70)

# ------------------------------------------------
# [0] Required-object verification
# ------------------------------------------------

required_objects = [
    "X_train_encoded",
    "X_validation_encoded",
    "X_test_encoded",
    "y_train_raw",
    "y_validation_raw",
    "y_test_raw",
    "final_model_features",
    "target_column",
]

for obj_name in required_objects:
    assert obj_name in globals(), f"Missing required object: {obj_name}"
    print(f"✓ {obj_name} exists")

# ------------------------------------------------
# [1] Input verification
# ------------------------------------------------

print("\n" + "-" * 70)
print("[1] INPUT VERIFICATION")
print("-" * 70)

X_train_original = np.asarray(
    X_train_encoded,
    dtype=np.int32
)

y_train_original = np.asarray(
    y_train_raw,
    dtype=np.int8
)

X_validation_final = np.asarray(
    X_validation_encoded,
    dtype=np.int32
)

X_test_final = np.asarray(
    X_test_encoded,
    dtype=np.int32
)

y_validation_final = np.asarray(
    y_validation_raw,
    dtype=np.int8
)

y_test_final = np.asarray(
    y_test_raw,
    dtype=np.int8
)

print(f"X_train_original      : {X_train_original.shape}")
print(f"y_train_original      : {y_train_original.shape}")
print(f"X_validation_final    : {X_validation_final.shape}")
print(f"X_test_final          : {X_test_final.shape}")

assert X_train_original.shape[0] == len(y_train_original)
assert X_validation_final.shape[0] == len(y_validation_final)
assert X_test_final.shape[0] == len(y_test_final)

assert X_train_original.shape[1] == len(final_model_features)

print("✓ Input alignment verified")

# ------------------------------------------------
# [2] Verify categorical-only feature matrix
# ------------------------------------------------

print("\n" + "-" * 70)
print("[2] FEATURE VERIFICATION")
print("-" * 70)

print("Final features:")

for i, feature in enumerate(final_model_features):
    print(f"  [{i}] {feature}")

print("\n✓ All four predictors are categorical")
print("✓ SMOTEN is being used")

# ------------------------------------------------
# [3] Original TRAIN distribution
# ------------------------------------------------

print("\n" + "-" * 70)
print("[3] ORIGINAL TRAIN CLASS DISTRIBUTION")
print("-" * 70)

train_counts_before = (
    pd.Series(y_train_original)
    .value_counts()
    .sort_index()
)

print(train_counts_before)

majority_count = int(train_counts_before.loc[0])
minority_count = int(train_counts_before.loc[1])

print(f"\nSusceptible (0) : {majority_count:,}")
print(f"Resistant (1)   : {minority_count:,}")

# ------------------------------------------------
# [4] Memory-controlled SMOTEN configuration
# ------------------------------------------------
#
# IMPORTANT:
#
# SMOTEN's distance calculation scales very poorly with the
# number of minority samples used for fitting.
#
# Therefore we limit the minority fitting population.
#
# 5,000 minority samples require a distance matrix on the
# order of:
#
#     5,000 x 5,000 x 8 bytes ≈ 190 MB
#
# This is dramatically smaller than the original ~187 GiB.
#
# The complete original TRAIN set is retained afterwards.
# ------------------------------------------------

SMOTEN_MINORITY_FIT_SIZE = 5000

sampling_strategy = 0.50
k_neighbors = 5
random_state = 42

print("\n" + "-" * 70)
print("[4] MEMORY-CONTROLLED SMOTEN CONFIGURATION")
print("-" * 70)

print(f"Minority fitting population : {SMOTEN_MINORITY_FIT_SIZE:,}")
print(f"Sampling strategy            : {sampling_strategy}")
print(f"k_neighbors                  : {k_neighbors}")
print(f"random_state                 : {random_state}")

assert SMOTEN_MINORITY_FIT_SIZE > k_neighbors

# ------------------------------------------------
# [5] Identify minority and majority observations
# ------------------------------------------------

print("\n" + "-" * 70)
print("[5] IDENTIFYING TRAIN CLASSES")
print("-" * 70)

majority_indices = np.flatnonzero(
    y_train_original == 0
)

minority_indices = np.flatnonzero(
    y_train_original == 1
)

print(f"Majority TRAIN observations : {len(majority_indices):,}")
print(f"Minority TRAIN observations : {len(minority_indices):,}")

assert len(majority_indices) == majority_count
assert len(minority_indices) == minority_count

# ------------------------------------------------
# [6] Controlled minority sample
# ------------------------------------------------

print("\n" + "-" * 70)
print("[6] CREATING CONTROLLED SMOTEN FITTING POPULATION")
print("-" * 70)

rng = np.random.default_rng(random_state)

if len(minority_indices) > SMOTEN_MINORITY_FIT_SIZE:

    smoten_minority_indices = rng.choice(
        minority_indices,
        size=SMOTEN_MINORITY_FIT_SIZE,
        replace=False
    )

else:

    smoten_minority_indices = minority_indices.copy()

smoten_minority_indices = np.asarray(
    smoten_minority_indices,
    dtype=np.int64
)

print(
    f"Minority observations used by SMOTEN : "
    f"{len(smoten_minority_indices):,}"
)

assert len(smoten_minority_indices) >= k_neighbors + 1

# ------------------------------------------------
# [7] Build SMOTEN fitting dataset
# ------------------------------------------------
#
# We use:
#   all majority observations
#   controlled minority subset
#
# SMOTEN generates synthetic minority observations.
#
# The original TRAIN data is NOT discarded.
# ------------------------------------------------

print("\n" + "-" * 70)
print("[7] BUILDING SMOTEN FITTING DATA")
print("-" * 70)

X_majority_for_smoten = X_train_original[
    majority_indices
]

y_majority_for_smoten = y_train_original[
    majority_indices
]

X_minority_for_smoten = X_train_original[
    smoten_minority_indices
]

y_minority_for_smoten = y_train_original[
    smoten_minority_indices
]

X_smoten_input = np.vstack([
    X_majority_for_smoten,
    X_minority_for_smoten
])

y_smoten_input = np.concatenate([
    y_majority_for_smoten,
    y_minority_for_smoten
])

print(f"SMOTEN input shape : {X_smoten_input.shape}")
print(f"SMOTEN target size : {len(y_smoten_input):,}")

print(
    "\nSMOTEN fitting population:"
)

print(
    pd.Series(y_smoten_input)
    .value_counts()
    .sort_index()
)

# ------------------------------------------------
# [8] Create SMOTEN
# ------------------------------------------------

print("\n" + "-" * 70)
print("[8] CREATING SMOTEN")
print("-" * 70)

smoten = SMOTEN(
    sampling_strategy=sampling_strategy,
    k_neighbors=k_neighbors,
    random_state=random_state
)

print("✓ SMOTEN object created")

# ------------------------------------------------
# [9] FIT + RESAMPLE
# ------------------------------------------------

print("\n" + "-" * 70)
print("[9] FITTING SMOTEN")
print("-" * 70)

print("Starting memory-controlled SMOTEN...")
print("VALIDATION and TEST are not being used.")

X_smoten_resampled, y_smoten_resampled = (
    smoten.fit_resample(
        X_smoten_input,
        y_smoten_input
    )
)

print("✓ SMOTEN fitting completed")

# ------------------------------------------------
# [10] Extract ONLY synthetic minority observations
# ------------------------------------------------
#
# The SMOTEN input already contains:
#   all majority observations
#   controlled minority observations
#
# Therefore, the additional rows produced at the end
# are synthetic minority observations.
# ------------------------------------------------

print("\n" + "-" * 70)
print("[10] EXTRACTING SYNTHETIC MINORITY OBSERVATIONS")
print("-" * 70)

original_smoten_rows = len(X_smoten_input)

X_synthetic = X_smoten_resampled[
    original_smoten_rows:
]

y_synthetic = y_smoten_resampled[
    original_smoten_rows:
]

print(
    f"Synthetic feature rows : "
    f"{len(X_synthetic):,}"
)

print(
    f"Synthetic target rows  : "
    f"{len(y_synthetic):,}"
)

if len(y_synthetic) > 0:
    assert np.all(y_synthetic == 1)

print("✓ Synthetic rows are resistant-class observations")

# ------------------------------------------------
# [11] Combine synthetic observations with COMPLETE TRAIN
# ------------------------------------------------

print("\n" + "-" * 70)
print("[11] COMBINING SYNTHETIC DATA WITH COMPLETE TRAIN")
print("-" * 70)

X_train_final = np.vstack([
    X_train_original,
    X_synthetic
])

y_train_final = np.concatenate([
    y_train_original,
    y_synthetic
])

print(
    f"Original TRAIN rows : "
    f"{len(X_train_original):,}"
)

print(
    f"Synthetic rows      : "
    f"{len(X_synthetic):,}"
)

print(
    f"Final TRAIN rows    : "
    f"{len(X_train_final):,}"
)

assert len(X_train_final) == len(y_train_final)

# ------------------------------------------------
# [12] Final class distribution
# ------------------------------------------------

print("\n" + "-" * 70)
print("[12] FINAL TRAIN CLASS DISTRIBUTION")
print("-" * 70)

train_counts_after = (
    pd.Series(y_train_final)
    .value_counts()
    .sort_index()
)

train_percent_after = (
    train_counts_after /
    len(y_train_final) *
    100
)

distribution_after = pd.DataFrame({
    "count": train_counts_after,
    "percentage": train_percent_after
})

print(distribution_after)

final_majority = int(train_counts_after.loc[0])
final_minority = int(train_counts_after.loc[1])

final_ratio = (
    final_minority /
    final_majority
)

print(
    f"\nFinal minority/majority ratio : "
    f"{final_ratio:.4f}"
)

# ------------------------------------------------
# [13] Categorical-value verification
# ------------------------------------------------

print("\n" + "-" * 70)
print("[13] CATEGORICAL-VALUE VERIFICATION")
print("-" * 70)

for idx, feature in enumerate(final_model_features):

    values = X_train_final[:, idx]

    print(
        f"{feature:24s} | "
        f"min={values.min()} | "
        f"max={values.max()}"
    )

    assert np.all(values >= 0)

print("✓ No negative categorical codes")
print("✓ Categorical encoding remains integer-valued")

# ------------------------------------------------
# [14] VALIDATION / TEST untouched
# ------------------------------------------------

print("\n" + "-" * 70)
print("[14] VALIDATION / TEST INTEGRITY")
print("-" * 70)

print(
    f"Validation : "
    f"{X_validation_final.shape}"
)

print(
    f"Test       : "
    f"{X_test_final.shape}"
)

assert X_validation_final.shape == X_validation_encoded.shape
assert X_test_final.shape == X_test_encoded.shape

assert len(y_validation_final) == len(y_validation_raw)
assert len(y_test_final) == len(y_test_raw)

print("✓ VALIDATION untouched")
print("✓ TEST untouched")
print("✓ No synthetic validation observations")
print("✓ No synthetic test observations")

# ------------------------------------------------
# [15] Final assertions
# ------------------------------------------------

print("\n" + "-" * 70)
print("[15] FINAL ASSERTIONS")
print("-" * 70)

assert X_train_final.shape[1] == len(final_model_features)

assert X_validation_final.shape[1] == len(final_model_features)

assert X_test_final.shape[1] == len(final_model_features)

assert set(np.unique(y_train_final)).issubset({0, 1})
assert set(np.unique(y_validation_final)).issubset({0, 1})
assert set(np.unique(y_test_final)).issubset({0, 1})

assert len(X_train_final) > len(X_train_original)

print("✓ TRAIN feature count correct")
print("✓ VALIDATION feature count correct")
print("✓ TEST feature count correct")
print("✓ TRAIN target remains binary")
print("✓ VALIDATION target remains binary")
print("✓ TEST target remains binary")
print("✓ Synthetic TRAIN observations added")
print("✓ Complete original TRAIN retained")
print("✓ VALIDATION untouched")
print("✓ TEST untouched")

# ------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 9 TRAIN-ONLY MEMORY-CONTROLLED SMOTEN COMPLETE")
print("=" * 70)

print("\nFINAL DATA:")
print(f"  X_train_final       : {X_train_final.shape}")
print(f"  y_train_final       : {y_train_final.shape}")

print(f"  X_validation_final  : {X_validation_final.shape}")
print(f"  y_validation_final  : {y_validation_final.shape}")

print(f"  X_test_final        : {X_test_final.shape}")
print(f"  y_test_final        : {y_test_final.shape}")

print("\nSMOTEN:")
print(f"  Minority fitting population : {len(smoten_minority_indices):,}")
print(f"  Synthetic observations      : {len(X_synthetic):,}")
print(f"  Final class ratio           : {final_ratio:.4f}")

print("\nDATA USAGE:")
print("  ✓ TRAIN only used for SMOTEN")
print("  ✓ Complete original TRAIN retained")
print("  ✓ VALIDATION never used by SMOTEN")
print("  ✓ TEST never used by SMOTEN")
print("  ✓ No patient-group split changed")
print("  ✓ No target leakage introduced")

print("\n✓ All STEP 9 assertions passed.")

STEP 9 — TRAIN-ONLY MEMORY-CONTROLLED SMOTEN
✓ X_train_encoded exists
✓ X_validation_encoded exists
✓ X_test_encoded exists
✓ y_train_raw exists
✓ y_validation_raw exists
✓ y_test_raw exists
✓ final_model_features exists
✓ target_column exists

----------------------------------------------------------------------
[1] INPUT VERIFICATION
----------------------------------------------------------------------
X_train_original      : (934500, 4)
y_train_original      : (934500,)
X_validation_final    : (308772, 4)
X_test_final          : (311057, 4)
✓ Input alignment verified

----------------------------------------------------------------------
[2] FEATURE VERIFICATION
----------------------------------------------------------------------
Final features:
  [0] ordering_mode
  [1] culture_description
  [2] organism
  [3] antibiotic

✓ All four predictors are categorical
✓ SMOTEN is being used

----------------------------------------------------------------------
[3] ORIGINAL TRAIN CLASS 

In [27]:
# ================================================================
# STEP 10 — LIGHTGBM TRAINING DATA PREPARATION + BASELINE MODEL
# ================================================================

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("STEP 10 — LIGHTGBM TRAINING PREPARATION + BASELINE")
print("=" * 70)


# ----------------------------------------------------------------
# [0] REQUIRED-OBJECT VERIFICATION
# ----------------------------------------------------------------

required_objects = [
    "X_train_final",
    "y_train_final",
    "X_validation_final",
    "y_validation_final",
    "X_test_final",
    "y_test_final",
    "final_model_features",
    "target_column"
]

print("\n[0] REQUIRED-OBJECT VERIFICATION")
print("-" * 70)

for obj_name in required_objects:
    assert obj_name in globals(), f"Missing required object: {obj_name}"
    print(f"✓ {obj_name} exists")


# ----------------------------------------------------------------
# [1] SHAPE VERIFICATION
# ----------------------------------------------------------------

print("\n[1] DATA SHAPE VERIFICATION")
print("-" * 70)

print(f"X_train_final       : {X_train_final.shape}")
print(f"y_train_final       : {y_train_final.shape}")
print(f"X_validation_final  : {X_validation_final.shape}")
print(f"y_validation_final  : {y_validation_final.shape}")
print(f"X_test_final        : {X_test_final.shape}")
print(f"y_test_final        : {y_test_final.shape}")

assert X_train_final.shape[0] == len(y_train_final)
assert X_validation_final.shape[0] == len(y_validation_final)
assert X_test_final.shape[0] == len(y_test_final)

assert X_train_final.shape[1] == len(final_model_features)
assert X_validation_final.shape[1] == len(final_model_features)
assert X_test_final.shape[1] == len(final_model_features)

print("✓ Feature/target alignment verified")
print("✓ Feature counts verified")


# ----------------------------------------------------------------
# [2] FEATURE ORDER VERIFICATION
# ----------------------------------------------------------------

print("\n[2] FINAL FEATURE ORDER")
print("-" * 70)

for i, feature in enumerate(final_model_features):
    print(f"[{i}] {feature}")

assert list(final_model_features) == [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic"
]

print("✓ Feature order verified")


# ----------------------------------------------------------------
# [3] CONVERT TO DATAFRAMES
# ----------------------------------------------------------------

print("\n[3] CREATING LIGHTGBM DATAFRAMES")
print("-" * 70)

X_train_lgb = pd.DataFrame(
    X_train_final,
    columns=final_model_features
)

X_validation_lgb = pd.DataFrame(
    X_validation_final,
    columns=final_model_features
)

X_test_lgb = pd.DataFrame(
    X_test_final,
    columns=final_model_features
)

y_train_lgb = np.asarray(y_train_final).astype(np.int8)
y_validation_lgb = np.asarray(y_validation_final).astype(np.int8)
y_test_lgb = np.asarray(y_test_final).astype(np.int8)

print("✓ LightGBM DataFrames created")


# ----------------------------------------------------------------
# [4] DECLARE CATEGORICAL FEATURES
# ----------------------------------------------------------------

print("\n[4] CATEGORICAL FEATURE DECLARATION")
print("-" * 70)

categorical_features = list(final_model_features)

for feature in categorical_features:
    X_train_lgb[feature] = X_train_lgb[feature].astype("category")

    # IMPORTANT:
    # Validation/test categories are aligned to TRAIN categories.
    X_validation_lgb[feature] = pd.Categorical(
        X_validation_lgb[feature],
        categories=X_train_lgb[feature].cat.categories
    )

    X_test_lgb[feature] = pd.Categorical(
        X_test_lgb[feature],
        categories=X_train_lgb[feature].cat.categories
    )

    print(f"✓ {feature} declared categorical")


# ----------------------------------------------------------------
# [5] CATEGORICAL CODE VERIFICATION
# ----------------------------------------------------------------

print("\n[5] CATEGORICAL VALUE VERIFICATION")
print("-" * 70)

for feature in categorical_features:

    train_codes = X_train_lgb[feature].cat.codes
    val_codes = X_validation_lgb[feature].cat.codes
    test_codes = X_test_lgb[feature].cat.codes

    print(f"\n{feature}")
    print(f"  TRAIN categories : {len(X_train_lgb[feature].cat.categories)}")
    print(f"  TRAIN code range : {train_codes.min()} to {train_codes.max()}")
    print(f"  VAL code range   : {val_codes.min()} to {val_codes.max()}")
    print(f"  TEST code range  : {test_codes.min()} to {test_codes.max()}")

    assert train_codes.min() >= 0
    assert val_codes.min() >= -1
    assert test_codes.min() >= -1

print("\n✓ Categorical representations verified")


# ----------------------------------------------------------------
# [6] TARGET VERIFICATION
# ----------------------------------------------------------------

print("\n[6] TARGET VERIFICATION")
print("-" * 70)

print("TRAIN:")
print(pd.Series(y_train_lgb).value_counts().sort_index())

print("\nVALIDATION:")
print(pd.Series(y_validation_lgb).value_counts().sort_index())

print("\nTEST:")
print(pd.Series(y_test_lgb).value_counts().sort_index())

assert set(np.unique(y_train_lgb)).issubset({0, 1})
assert set(np.unique(y_validation_lgb)).issubset({0, 1})
assert set(np.unique(y_test_lgb)).issubset({0, 1})

print("✓ All targets are binary")


# ----------------------------------------------------------------
# [7] IMPORTANT DATA-USAGE CHECK
# ----------------------------------------------------------------

print("\n[7] DATA-USAGE POLICY")
print("-" * 70)

print("TRAIN:")
print("  ✓ Original TRAIN + synthetic SMOTEN observations")

print("VALIDATION:")
print("  ✓ Original validation data only")
print("  ✓ No SMOTEN")

print("TEST:")
print("  ✓ Original test data only")
print("  ✓ No SMOTEN")

print("✓ Validation and test remain untouched")


# ----------------------------------------------------------------
# [8] LIGHTGBM BASELINE PARAMETERS
# ----------------------------------------------------------------

print("\n[8] LIGHTGBM BASELINE CONFIGURATION")
print("-" * 70)

lightgbm_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",

    # Baseline learning rate.
    # Hyperparameter tuning comes later.
    "learning_rate": 0.01,

    "n_estimators": 1000,

    "num_leaves": 31,
    "max_depth": -1,

    "min_child_samples": 20,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.0,
    "reg_lambda": 0.0,

    "random_state": 42,
    "n_jobs": -1,

    "verbosity": -1
}

for key, value in lightgbm_params.items():
    print(f"{key:20s}: {value}")


# ----------------------------------------------------------------
# [9] CREATE LIGHTGBM MODEL
# ----------------------------------------------------------------

print("\n[9] CREATING LIGHTGBM MODEL")
print("-" * 70)

lgbm_model = lgb.LGBMClassifier(
    **lightgbm_params
)

print("✓ LightGBM model created")


# ----------------------------------------------------------------
# [10] TRAINING
# ----------------------------------------------------------------

print("\n[10] TRAINING LIGHTGBM")
print("-" * 70)

print("Training data:")
print(f"  X_train_lgb : {X_train_lgb.shape}")
print(f"  y_train_lgb : {y_train_lgb.shape}")

print("\nValidation data:")
print(f"  X_validation_lgb : {X_validation_lgb.shape}")
print(f"  y_validation_lgb : {y_validation_lgb.shape}")

print("\nTEST DATA WILL NOT BE USED FOR TRAINING.")

lgbm_model.fit(
    X_train_lgb,
    y_train_lgb,

    categorical_feature=categorical_features,

    eval_set=[
        (X_validation_lgb, y_validation_lgb)
    ],

    eval_names=[
        "validation"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True
        ),
        lgb.log_evaluation(
            period=50
        )
    ]
)

print("\n✓ LightGBM training completed")


# ----------------------------------------------------------------
# [11] BEST ITERATION
# ----------------------------------------------------------------

print("\n[11] BEST ITERATION")
print("-" * 70)

best_iteration = lgbm_model.best_iteration_

print(f"Best iteration : {best_iteration}")

assert best_iteration is not None
assert best_iteration > 0

print("✓ Early stopping selected the best validation iteration")


# ----------------------------------------------------------------
# [12] VALIDATION PREDICTIONS
# ----------------------------------------------------------------

print("\n[12] VALIDATION PREDICTIONS")
print("-" * 70)

validation_probability = lgbm_model.predict_proba(
    X_validation_lgb,
    num_iteration=best_iteration
)[:, 1]

validation_prediction = (
    validation_probability >= 0.5
).astype(np.int8)

print("✓ Validation predictions generated")


# ----------------------------------------------------------------
# [13] VALIDATION METRICS
# ----------------------------------------------------------------

print("\n[13] VALIDATION METRICS")
print("-" * 70)

validation_accuracy = accuracy_score(
    y_validation_lgb,
    validation_prediction
)

validation_balanced_accuracy = balanced_accuracy_score(
    y_validation_lgb,
    validation_prediction
)

validation_precision = precision_score(
    y_validation_lgb,
    validation_prediction,
    zero_division=0
)

validation_recall = recall_score(
    y_validation_lgb,
    validation_prediction,
    zero_division=0
)

validation_f1 = f1_score(
    y_validation_lgb,
    validation_prediction,
    zero_division=0
)

validation_roc_auc = roc_auc_score(
    y_validation_lgb,
    validation_probability
)

validation_pr_auc = average_precision_score(
    y_validation_lgb,
    validation_probability
)

print(f"Accuracy            : {validation_accuracy:.4f}")
print(f"Balanced Accuracy   : {validation_balanced_accuracy:.4f}")
print(f"Precision            : {validation_precision:.4f}")
print(f"Recall               : {validation_recall:.4f}")
print(f"F1 Score             : {validation_f1:.4f}")
print(f"ROC-AUC              : {validation_roc_auc:.4f}")
print(f"PR-AUC               : {validation_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation_lgb,
        validation_prediction
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_validation_lgb,
        validation_prediction,
        digits=4,
        zero_division=0
    )
)


# ----------------------------------------------------------------
# [14] FINAL ASSERTIONS
# ----------------------------------------------------------------

print("\n[14] FINAL ASSERTIONS")
print("-" * 70)

assert hasattr(lgbm_model, "predict_proba")
assert len(validation_probability) == len(y_validation_lgb)
assert len(validation_prediction) == len(y_validation_lgb)

assert np.all(np.isfinite(validation_probability))
assert np.all((validation_probability >= 0) &
              (validation_probability <= 1))

print("✓ Model exists")
print("✓ Validation predictions aligned")
print("✓ Probabilities are valid")
print("✓ Validation evaluation completed")


# ----------------------------------------------------------------
# FINAL OUTPUT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 10 LIGHTGBM BASELINE COMPLETE")
print("=" * 70)

print("\nMODEL:")
print("  ✓ LightGBM binary classifier")
print("  ✓ Categorical features handled natively")
print("  ✓ TRAIN contains SMOTEN synthetic observations")
print("  ✓ VALIDATION remains untouched")
print("  ✓ TEST remains untouched")

print("\nTRAINING:")
print(f"  Training rows       : {len(X_train_lgb):,}")
print(f"  Validation rows     : {len(X_validation_lgb):,}")
print(f"  Best iteration      : {best_iteration}")

print("\nVALIDATION:")
print(f"  Accuracy            : {validation_accuracy:.4f}")
print(f"  Balanced Accuracy   : {validation_balanced_accuracy:.4f}")
print(f"  Precision           : {validation_precision:.4f}")
print(f"  Recall              : {validation_recall:.4f}")
print(f"  F1                  : {validation_f1:.4f}")
print(f"  ROC-AUC             : {validation_roc_auc:.4f}")
print(f"  PR-AUC              : {validation_pr_auc:.4f}")

print("\n✓ STEP 10 COMPLETE")

STEP 10 — LIGHTGBM TRAINING PREPARATION + BASELINE

[0] REQUIRED-OBJECT VERIFICATION
----------------------------------------------------------------------
✓ X_train_final exists
✓ y_train_final exists
✓ X_validation_final exists
✓ y_validation_final exists
✓ X_test_final exists
✓ y_test_final exists
✓ final_model_features exists
✓ target_column exists

[1] DATA SHAPE VERIFICATION
----------------------------------------------------------------------
X_train_final       : (1317556, 4)
y_train_final       : (1317556,)
X_validation_final  : (308772, 4)
y_validation_final  : (308772,)
X_test_final        : (311057, 4)
y_test_final        : (311057,)
✓ Feature/target alignment verified
✓ Feature counts verified

[2] FINAL FEATURE ORDER
----------------------------------------------------------------------
[0] ordering_mode
[1] culture_description
[2] organism
[3] antibiotic
✓ Feature order verified

[3] CREATING LIGHTGBM DATAFRAMES
----------------------------------------------------------

C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[50]	validation's binary_logloss: 0.51965
[100]	validation's binary_logloss: 0.474955
[150]	validation's binary_logloss: 0.446627
[200]	validation's binary_logloss: 0.429539
[250]	validation's binary_logloss: 0.421034
[300]	validation's binary_logloss: 0.410877
[350]	validation's binary_logloss: 0.405947
[400]	validation's binary_logloss: 0.402335
[450]	validation's binary_logloss: 0.399064
[500]	validation's binary_logloss: 0.396465
[550]	validation's binary_logloss: 0.394929
[600]	validation's binary_logloss: 0.39377
[650]	validation's binary_logloss: 0.392451
[700]	validation's binary_logloss: 0.391435
[750]	validation's binary_logloss: 0.390704
[800]	validation's binary_logloss: 0.390086
[850]	validation's binary_logloss: 0.389121
[900]	validation's binary_logloss: 0.388707
[950]	validation's binary_logloss: 0.388348
[1000]	validation's binary_logloss: 0.387809
Did not meet early stopping. Best iteration is:
[1000]	valid

In [28]:
# ======================================================================
# STEP 11 — LIGHTGBM HYPERPARAMETER TUNING
# ======================================================================
#
# PURPOSE:
#   Tune LightGBM using the SMOTEN-processed TRAIN set and untouched
#   VALIDATION set.
#
# IMPORTANT:
#   TRAIN  -> used for fitting
#   VALIDATION -> used for hyperparameter selection + early stopping
#   TEST -> COMPLETELY UNTOUCHED
#
# DO NOT:
#   - apply SMOTEN again
#   - fit anything on TEST
#   - use TEST for hyperparameter selection
#   - modify the original train/validation/test split
#
# This cell creates:
#   best_lgbm_model
#   best_lgbm_params
#   best_lgbm_iteration
#   tuning_results
#   X_test_final / y_test_final remain untouched
# ======================================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("STEP 11 — LIGHTGBM HYPERPARAMETER TUNING")
print("=" * 70)

# ----------------------------------------------------------------------
# [0] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

required_objects = [
    "X_train_final",
    "y_train_final",
    "X_validation_final",
    "y_validation_final",
    "X_test_final",
    "y_test_final",
    "final_model_features"
]

for obj in required_objects:
    if obj not in globals():
        raise RuntimeError(f"Required object missing: {obj}")

print("\n[0] Required-object verification")
print("-" * 70)

for obj in required_objects:
    print(f"✓ {obj} exists")


# ----------------------------------------------------------------------
# [1] INPUT SHAPE VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] INPUT SHAPES")
print("-" * 70)

print(f"X_train_final       : {X_train_final.shape}")
print(f"y_train_final       : {y_train_final.shape}")
print(f"X_validation_final  : {X_validation_final.shape}")
print(f"y_validation_final  : {y_validation_final.shape}")
print(f"X_test_final        : {X_test_final.shape}")
print(f"y_test_final        : {y_test_final.shape}")

assert X_train_final.shape[0] == len(y_train_final)
assert X_validation_final.shape[0] == len(y_validation_final)
assert X_test_final.shape[0] == len(y_test_final)

assert X_train_final.shape[1] == len(final_model_features)
assert X_validation_final.shape[1] == len(final_model_features)
assert X_test_final.shape[1] == len(final_model_features)

print("✓ TRAIN alignment verified")
print("✓ VALIDATION alignment verified")
print("✓ TEST alignment verified")


# ----------------------------------------------------------------------
# [2] FEATURE ORDER VERIFICATION
# ----------------------------------------------------------------------

print("\n[2] FEATURE ORDER")
print("-" * 70)

print("Final model features:")

for i, feature in enumerate(final_model_features):
    print(f"  [{i}] {feature}")

assert list(final_model_features) == [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic"
]

print("✓ Feature order verified")


# ----------------------------------------------------------------------
# [3] CATEGORICAL FEATURE CONFIGURATION
# ----------------------------------------------------------------------

print("\n[3] CATEGORICAL FEATURE CONFIGURATION")
print("-" * 70)

categorical_feature_indices = [0, 1, 2, 3]

print("Categorical feature indices:")
print(categorical_feature_indices)

print("✓ All four predictors treated as categorical by LightGBM")


# ----------------------------------------------------------------------
# [4] TARGET VERIFICATION
# ----------------------------------------------------------------------

print("\n[4] TARGET VERIFICATION")
print("-" * 70)

train_classes = np.unique(np.asarray(y_train_final))
val_classes = np.unique(np.asarray(y_validation_final))
test_classes = np.unique(np.asarray(y_test_final))

print(f"TRAIN classes      : {train_classes}")
print(f"VALIDATION classes : {val_classes}")
print(f"TEST classes       : {test_classes}")

assert set(train_classes).issubset({0, 1})
assert set(val_classes).issubset({0, 1})
assert set(test_classes).issubset({0, 1})

print("✓ TRAIN target binary")
print("✓ VALIDATION target binary")
print("✓ TEST target binary")


# ----------------------------------------------------------------------
# [5] HYPERPARAMETER SEARCH SPACE
# ----------------------------------------------------------------------
#
# We deliberately keep the search moderate because the dataset is large.
#
# The parameters being investigated:
#
#   learning_rate
#   num_leaves
#   max_depth
#   min_child_samples
#   subsample
#   colsample_bytree
#   reg_alpha
#   reg_lambda
#   min_split_gain
#
# TEST IS NOT USED HERE.
# ----------------------------------------------------------------------

print("\n[5] HYPERPARAMETER SEARCH SPACE")
print("-" * 70)

parameter_configs = [

    {
        "learning_rate": 0.01,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 100,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "min_split_gain": 0.0
    },

    {
        "learning_rate": 0.01,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 100,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "min_split_gain": 0.0
    },

    {
        "learning_rate": 0.01,
        "num_leaves": 127,
        "max_depth": -1,
        "min_child_samples": 150,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 2.0,
        "min_split_gain": 0.0
    },

    {
        "learning_rate": 0.005,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 100,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 2.0,
        "min_split_gain": 0.0
    },

    {
        "learning_rate": 0.01,
        "num_leaves": 63,
        "max_depth": 10,
        "min_child_samples": 150,
        "subsample": 0.85,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.1,
        "reg_lambda": 2.0,
        "min_split_gain": 0.0
    },

    {
        "learning_rate": 0.01,
        "num_leaves": 127,
        "max_depth": 12,
        "min_child_samples": 200,
        "subsample": 0.85,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
        "min_split_gain": 0.0
    }
]

print(f"Number of configurations: {len(parameter_configs)}")
print("✓ Search configurations prepared")


# ----------------------------------------------------------------------
# [6] BASE LIGHTGBM SETTINGS
# ----------------------------------------------------------------------

base_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",

    "verbosity": -1,

    "n_estimators": 1500,

    "random_state": 42,
    "bagging_seed": 42,
    "feature_fraction_seed": 42,
    "data_random_seed": 42,

    "n_jobs": -1
}


# ----------------------------------------------------------------------
# [7] RUN HYPERPARAMETER SEARCH
# ----------------------------------------------------------------------

print("\n[7] STARTING HYPERPARAMETER SEARCH")
print("-" * 70)

print("TRAIN will be used for fitting.")
print("VALIDATION will be used for early stopping and model selection.")
print("TEST will NOT be used.")

tuning_results = []

best_model = None
best_params = None
best_iteration = None
best_pr_auc = -np.inf

for trial_number, config in enumerate(parameter_configs, start=1):

    print("\n" + "=" * 70)
    print(f"TRIAL {trial_number}/{len(parameter_configs)}")
    print("=" * 70)

    print("Parameters:")
    for key, value in config.items():
        print(f"  {key}: {value}")

    params = base_params.copy()
    params.update(config)

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_final,
        y_train_final,

        eval_set=[
            (X_validation_final, y_validation_final)
        ],

        eval_names=["validation"],

        categorical_feature=categorical_feature_indices,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            )
        ]
    )

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    val_probability = model.predict_proba(
        X_validation_final,
        num_iteration=model.best_iteration_
    )[:, 1]

    # --------------------------------------------------------------
    # VALIDATION METRICS
    # --------------------------------------------------------------

    val_pr_auc = average_precision_score(
        y_validation_final,
        val_probability
    )

    val_roc_auc = roc_auc_score(
        y_validation_final,
        val_probability
    )

    val_pred = (val_probability >= 0.5).astype(int)

    val_balanced_accuracy = balanced_accuracy_score(
        y_validation_final,
        val_pred
    )

    val_f1 = f1_score(
        y_validation_final,
        val_pred,
        zero_division=0
    )

    val_recall = recall_score(
        y_validation_final,
        val_pred,
        zero_division=0
    )

    val_precision = precision_score(
        y_validation_final,
        val_pred,
        zero_division=0
    )

    val_accuracy = accuracy_score(
        y_validation_final,
        val_pred
    )

    result = {
        "trial": trial_number,
        **config,
        "best_iteration": model.best_iteration_,
        "validation_pr_auc": val_pr_auc,
        "validation_roc_auc": val_roc_auc,
        "validation_accuracy": val_accuracy,
        "validation_balanced_accuracy": val_balanced_accuracy,
        "validation_precision": val_precision,
        "validation_recall": val_recall,
        "validation_f1": val_f1
    }

    tuning_results.append(result)

    print("\nValidation results:")
    print(f"  Best iteration       : {model.best_iteration_}")
    print(f"  PR-AUC               : {val_pr_auc:.4f}")
    print(f"  ROC-AUC              : {val_roc_auc:.4f}")
    print(f"  Balanced Accuracy    : {val_balanced_accuracy:.4f}")
    print(f"  F1                   : {val_f1:.4f}")
    print(f"  Recall               : {val_recall:.4f}")
    print(f"  Precision            : {val_precision:.4f}")
    print(f"  Accuracy             : {val_accuracy:.4f}")

    # --------------------------------------------------------------
    # MODEL SELECTION
    #
    # Primary metric = PR-AUC
    #
    # Reason:
    # AMR resistant class is the clinically important minority class.
    # PR-AUC is more informative than raw accuracy under imbalance.
    # --------------------------------------------------------------

    if val_pr_auc > best_pr_auc:

        best_pr_auc = val_pr_auc
        best_model = model
        best_params = config.copy()
        best_iteration = model.best_iteration_

        print("\n✓ CURRENT BEST MODEL")


# ----------------------------------------------------------------------
# [8] TUNING RESULTS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[8] HYPERPARAMETER TUNING RESULTS")
print("=" * 70)

tuning_results = pd.DataFrame(tuning_results)

tuning_results = tuning_results.sort_values(
    by="validation_pr_auc",
    ascending=False
).reset_index(drop=True)

print(
    tuning_results[
        [
            "trial",
            "learning_rate",
            "num_leaves",
            "max_depth",
            "min_child_samples",
            "best_iteration",
            "validation_pr_auc",
            "validation_roc_auc",
            "validation_balanced_accuracy",
            "validation_f1"
        ]
    ].to_string(index=False)
)


# ----------------------------------------------------------------------
# [9] BEST MODEL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[9] BEST HYPERPARAMETER CONFIGURATION")
print("=" * 70)

print(f"Best validation PR-AUC : {best_pr_auc:.4f}")
print(f"Best iteration         : {best_iteration}")

print("\nBest parameters:")

for key, value in best_params.items():
    print(f"  {key}: {value}")

assert best_model is not None
assert best_iteration is not None

print("\n✓ Best LightGBM model selected")


# ----------------------------------------------------------------------
# [10] FINAL VALIDATION PREDICTIONS FROM BEST MODEL
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[10] BEST MODEL VALIDATION EVALUATION")
print("=" * 70)

validation_probability_tuned = best_model.predict_proba(
    X_validation_final,
    num_iteration=best_iteration
)[:, 1]

validation_prediction_tuned = (
    validation_probability_tuned >= 0.5
).astype(int)

tuned_accuracy = accuracy_score(
    y_validation_final,
    validation_prediction_tuned
)

tuned_balanced_accuracy = balanced_accuracy_score(
    y_validation_final,
    validation_prediction_tuned
)

tuned_precision = precision_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

tuned_recall = recall_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

tuned_f1 = f1_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

tuned_roc_auc = roc_auc_score(
    y_validation_final,
    validation_probability_tuned
)

tuned_pr_auc = average_precision_score(
    y_validation_final,
    validation_probability_tuned
)

print(f"Accuracy            : {tuned_accuracy:.4f}")
print(f"Balanced Accuracy   : {tuned_balanced_accuracy:.4f}")
print(f"Precision            : {tuned_precision:.4f}")
print(f"Recall               : {tuned_recall:.4f}")
print(f"F1 Score             : {tuned_f1:.4f}")
print(f"ROC-AUC              : {tuned_roc_auc:.4f}")
print(f"PR-AUC               : {tuned_pr_auc:.4f}")


# ----------------------------------------------------------------------
# [11] TEST INTEGRITY CHECK
# ----------------------------------------------------------------------
#
# We do NOT generate test predictions here.
#
# This is deliberate.
#
# TEST must remain untouched until the final locked model,
# threshold, and feature set have been finalized.
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[11] TEST INTEGRITY")
print("=" * 70)

print(f"X_test_final shape : {X_test_final.shape}")
print(f"y_test_final shape : {y_test_final.shape}")

print("✓ TEST data was not used for hyperparameter tuning")
print("✓ TEST data was not used for early stopping")
print("✓ TEST predictions were NOT generated")
print("✓ TEST remains untouched")


# ----------------------------------------------------------------------
# [12] SAVE IMPORTANT OBJECTS
# ----------------------------------------------------------------------

best_lgbm_model = best_model
best_lgbm_params = best_params
best_lgbm_iteration = best_iteration

# Keep the baseline model separate if it already exists.
# Do NOT overwrite it.

print("\n" + "=" * 70)
print("[12] FINAL OBJECTS")
print("=" * 70)

print("✓ best_lgbm_model")
print("✓ best_lgbm_params")
print("✓ best_lgbm_iteration")
print("✓ tuning_results")
print("✓ validation_probability_tuned")
print("✓ validation_prediction_tuned")


# ----------------------------------------------------------------------
# [13] FINAL ASSERTIONS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[13] FINAL ASSERTIONS")
print("=" * 70)

assert best_lgbm_model is not None
assert isinstance(best_lgbm_iteration, (int, np.integer))
assert best_lgbm_iteration > 0

assert len(validation_probability_tuned) == len(y_validation_final)
assert len(validation_prediction_tuned) == len(y_validation_final)

assert np.all(
    (validation_probability_tuned >= 0) &
    (validation_probability_tuned <= 1)
)

assert X_test_final.shape[0] == len(y_test_final)

print("✓ Best model exists")
print("✓ Best iteration is valid")
print("✓ Validation probabilities aligned")
print("✓ Validation predictions aligned")
print("✓ Validation probabilities are valid")
print("✓ TEST remains untouched")

print("\n" + "=" * 70)
print("STEP 11 LIGHTGBM HYPERPARAMETER TUNING COMPLETE")
print("=" * 70)

print("\nSELECTED MODEL:")
print(f"  Best PR-AUC       : {best_pr_auc:.4f}")
print(f"  Best iteration    : {best_lgbm_iteration}")

print("\nDATA USAGE:")
print("  ✓ SMOTEN TRAIN used for model fitting")
print("  ✓ VALIDATION used for tuning")
print("  ✓ VALIDATION used for early stopping")
print("  ✓ TEST NOT USED")

print("\n✓ All STEP 11 assertions passed.")

STEP 11 — LIGHTGBM HYPERPARAMETER TUNING

[0] Required-object verification
----------------------------------------------------------------------
✓ X_train_final exists
✓ y_train_final exists
✓ X_validation_final exists
✓ y_validation_final exists
✓ X_test_final exists
✓ y_test_final exists
✓ final_model_features exists

[1] INPUT SHAPES
----------------------------------------------------------------------
X_train_final       : (1317556, 4)
y_train_final       : (1317556,)
X_validation_final  : (308772, 4)
y_validation_final  : (308772,)
X_test_final        : (311057, 4)
y_test_final        : (311057,)
✓ TRAIN alignment verified
✓ VALIDATION alignment verified
✓ TEST alignment verified

[2] FEATURE ORDER
----------------------------------------------------------------------
Final model features:
  [0] ordering_mode
  [1] culture_description
  [2] organism
  [3] antibiotic
✓ Feature order verified

[3] CATEGORICAL FEATURE CONFIGURATION
--------------------------------------------------

C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation results:
  Best iteration       : 1500
  PR-AUC               : 0.6411
  ROC-AUC              : 0.8600
  Balanced Accuracy    : 0.7665
  F1                   : 0.5545
  Recall               : 0.7084
  Precision            : 0.4556
  Accuracy             : 0.8046

✓ CURRENT BEST MODEL

TRIAL 2/6
Parameters:
  learning_rate: 0.01
  num_leaves: 63
  max_depth: -1
  min_child_samples: 100
  subsample: 0.9
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 1.0
  min_split_gain: 0.0


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation results:
  Best iteration       : 1436
  PR-AUC               : 0.6404
  ROC-AUC              : 0.8593
  Balanced Accuracy    : 0.7658
  F1                   : 0.5539
  Recall               : 0.7067
  Precision            : 0.4554
  Accuracy             : 0.8045

TRIAL 3/6
Parameters:
  learning_rate: 0.01
  num_leaves: 127
  max_depth: -1
  min_child_samples: 150
  subsample: 0.9
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 2.0
  min_split_gain: 0.0


...


Validation results:
  Best iteration       : 1500
  PR-AUC               : 0.6388
  ROC-AUC              : 0.8593
  Balanced Accuracy    : 0.7657
  F1                   : 0.5532
  Recall               : 0.7080
  Precision            : 0.4539
  Accuracy             : 0.8036

TRIAL 5/6
Parameters:
  learning_rate: 0.01
  num_leaves: 63
  max_depth: 10
  min_child_samples: 150
  subsample: 0.85
  colsample_bytree: 0.9
  reg_alpha: 0.1
  reg_lambda: 2.0
  min_split_gain: 0.0


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation results:
  Best iteration       : 1434
  PR-AUC               : 0.6408
  ROC-AUC              : 0.8596
  Balanced Accuracy    : 0.7663
  F1                   : 0.5544
  Recall               : 0.7081
  Precision            : 0.4555
  Accuracy             : 0.8045

TRIAL 6/6
Parameters:
  learning_rate: 0.01
  num_leaves: 127
  max_depth: 12
  min_child_samples: 200
  subsample: 0.85
  colsample_bytree: 0.9
  reg_alpha: 0.1
  reg_lambda: 3.0
  min_split_gain: 0.0


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Validation results:
  Best iteration       : 1319
  PR-AUC               : 0.6408
  ROC-AUC              : 0.8596
  Balanced Accuracy    : 0.7664
  F1                   : 0.5544
  Recall               : 0.7083
  Precision            : 0.4554
  Accuracy             : 0.8045

[8] HYPERPARAMETER TUNING RESULTS
 trial  learning_rate  num_leaves  max_depth  min_child_samples  best_iteration  validation_pr_auc  validation_roc_auc  validation_balanced_accuracy  validation_f1
     1          0.010          31         -1                100            1500           0.641063            0.860017                      0.766455       0.554534
     5          0.010          63         10                150            1434           0.640848            0.859582                      0.766286       0.554352
     6          0.010         127         12                200            1319           0.640836            0.859588                      0.766359       0.554389
     2          0.010          63 

In [29]:
# ======================================================================
# STEP 12A — LIGHTGBM LEARNING-RATE-SCHEDULED TRAINING
# ======================================================================

import numpy as np
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("STEP 12A — LIGHTGBM LEARNING-RATE-SCHEDULED TRAINING")
print("=" * 70)

# ----------------------------------------------------------------------
# [0] REQUIRED-OBJECT VERIFICATION
# ----------------------------------------------------------------------

required_objects = [
    "X_train_final",
    "y_train_final",
    "X_validation_final",
    "y_validation_final",
    "X_test_final",
    "y_test_final",
    "final_model_features",
    "best_lgbm_params"
]

for obj in required_objects:
    assert obj in globals(), f"Required object missing: {obj}"
    print(f"✓ {obj} exists")

# ----------------------------------------------------------------------
# [1] INPUT SHAPE VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[1] INPUT SHAPES")
print("-" * 70)

print(f"X_train_final       : {X_train_final.shape}")
print(f"y_train_final       : {y_train_final.shape}")
print(f"X_validation_final  : {X_validation_final.shape}")
print(f"y_validation_final  : {y_validation_final.shape}")
print(f"X_test_final        : {X_test_final.shape}")
print(f"y_test_final        : {y_test_final.shape}")

assert len(X_train_final) == len(y_train_final)
assert len(X_validation_final) == len(y_validation_final)
assert len(X_test_final) == len(y_test_final)

print("✓ TRAIN alignment verified")
print("✓ VALIDATION alignment verified")
print("✓ TEST alignment verified")

# ----------------------------------------------------------------------
# [2] FEATURE CONFIGURATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] FEATURE CONFIGURATION")
print("-" * 70)

print("Final model features:")

for i, feature in enumerate(final_model_features):
    print(f"  [{i}] {feature}")

categorical_feature_indices = list(range(len(final_model_features)))

print(f"\nCategorical feature indices: {categorical_feature_indices}")

assert len(final_model_features) == 4
assert categorical_feature_indices == [0, 1, 2, 3]

print("✓ All four predictors configured as categorical")

# ----------------------------------------------------------------------
# [3] TARGET VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] TARGET VERIFICATION")
print("-" * 70)

assert set(np.unique(y_train_final)).issubset({0, 1})
assert set(np.unique(y_validation_final)).issubset({0, 1})
assert set(np.unique(y_test_final)).issubset({0, 1})

print("TRAIN classes      :", np.unique(y_train_final))
print("VALIDATION classes :", np.unique(y_validation_final))
print("TEST classes       :", np.unique(y_test_final))

print("✓ All targets remain binary")

# ----------------------------------------------------------------------
# [4] LEARNING-RATE SCHEDULE
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[4] LEARNING-RATE SCHEDULE")
print("-" * 70)

# Piecewise schedule:
#   1–500       : 0.010
#   501–1000    : 0.005
#   1001–1500   : 0.001
#
# The callback receives the current boosting iteration.

def learning_rate_schedule(current_iteration):
    iteration = current_iteration + 1

    if iteration <= 500:
        return 0.01
    elif iteration <= 1000:
        return 0.005
    else:
        return 0.001

print("Iterations 1–500     : learning_rate = 0.010")
print("Iterations 501–1000  : learning_rate = 0.005")
print("Iterations 1001–1500 : learning_rate = 0.001")

# ----------------------------------------------------------------------
# [5] MODEL PARAMETERS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[5] MODEL PARAMETERS")
print("-" * 70)

scheduled_params = {
    "objective": "binary",
    "metric": "binary_logloss",

    # Starting learning rate is controlled by callback.
    "learning_rate": 0.01,

    # Best configuration from STEP 11.
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,

    "subsample": 0.9,
    "colsample_bytree": 0.9,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_split_gain": 0.0,

    "verbosity": -1,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "data_random_seed": 42
}

for key, value in scheduled_params.items():
    print(f"{key:25s}: {value}")

# ----------------------------------------------------------------------
# [6] CREATE LEARNING-RATE CALLBACK
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[6] CREATING LEARNING-RATE CALLBACK")
print("-" * 70)

lr_callback = lgb.reset_parameter(
    learning_rate=learning_rate_schedule
)

print("✓ Learning-rate scheduling callback created")

# ----------------------------------------------------------------------
# [7] CREATE EARLY-STOPPING CALLBACK
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[7] EARLY-STOPPING CONFIGURATION")
print("-" * 70)

early_stopping_callback = lgb.early_stopping(
    stopping_rounds=150,
    first_metric_only=True,
    verbose=True
)

print("Early stopping patience : 150 rounds")
print("Monitoring              : validation binary_logloss")

# ----------------------------------------------------------------------
# [8] TRAIN MODEL
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[8] STARTING SCHEDULED LIGHTGBM TRAINING")
print("-" * 70)

print("TRAIN       : SMOTEN-balanced training data")
print("VALIDATION  : untouched validation data")
print("TEST        : completely untouched")
print()
print("Learning-rate schedule:")
print("  0.010 → 0.005 → 0.001")
print()

scheduled_lgbm_model = lgb.LGBMClassifier(
    **scheduled_params,
    n_estimators=1500
)

scheduled_lgbm_model.fit(
    X_train_final,
    y_train_final,

    eval_set=[
        (X_validation_final, y_validation_final)
    ],

    eval_names=["validation"],

    categorical_feature=categorical_feature_indices,

    callbacks=[
        lr_callback,
        early_stopping_callback,
        lgb.log_evaluation(period=50)
    ]
)

print("\n✓ Learning-rate-scheduled LightGBM training completed")

# ----------------------------------------------------------------------
# [9] BEST ITERATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[9] BEST ITERATION")
print("-" * 70)

scheduled_best_iteration = scheduled_lgbm_model.best_iteration_

print(f"Best iteration : {scheduled_best_iteration}")

assert scheduled_best_iteration is not None
assert scheduled_best_iteration > 0

print("✓ Valid best iteration obtained")

# ----------------------------------------------------------------------
# [10] VALIDATION PREDICTIONS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[10] VALIDATION PREDICTIONS")
print("-" * 70)

validation_probability_scheduled = (
    scheduled_lgbm_model.predict_proba(
        X_validation_final,
        num_iteration=scheduled_best_iteration
    )[:, 1]
)

assert len(validation_probability_scheduled) == len(y_validation_final)
assert np.all(np.isfinite(validation_probability_scheduled))
assert np.all(
    (validation_probability_scheduled >= 0) &
    (validation_probability_scheduled <= 1)
)

# Use the same 0.5 decision threshold here.
validation_prediction_scheduled = (
    validation_probability_scheduled >= 0.5
).astype(int)

print("✓ Validation probabilities generated")
print("✓ Validation predictions generated")

# ----------------------------------------------------------------------
# [11] VALIDATION METRICS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[11] SCHEDULED MODEL VALIDATION METRICS")
print("-" * 70)

scheduled_val_accuracy = accuracy_score(
    y_validation_final,
    validation_prediction_scheduled
)

scheduled_val_balanced_accuracy = balanced_accuracy_score(
    y_validation_final,
    validation_prediction_scheduled
)

scheduled_val_precision = precision_score(
    y_validation_final,
    validation_prediction_scheduled,
    zero_division=0
)

scheduled_val_recall = recall_score(
    y_validation_final,
    validation_prediction_scheduled,
    zero_division=0
)

scheduled_val_f1 = f1_score(
    y_validation_final,
    validation_prediction_scheduled,
    zero_division=0
)

scheduled_val_roc_auc = roc_auc_score(
    y_validation_final,
    validation_probability_scheduled
)

scheduled_val_pr_auc = average_precision_score(
    y_validation_final,
    validation_probability_scheduled
)

print(f"Accuracy            : {scheduled_val_accuracy:.4f}")
print(f"Balanced Accuracy   : {scheduled_val_balanced_accuracy:.4f}")
print(f"Precision           : {scheduled_val_precision:.4f}")
print(f"Recall              : {scheduled_val_recall:.4f}")
print(f"F1 Score            : {scheduled_val_f1:.4f}")
print(f"ROC-AUC             : {scheduled_val_roc_auc:.4f}")
print(f"PR-AUC              : {scheduled_val_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation_final,
        validation_prediction_scheduled
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_validation_final,
        validation_prediction_scheduled,
        digits=4,
        zero_division=0
    )
)

# ----------------------------------------------------------------------
# [12] COMPARE WITH STEP 11 TUNED MODEL
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[12] STEP 11 vs LEARNING-RATE-SCHEDULED MODEL")
print("-" * 70)

print(
    f"{'Metric':<22}"
    f"{'STEP 11':>12}"
    f"{'Scheduled':>15}"
    f"{'Change':>12}"
)

print("-" * 61)

comparison_metrics = [
    ("PR-AUC", validation_probability_tuned, "prob"),
]

# Direct metric comparison
step11_metrics = {
    "PR-AUC": average_precision_score(
        y_validation_final,
        validation_probability_tuned
    ),
    "ROC-AUC": roc_auc_score(
        y_validation_final,
        validation_probability_tuned
    ),
    "Balanced Accuracy": balanced_accuracy_score(
        y_validation_final,
        validation_prediction_tuned
    ),
    "F1": f1_score(
        y_validation_final,
        validation_prediction_tuned,
        zero_division=0
    ),
    "Recall": recall_score(
        y_validation_final,
        validation_prediction_tuned,
        zero_division=0
    ),
    "Precision": precision_score(
        y_validation_final,
        validation_prediction_tuned,
        zero_division=0
    ),
    "Accuracy": accuracy_score(
        y_validation_final,
        validation_prediction_tuned
    )
}

scheduled_metrics = {
    "PR-AUC": scheduled_val_pr_auc,
    "ROC-AUC": scheduled_val_roc_auc,
    "Balanced Accuracy": scheduled_val_balanced_accuracy,
    "F1": scheduled_val_f1,
    "Recall": scheduled_val_recall,
    "Precision": scheduled_val_precision,
    "Accuracy": scheduled_val_accuracy
}

for metric in step11_metrics:
    old = step11_metrics[metric]
    new = scheduled_metrics[metric]
    change = new - old

    print(
        f"{metric:<22}"
        f"{old:>12.4f}"
        f"{new:>15.4f}"
        f"{change:>+12.4f}"
    )

# ----------------------------------------------------------------------
# [13] TEST INTEGRITY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[13] TEST INTEGRITY")
print("-" * 70)

print("✓ TEST was not used for fitting")
print("✓ TEST was not used for learning-rate scheduling")
print("✓ TEST was not used for early stopping")
print("✓ TEST predictions were NOT generated")
print("✓ TEST remains untouched")

# ----------------------------------------------------------------------
# [14] FINAL OBJECTS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[14] FINAL OBJECTS")
print("-" * 70)

print("✓ scheduled_lgbm_model")
print("✓ scheduled_best_iteration")
print("✓ validation_probability_scheduled")
print("✓ validation_prediction_scheduled")

# ----------------------------------------------------------------------
# [15] FINAL ASSERTIONS
# ----------------------------------------------------------------------

assert scheduled_lgbm_model is not None
assert scheduled_best_iteration > 0

assert len(validation_probability_scheduled) == len(
    y_validation_final
)

assert len(validation_prediction_scheduled) == len(
    y_validation_final
)

assert np.all(
    (validation_probability_scheduled >= 0) &
    (validation_probability_scheduled <= 1)
)

assert set(np.unique(validation_prediction_scheduled)).issubset({0, 1})

print("✓ Model exists")
print("✓ Best iteration valid")
print("✓ Validation probabilities aligned")
print("✓ Validation predictions aligned")
print("✓ Probabilities valid")
print("✓ TEST remains untouched")

print("\n" + "=" * 70)
print("STEP 12A LEARNING-RATE-SCHEDULED TRAINING COMPLETE")
print("=" * 70)

print("\nSCHEDULE:")
print("  Iterations 1–500     → 0.010")
print("  Iterations 501–1000  → 0.005")
print("  Iterations 1001–1500 → 0.001")

print("\nVALIDATION:")
print(f"  PR-AUC             : {scheduled_val_pr_auc:.4f}")
print(f"  ROC-AUC            : {scheduled_val_roc_auc:.4f}")
print(f"  Balanced Accuracy  : {scheduled_val_balanced_accuracy:.4f}")
print(f"  F1                 : {scheduled_val_f1:.4f}")
print(f"  Recall             : {scheduled_val_recall:.4f}")
print(f"  Precision          : {scheduled_val_precision:.4f}")
print(f"  Accuracy           : {scheduled_val_accuracy:.4f}")

print("\n✓ All STEP 12A assertions passed.")

STEP 12A — LIGHTGBM LEARNING-RATE-SCHEDULED TRAINING
✓ X_train_final exists
✓ y_train_final exists
✓ X_validation_final exists
✓ y_validation_final exists
✓ X_test_final exists
✓ y_test_final exists
✓ final_model_features exists
✓ best_lgbm_params exists

----------------------------------------------------------------------
[1] INPUT SHAPES
----------------------------------------------------------------------
X_train_final       : (1317556, 4)
y_train_final       : (1317556,)
X_validation_final  : (308772, 4)
y_validation_final  : (308772,)
X_test_final        : (311057, 4)
y_test_final        : (311057,)
✓ TRAIN alignment verified
✓ VALIDATION alignment verified
✓ TEST alignment verified

----------------------------------------------------------------------
[2] FEATURE CONFIGURATION
----------------------------------------------------------------------
Final model features:
  [0] ordering_mode
  [1] culture_description
  [2] organism
  [3] antibiotic

Categorical feature indices: [

c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

[50]	validation's binary_logloss: 0.499448
[100]	validation's binary_logloss: 0.455005
[150]	validation's binary_logloss: 0.430618
[200]	validation's binary_logloss: 0.416367
[250]	validation's binary_logloss: 0.406228
[300]	validation's binary_logloss: 0.400307
[350]	validation's binary_logloss: 0.39615
[400]	validation's binary_logloss: 0.393405
[450]	validation's binary_logloss: 0.391384
[500]	validation's binary_logloss: 0.389931
[550]	validation's binary_logloss: 0.389293
[600]	validation's binary_logloss: 0.388806
[650]	validation's binary_logloss: 0.388314
[700]	validation's binary_logloss: 0.387787
[750]	validation's binary_logloss: 0.387378
[800]	validation's binary_logloss: 0.386911
[850]	validation's binary_logloss: 0.386476
[900]	validation's binary_logloss: 0.38608
[950]	validation's binary_logloss: 0.385664
[1000]	validation's binary_logloss: 0.385395
[1050]	validation's binary_logloss: 0.385321
[1100]	validation's binary_logloss: 0.385262
[1150]	validation's binary_loglo

In [31]:
# ======================================================================
# STEP 12B — FT-TRANSFORMER TRAINING
# ======================================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("STEP 12B — FT-TRANSFORMER TRAINING")
print("=" * 70)

# ----------------------------------------------------------------------
# [0] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

required_objects = [
    "X_train_final",
    "y_train_final",
    "X_validation_final",
    "y_validation_final",
    "X_test_final",
    "y_test_final",
    "final_model_features"
]

for obj in required_objects:
    assert obj in globals(), f"Required object missing: {obj}"
    print(f"✓ {obj} exists")

# ----------------------------------------------------------------------
# [1] FEATURE ORDER
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[1] FEATURE ORDER")
print("-" * 70)

expected_features = [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic"
]

assert list(final_model_features) == expected_features

for i, feature in enumerate(final_model_features):
    print(f"[{i}] {feature}")

print("✓ Feature order verified")

# ----------------------------------------------------------------------
# [2] INPUT SHAPE VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[2] INPUT SHAPES")
print("-" * 70)

print("X_train_final       :", X_train_final.shape)
print("X_validation_final  :", X_validation_final.shape)
print("X_test_final        :", X_test_final.shape)

print("y_train_final       :", np.shape(y_train_final))
print("y_validation_final  :", np.shape(y_validation_final))
print("y_test_final        :", np.shape(y_test_final))

assert X_train_final.ndim == 2
assert X_validation_final.ndim == 2
assert X_test_final.ndim == 2

assert X_train_final.shape[1] == 4
assert X_validation_final.shape[1] == 4
assert X_test_final.shape[1] == 4

assert len(X_train_final) == len(y_train_final)
assert len(X_validation_final) == len(y_validation_final)
assert len(X_test_final) == len(y_test_final)

print("✓ Four features confirmed")
print("✓ TRAIN alignment verified")
print("✓ VALIDATION alignment verified")
print("✓ TEST alignment verified")

# ----------------------------------------------------------------------
# [3] VERIFY NUMPY ARRAY REPRESENTATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[3] DATA REPRESENTATION")
print("-" * 70)

print("TRAIN type      :", type(X_train_final))
print("VALIDATION type :", type(X_validation_final))
print("TEST type       :", type(X_test_final))

X_train_np = np.asarray(X_train_final, dtype=np.int64)
X_validation_np = np.asarray(X_validation_final, dtype=np.int64)
X_test_np = np.asarray(X_test_final, dtype=np.int64)

y_train_np = np.asarray(y_train_final, dtype=np.float32)
y_validation_np = np.asarray(y_validation_final, dtype=np.float32)
y_test_np = np.asarray(y_test_final, dtype=np.float32)

print("✓ Converted features to int64")
print("✓ Converted targets to float32")

# ----------------------------------------------------------------------
# [4] CATEGORY CARDINALITIES
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[4] CATEGORY CARDINALITIES")
print("-" * 70)

category_sizes = []

for idx, feature in enumerate(final_model_features):

    train_values = X_train_np[:, idx]
    val_values = X_validation_np[:, idx]
    test_values = X_test_np[:, idx]

    train_min = int(train_values.min())
    train_max = int(train_values.max())

    val_min = int(val_values.min())
    val_max = int(val_values.max())

    test_min = int(test_values.min())
    test_max = int(test_values.max())

    maximum_code = max(
        train_max,
        val_max,
        test_max
    )

    # Code 0 = unseen category.
    # Codes 1...N = known TRAIN categories.
    cardinality = maximum_code + 1

    category_sizes.append(cardinality)

    print(
        f"{feature:25s} | "
        f"TRAIN [{train_min}, {train_max}] | "
        f"VAL [{val_min}, {val_max}] | "
        f"TEST [{test_min}, {test_max}] | "
        f"cardinality={cardinality}"
    )

print("\nCategory sizes:", category_sizes)

# Expected:
# ordering_mode          -> 4
# culture_description    -> 4
# organism               -> 281
# antibiotic             -> 55

# ----------------------------------------------------------------------
# [5] CATEGORY CODE VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[5] CATEGORY CODE VALIDATION")
print("-" * 70)

for idx, feature in enumerate(final_model_features):

    cardinality = category_sizes[idx]

    assert X_train_np[:, idx].min() >= 0
    assert X_validation_np[:, idx].min() >= 0
    assert X_test_np[:, idx].min() >= 0

    assert X_train_np[:, idx].max() < cardinality
    assert X_validation_np[:, idx].max() < cardinality
    assert X_test_np[:, idx].max() < cardinality

    print(
        f"✓ {feature}: valid codes 0 ... {cardinality - 1}"
    )

print("✓ All categorical codes valid")

# ----------------------------------------------------------------------
# [6] TARGET VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[6] TARGET VALIDATION")
print("-" * 70)

assert set(np.unique(y_train_np)).issubset({0, 1})
assert set(np.unique(y_validation_np)).issubset({0, 1})
assert set(np.unique(y_test_np)).issubset({0, 1})

print("TRAIN classes      :", np.unique(y_train_np))
print("VALIDATION classes :", np.unique(y_validation_np))
print("TEST classes       :", np.unique(y_test_np))

print("✓ Binary targets verified")

# ----------------------------------------------------------------------
# [7] DEVICE
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[7] PYTORCH DEVICE")
print("-" * 70)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device :", device)

if device.type == "cuda":
    print("✓ CUDA available")
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print("⚠ CUDA unavailable — CPU will be used")

# ----------------------------------------------------------------------
# [8] PYTORCH DATASETS
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[8] BUILDING PYTORCH DATASETS")
print("-" * 70)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_np),
    torch.from_numpy(y_train_np)
)

validation_dataset = TensorDataset(
    torch.from_numpy(X_validation_np),
    torch.from_numpy(y_validation_np)
)

test_dataset = TensorDataset(
    torch.from_numpy(X_test_np),
    torch.from_numpy(y_test_np)
)

BATCH_SIZE = 2048

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Batch size :", BATCH_SIZE)

print("TRAIN batches      :", len(train_loader))
print("VALIDATION batches :", len(validation_loader))
print("TEST batches       :", len(test_loader))

print("✓ DataLoaders created")

# ----------------------------------------------------------------------
# [9] FT-TRANSFORMER
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[9] DEFINING FT-TRANSFORMER")
print("-" * 70)


class FTTransformer(nn.Module):

    def __init__(
        self,
        category_sizes,
        d_token=64,
        n_blocks=3,
        n_heads=8,
        dropout=0.15,
        ffn_multiplier=4
    ):
        super().__init__()

        self.n_features = len(category_sizes)
        self.d_token = d_token

        # Separate embedding for every categorical feature.
        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=size,
                embedding_dim=d_token
            )
            for size in category_sizes
        ])

        # Feature-specific learnable embeddings.
        self.feature_embeddings = nn.Parameter(
            torch.randn(
                self.n_features,
                d_token
            ) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ffn_multiplier,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_blocks
        )

        self.norm = nn.LayerNorm(d_token)

        self.head = nn.Sequential(
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):

        tokens = []

        for i in range(self.n_features):

            token = self.embeddings[i](
                x[:, i]
            )

            token = (
                token +
                self.feature_embeddings[i]
            )

            tokens.append(token)

        x = torch.stack(
            tokens,
            dim=1
        )

        x = self.transformer(x)

        # Pool the four feature tokens.
        x = x.mean(dim=1)

        x = self.norm(x)

        logits = self.head(x)

        return logits.squeeze(-1)


ft_transformer = FTTransformer(
    category_sizes=category_sizes,
    d_token=64,
    n_blocks=3,
    n_heads=8,
    dropout=0.15
).to(device)

total_parameters = sum(
    p.numel()
    for p in ft_transformer.parameters()
    if p.requires_grad
)

print(ft_transformer)

print(
    f"\nTrainable parameters: {total_parameters:,}"
)

print("✓ FT-Transformer created")

# ----------------------------------------------------------------------
# [10] LOSS / OPTIMIZER / SCHEDULER
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[10] OPTIMIZATION CONFIGURATION")
print("-" * 70)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    ft_transformer.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-5
)

print("Loss              : BCEWithLogitsLoss")
print("Optimizer         : AdamW")
print("Initial LR        : 0.001")
print("Weight decay      : 0.0001")
print("Scheduler         : ReduceLROnPlateau")
print("Reduction factor  : 0.5")
print("Scheduler patience: 5 epochs")
print("Minimum LR        : 0.00001")

# ----------------------------------------------------------------------
# [11] EPOCH FUNCTION
# ----------------------------------------------------------------------

def run_epoch(model, loader, optimizer=None):

    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    all_probabilities = []
    all_targets = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(training):

            logits = model(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            probabilities = torch.sigmoid(
                logits
            )

            if training:

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

        batch_size = X_batch.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

        all_probabilities.append(
            probabilities.detach()
            .cpu()
            .numpy()
        )

        all_targets.append(
            y_batch.detach()
            .cpu()
            .numpy()
        )

    mean_loss = (
        total_loss /
        total_samples
    )

    probabilities = np.concatenate(
        all_probabilities
    )

    targets = np.concatenate(
        all_targets
    )

    return (
        mean_loss,
        probabilities,
        targets
    )

# ----------------------------------------------------------------------
# [12] TRAINING
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[12] STARTING FT-TRANSFORMER TRAINING")
print("-" * 70)

MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 10

best_validation_pr_auc = -np.inf
best_validation_loss = np.inf
best_epoch = 0
epochs_without_improvement = 0

best_model_state = None

ft_training_history = []

for epoch in range(1, MAX_EPOCHS + 1):

    train_loss, train_probability, train_target = run_epoch(
        ft_transformer,
        train_loader,
        optimizer
    )

    validation_loss, validation_probability, validation_target = (
        run_epoch(
            ft_transformer,
            validation_loader,
            None
        )
    )

    validation_pr_auc = average_precision_score(
        validation_target,
        validation_probability
    )

    validation_roc_auc = roc_auc_score(
        validation_target,
        validation_probability
    )

    current_lr = optimizer.param_groups[0]["lr"]

    ft_training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "validation_pr_auc": validation_pr_auc,
        "validation_roc_auc": validation_roc_auc,
        "learning_rate": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {validation_loss:.5f} | "
        f"Val PR-AUC: {validation_pr_auc:.5f} | "
        f"Val ROC-AUC: {validation_roc_auc:.5f} | "
        f"LR: {current_lr:.6f}"
    )

    # Scheduler uses validation loss.
    scheduler.step(validation_loss)

    # Best model selected using validation PR-AUC.
    if validation_pr_auc > best_validation_pr_auc:

        best_validation_pr_auc = validation_pr_auc
        best_validation_loss = validation_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        best_model_state = {
            key: value.detach()
            .cpu()
            .clone()
            for key, value
            in ft_transformer.state_dict().items()
        }

        print(
            f"  ✓ New best model — "
            f"PR-AUC = {validation_pr_auc:.5f}"
        )

    else:

        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\n✓ Early stopping triggered."
        )

        break

# ----------------------------------------------------------------------
# [13] RESTORE BEST MODEL
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[13] RESTORING BEST MODEL")
print("-" * 70)

assert best_model_state is not None

ft_transformer.load_state_dict(
    best_model_state
)

ft_transformer = ft_transformer.to(device)

print("Best epoch :", best_epoch)
print(
    "Best validation PR-AUC : "
    f"{best_validation_pr_auc:.5f}"
)

print(
    "Best validation loss   : "
    f"{best_validation_loss:.5f}"
)

print("✓ Best model restored")

# ----------------------------------------------------------------------
# [14] FINAL VALIDATION
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[14] FINAL VALIDATION EVALUATION")
print("-" * 70)

validation_loss_ft, validation_probability_ft, validation_target_ft = (
    run_epoch(
        ft_transformer,
        validation_loader,
        None
    )
)

validation_prediction_ft = (
    validation_probability_ft >= 0.5
).astype(int)

ft_val_accuracy = accuracy_score(
    validation_target_ft,
    validation_prediction_ft
)

ft_val_balanced_accuracy = balanced_accuracy_score(
    validation_target_ft,
    validation_prediction_ft
)

ft_val_precision = precision_score(
    validation_target_ft,
    validation_prediction_ft,
    zero_division=0
)

ft_val_recall = recall_score(
    validation_target_ft,
    validation_prediction_ft,
    zero_division=0
)

ft_val_f1 = f1_score(
    validation_target_ft,
    validation_prediction_ft,
    zero_division=0
)

ft_val_roc_auc = roc_auc_score(
    validation_target_ft,
    validation_probability_ft
)

ft_val_pr_auc = average_precision_score(
    validation_target_ft,
    validation_probability_ft
)

print(f"Accuracy            : {ft_val_accuracy:.4f}")
print(f"Balanced Accuracy   : {ft_val_balanced_accuracy:.4f}")
print(f"Precision           : {ft_val_precision:.4f}")
print(f"Recall              : {ft_val_recall:.4f}")
print(f"F1 Score            : {ft_val_f1:.4f}")
print(f"ROC-AUC             : {ft_val_roc_auc:.4f}")
print(f"PR-AUC              : {ft_val_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        validation_target_ft,
        validation_prediction_ft
    )
)

print("\nClassification Report:")
print(
    classification_report(
        validation_target_ft,
        validation_prediction_ft,
        digits=4,
        zero_division=0
    )
)

# ----------------------------------------------------------------------
# [15] COMPARE WITH LIGHTGBM
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[15] LIGHTGBM VS FT-TRANSFORMER")
print("-" * 70)

lgbm_pr_auc = average_precision_score(
    y_validation_final,
    validation_probability_tuned
)

lgbm_roc_auc = roc_auc_score(
    y_validation_final,
    validation_probability_tuned
)

lgbm_balanced_accuracy = balanced_accuracy_score(
    y_validation_final,
    validation_prediction_tuned
)

lgbm_f1 = f1_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

lgbm_recall = recall_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

lgbm_precision = precision_score(
    y_validation_final,
    validation_prediction_tuned,
    zero_division=0
)

lgbm_accuracy = accuracy_score(
    y_validation_final,
    validation_prediction_tuned
)

comparison = [
    ("PR-AUC", lgbm_pr_auc, ft_val_pr_auc),
    ("ROC-AUC", lgbm_roc_auc, ft_val_roc_auc),
    ("Balanced Accuracy", lgbm_balanced_accuracy, ft_val_balanced_accuracy),
    ("F1", lgbm_f1, ft_val_f1),
    ("Recall", lgbm_recall, ft_val_recall),
    ("Precision", lgbm_precision, ft_val_precision),
    ("Accuracy", lgbm_accuracy, ft_val_accuracy)
]

print(
    f"{'Metric':<22}"
    f"{'LightGBM':>12}"
    f"{'FT-Transformer':>18}"
    f"{'Difference':>14}"
)

print("-" * 66)

for metric, lgbm_value, ft_value in comparison:

    print(
        f"{metric:<22}"
        f"{lgbm_value:>12.4f}"
        f"{ft_value:>18.4f}"
        f"{ft_value - lgbm_value:>+14.4f}"
    )

# ----------------------------------------------------------------------
# [16] TEST INTEGRITY
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[16] TEST INTEGRITY")
print("-" * 70)

print("✓ TEST was NOT used for training")
print("✓ TEST was NOT used for scheduler decisions")
print("✓ TEST was NOT used for early stopping")
print("✓ TEST predictions were NOT generated")
print("✓ TEST remains untouched")

# ----------------------------------------------------------------------
# [17] FINAL ASSERTIONS
# ----------------------------------------------------------------------

assert ft_transformer is not None

assert best_epoch > 0

assert len(validation_probability_ft) == len(
    y_validation_final
)

assert len(validation_prediction_ft) == len(
    y_validation_final
)

assert np.all(
    (validation_probability_ft >= 0) &
    (validation_probability_ft <= 1)
)

assert set(
    np.unique(validation_prediction_ft)
).issubset({0, 1})

assert len(ft_training_history) > 0

print("\n✓ FT-Transformer exists")
print("✓ Best epoch valid")
print("✓ Validation probabilities aligned")
print("✓ Validation predictions aligned")
print("✓ Probabilities valid")
print("✓ Training history available")
print("✓ TEST untouched")

print("\n" + "=" * 70)
print("STEP 12B FT-TRANSFORMER TRAINING COMPLETE")
print("=" * 70)

print("\nBEST MODEL")
print(
    f"  Epoch    : {best_epoch}"
)

print(
    f"  PR-AUC   : {best_validation_pr_auc:.4f}"
)

print(
    f"  ROC-AUC  : {ft_val_roc_auc:.4f}"
)

print("\nVALIDATION")

print(
    f"  Accuracy            : "
    f"{ft_val_accuracy:.4f}"
)

print(
    f"  Balanced Accuracy   : "
    f"{ft_val_balanced_accuracy:.4f}"
)

print(
    f"  Precision           : "
    f"{ft_val_precision:.4f}"
)

print(
    f"  Recall              : "
    f"{ft_val_recall:.4f}"
)

print(
    f"  F1                  : "
    f"{ft_val_f1:.4f}"
)

print(
    f"  ROC-AUC             : "
    f"{ft_val_roc_auc:.4f}"
)

print(
    f"  PR-AUC              : "
    f"{ft_val_pr_auc:.4f}"
)

print("\n✓ All STEP 12B assertions passed.")

STEP 12B — FT-TRANSFORMER TRAINING
✓ X_train_final exists
✓ y_train_final exists
✓ X_validation_final exists
✓ y_validation_final exists
✓ X_test_final exists
✓ y_test_final exists
✓ final_model_features exists

----------------------------------------------------------------------
[1] FEATURE ORDER
----------------------------------------------------------------------
[0] ordering_mode
[1] culture_description
[2] organism
[3] antibiotic
✓ Feature order verified

----------------------------------------------------------------------
[2] INPUT SHAPES
----------------------------------------------------------------------
X_train_final       : (1317556, 4)
X_validation_final  : (308772, 4)
X_test_final        : (311057, 4)
y_train_final       : (1317556,)
y_validation_final  : (308772,)
y_test_final        : (311057,)
✓ Four features confirmed
✓ TRAIN alignment verified
✓ VALIDATION alignment verified
✓ TEST alignment verified

-------------------------------------------------------------

C:\Users\HPCLAB\AppData\Local\Temp\ipykernel_22836\529044852.py:344: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Loss              : BCEWithLogitsLoss
Optimizer         : AdamW
Initial LR        : 0.001
Weight decay      : 0.0001
Scheduler         : ReduceLROnPlateau
Reduction factor  : 0.5
Scheduler patience: 5 epochs
Minimum LR        : 0.00001

----------------------------------------------------------------------
[12] STARTING FT-TRANSFORMER TRAINING
----------------------------------------------------------------------
Epoch 01/40 | Train Loss: 0.44294 | Val Loss: 0.37755 | Val PR-AUC: 0.63257 | Val ROC-AUC: 0.85622 | LR: 0.001000
  ✓ New best model — PR-AUC = 0.63257
Epoch 02/40 | Train Loss: 0.41991 | Val Loss: 0.39192 | Val PR-AUC: 0.63542 | Val ROC-AUC: 0.85750 | LR: 0.001000
  ✓ New best model — PR-AUC = 0.63542
Epoch 03/40 | Train Loss: 0.41686 | Val Loss: 0.38573 | Val PR-AUC: 0.63539 | Val ROC-AUC: 0.85697 | LR: 0.001000
Epoch 04/40 | Train Loss: 0.41504 | Val Loss: 0.38512 | Val PR-AUC: 0.63595 | Val ROC-AUC: 0.85860 | LR: 0.001000
  ✓ New best model — PR-AUC = 0.63595
Epoch 05/40 |

In [32]:
# ======================================================================
# STEP 13 — LIGHTGBM + FT-TRANSFORMER WEIGHTED SOFT-VOTING ENSEMBLE
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("STEP 13 — LIGHTGBM + FT-TRANSFORMER ENSEMBLE")
print("=" * 70)


# ======================================================================
# [0] REQUIRED OBJECT VERIFICATION
# ======================================================================

required_objects = [
    "best_lgbm_model",
    "best_lgbm_iteration",
    "X_validation_final",
    "y_validation_final",
]

for obj in required_objects:
    assert obj in globals(), f"Missing required object: {obj}"
    print(f"✓ {obj} exists")

print("✓ TEST will NOT be used")


# ======================================================================
# [1] FIND FT-TRANSFORMER VALIDATION PROBABILITIES
# ======================================================================

print("\n" + "-" * 70)
print("[1] FINDING FT-TRANSFORMER VALIDATION PROBABILITIES")
print("-" * 70)

# Step 12B should have created a validation probability array.
# We search common variable names rather than retraining the model.

ft_probability_candidates = [
    "ft_validation_probability",
    "validation_probability_ft",
    "validation_probability_transformer",
    "ft_val_probability",
    "validation_probability_ft_transformer",
]

ft_val_probability = None
ft_probability_source = None

for name in ft_probability_candidates:

    if name in globals():

        candidate = np.asarray(
            globals()[name],
            dtype=np.float64
        ).reshape(-1)

        if len(candidate) == len(y_validation_final):

            ft_val_probability = candidate
            ft_probability_source = name
            break


if ft_val_probability is None:

    raise RuntimeError(
        "FT-Transformer validation probabilities were not found.\n"
        "Step 12B must expose its validation probability array before "
        "the ensemble can be created.\n\n"
        "Expected one of:\n"
        + "\n".join(
            f"  - {x}" for x in ft_probability_candidates
        )
    )

print(
    f"✓ FT-Transformer validation probabilities found: "
    f"{ft_probability_source}"
)

assert len(ft_val_probability) == len(y_validation_final)

assert np.all(np.isfinite(ft_val_probability))

assert np.all(
    (ft_val_probability >= 0) &
    (ft_val_probability <= 1)
)

print("✓ FT-Transformer probabilities valid")


# ======================================================================
# [2] LIGHTGBM VALIDATION PROBABILITIES
# ======================================================================

print("\n" + "-" * 70)
print("[2] GENERATING LIGHTGBM VALIDATION PROBABILITIES")
print("-" * 70)

lgbm_val_probability = best_lgbm_model.predict_proba(
    X_validation_final,
    num_iteration=best_lgbm_iteration
)[:, 1]

lgbm_val_probability = np.asarray(
    lgbm_val_probability,
    dtype=np.float64
).reshape(-1)

print(
    f"LightGBM validation probability shape : "
    f"{lgbm_val_probability.shape}"
)

assert len(lgbm_val_probability) == len(y_validation_final)

assert np.all(np.isfinite(lgbm_val_probability))

assert np.all(
    (lgbm_val_probability >= 0) &
    (lgbm_val_probability <= 1)
)

print("✓ LightGBM probabilities valid")


# ======================================================================
# [3] VALIDATION TARGET
# ======================================================================

print("\n" + "-" * 70)
print("[3] VALIDATION ALIGNMENT")
print("-" * 70)

y_val = np.asarray(
    y_validation_final,
    dtype=np.int8
).reshape(-1)

print(f"Validation target rows       : {len(y_val):,}")
print(f"LightGBM probability rows    : {len(lgbm_val_probability):,}")
print(f"FT probability rows          : {len(ft_val_probability):,}")

assert len(y_val) == len(lgbm_val_probability)
assert len(y_val) == len(ft_val_probability)

assert set(np.unique(y_val)) == {0, 1}

print("✓ Target/probability alignment verified")
print("✓ Both models predict the same validation observations")


# ======================================================================
# [4] INDIVIDUAL MODEL BASELINES
# ======================================================================

print("\n" + "-" * 70)
print("[4] INDIVIDUAL MODEL BASELINES")
print("-" * 70)

lgbm_pr_auc = average_precision_score(
    y_val,
    lgbm_val_probability
)

ft_pr_auc = average_precision_score(
    y_val,
    ft_val_probability
)

lgbm_roc_auc = roc_auc_score(
    y_val,
    lgbm_val_probability
)

ft_roc_auc = roc_auc_score(
    y_val,
    ft_val_probability
)

print(f"LightGBM       PR-AUC  : {lgbm_pr_auc:.6f}")
print(f"FT-Transformer PR-AUC  : {ft_pr_auc:.6f}")

print(f"LightGBM       ROC-AUC : {lgbm_roc_auc:.6f}")
print(f"FT-Transformer ROC-AUC : {ft_roc_auc:.6f}")


# ======================================================================
# [5] WEIGHTED SOFT-VOTING SEARCH
# ======================================================================

print("\n" + "-" * 70)
print("[5] WEIGHTED SOFT-VOTING SEARCH")
print("-" * 70)

print(
    "Ensemble probability = "
    "w × LightGBM probability + "
    "(1-w) × FT-Transformer probability"
)

print("\nSearching LightGBM weights:")
print("0.00 → 1.00 in increments of 0.05")
print("TEST will NOT be used.")


weight_values = np.round(
    np.arange(0.00, 1.001, 0.05),
    2
)

ensemble_results = []

for w_lgbm in weight_values:

    w_ft = 1.0 - w_lgbm

    ensemble_probability = (
        w_lgbm * lgbm_val_probability
        +
        w_ft * ft_val_probability
    )

    pr_auc = average_precision_score(
        y_val,
        ensemble_probability
    )

    roc_auc = roc_auc_score(
        y_val,
        ensemble_probability
    )

    ensemble_results.append({
        "lightgbm_weight": w_lgbm,
        "ft_transformer_weight": w_ft,
        "validation_pr_auc": pr_auc,
        "validation_roc_auc": roc_auc
    })


ensemble_results = pd.DataFrame(
    ensemble_results
)

ensemble_results = ensemble_results.sort_values(
    "validation_pr_auc",
    ascending=False
).reset_index(drop=True)


print("\nWeight search results:")
print(
    ensemble_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ======================================================================
# [6] BEST ENSEMBLE WEIGHT
# ======================================================================

print("\n" + "-" * 70)
print("[6] BEST ENSEMBLE WEIGHT")
print("-" * 70)

best_row = ensemble_results.iloc[0]

best_lgbm_weight = float(
    best_row["lightgbm_weight"]
)

best_ft_weight = float(
    best_row["ft_transformer_weight"]
)

best_ensemble_pr_auc = float(
    best_row["validation_pr_auc"]
)

best_ensemble_roc_auc = float(
    best_row["validation_roc_auc"]
)

print(
    f"Best LightGBM weight       : "
    f"{best_lgbm_weight:.2f}"
)

print(
    f"Best FT-Transformer weight : "
    f"{best_ft_weight:.2f}"
)

print(
    f"Best validation PR-AUC     : "
    f"{best_ensemble_pr_auc:.6f}"
)

print(
    f"Best validation ROC-AUC    : "
    f"{best_ensemble_roc_auc:.6f}"
)


# ======================================================================
# [7] CREATE FINAL ENSEMBLE PROBABILITIES
# ======================================================================

print("\n" + "-" * 70)
print("[7] CREATING FINAL ENSEMBLE")
print("-" * 70)

ensemble_validation_probability = (
    best_lgbm_weight * lgbm_val_probability
    +
    best_ft_weight * ft_val_probability
)

assert len(
    ensemble_validation_probability
) == len(y_val)

assert np.all(
    np.isfinite(
        ensemble_validation_probability
    )
)

assert np.all(
    (ensemble_validation_probability >= 0)
    &
    (ensemble_validation_probability <= 1)
)

print("✓ Ensemble probabilities generated")
print("✓ Ensemble probabilities are valid")


# ======================================================================
# [8] 0.50 THRESHOLD EVALUATION
# ======================================================================

print("\n" + "-" * 70)
print("[8] ENSEMBLE VALIDATION EVALUATION")
print("-" * 70)

ensemble_validation_prediction = (
    ensemble_validation_probability >= 0.50
).astype(np.int8)


ensemble_accuracy = accuracy_score(
    y_val,
    ensemble_validation_prediction
)

ensemble_balanced_accuracy = balanced_accuracy_score(
    y_val,
    ensemble_validation_prediction
)

ensemble_precision = precision_score(
    y_val,
    ensemble_validation_prediction,
    zero_division=0
)

ensemble_recall = recall_score(
    y_val,
    ensemble_validation_prediction,
    zero_division=0
)

ensemble_f1 = f1_score(
    y_val,
    ensemble_validation_prediction,
    zero_division=0
)

ensemble_roc_auc = roc_auc_score(
    y_val,
    ensemble_validation_probability
)

ensemble_pr_auc = average_precision_score(
    y_val,
    ensemble_validation_probability
)


print(f"Accuracy            : {ensemble_accuracy:.4f}")
print(f"Balanced Accuracy   : {ensemble_balanced_accuracy:.4f}")
print(f"Precision            : {ensemble_precision:.4f}")
print(f"Recall               : {ensemble_recall:.4f}")
print(f"F1 Score             : {ensemble_f1:.4f}")
print(f"ROC-AUC              : {ensemble_roc_auc:.4f}")
print(f"PR-AUC               : {ensemble_pr_auc:.4f}")


# ======================================================================
# [9] CONFUSION MATRIX
# ======================================================================

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        ensemble_validation_prediction
    )
)


# ======================================================================
# [10] CLASSIFICATION REPORT
# ======================================================================

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        ensemble_validation_prediction,
        digits=4,
        zero_division=0
    )
)


# ======================================================================
# [11] MODEL COMPARISON
# ======================================================================

print("\n" + "-" * 70)
print("[11] MODEL COMPARISON")
print("-" * 70)

comparison = pd.DataFrame({

    "Model": [
        "LightGBM",
        "FT-Transformer",
        "Ensemble"
    ],

    "PR-AUC": [
        lgbm_pr_auc,
        ft_pr_auc,
        ensemble_pr_auc
    ],

    "ROC-AUC": [
        lgbm_roc_auc,
        ft_roc_auc,
        ensemble_roc_auc
    ]
})

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ======================================================================
# [12] ENSEMBLE GAIN
# ======================================================================

print("\n" + "-" * 70)
print("[12] ENSEMBLE GAIN")
print("-" * 70)

pr_auc_gain_vs_lgbm = (
    ensemble_pr_auc - lgbm_pr_auc
)

pr_auc_gain_vs_ft = (
    ensemble_pr_auc - ft_pr_auc
)

print(
    f"PR-AUC gain vs LightGBM       : "
    f"{pr_auc_gain_vs_lgbm:+.6f}"
)

print(
    f"PR-AUC gain vs FT-Transformer : "
    f"{pr_auc_gain_vs_ft:+.6f}"
)


if pr_auc_gain_vs_lgbm > 0:

    print(
        "\n✓ Ensemble improves validation PR-AUC "
        "over LightGBM."
    )

elif pr_auc_gain_vs_lgbm == 0:

    print(
        "\n= Ensemble matches LightGBM "
        "validation PR-AUC."
    )

else:

    print(
        "\n⚠ Ensemble does NOT improve "
        "validation PR-AUC over LightGBM."
    )

    print(
        "LightGBM remains the stronger validation model."
    )


# ======================================================================
# [13] FINAL ENSEMBLE OBJECTS
# ======================================================================

print("\n" + "-" * 70)
print("[13] FINAL ENSEMBLE OBJECTS")
print("-" * 70)

ensemble_weights_selected = {
    "LightGBM": best_lgbm_weight,
    "FT-Transformer": best_ft_weight
}

final_ensemble_model_name = (
    "LightGBM + FT-Transformer Weighted Soft Voting"
)

final_ensemble_probability = (
    ensemble_validation_probability
)

final_ensemble_prediction = (
    ensemble_validation_prediction
)

print(
    f"Model : {final_ensemble_model_name}"
)

print(
    f"LightGBM weight       : "
    f"{best_lgbm_weight:.2f}"
)

print(
    f"FT-Transformer weight : "
    f"{best_ft_weight:.2f}"
)

print("✓ Ensemble objects created")


# ======================================================================
# [14] TEST INTEGRITY
# ======================================================================

print("\n" + "-" * 70)
print("[14] TEST INTEGRITY")
print("-" * 70)

print(
    f"X_test_final shape : "
    f"{X_test_final.shape}"
)

print(
    f"y_test_final shape : "
    f"{y_test_final.shape}"
)

print("✓ TEST was NOT used")
print("✓ TEST was NOT used for weight selection")
print("✓ TEST was NOT used for evaluation")
print("✓ TEST predictions were NOT generated")
print("✓ TEST remains untouched")


# ======================================================================
# [15] FINAL ASSERTIONS
# ======================================================================

assert len(
    final_ensemble_probability
) == len(y_validation_final)

assert len(
    final_ensemble_prediction
) == len(y_validation_final)

assert (
    abs(
        best_lgbm_weight
        + best_ft_weight
        - 1.0
    ) < 1e-8
)

assert np.all(
    (final_ensemble_probability >= 0)
    &
    (final_ensemble_probability <= 1)
)

assert set(
    np.unique(final_ensemble_prediction)
).issubset({0, 1})

assert X_test_final.shape[0] == len(y_test_final)

print("\n" + "=" * 70)
print("STEP 13 WEIGHTED SOFT-VOTING ENSEMBLE COMPLETE")
print("=" * 70)

print("\nSELECTED WEIGHTS:")
print(
    f"  LightGBM       : "
    f"{best_lgbm_weight:.2f}"
)

print(
    f"  FT-Transformer : "
    f"{best_ft_weight:.2f}"
)

print("\nVALIDATION PERFORMANCE:")
print(
    f"  PR-AUC  : "
    f"{ensemble_pr_auc:.4f}"
)

print(
    f"  ROC-AUC : "
    f"{ensemble_roc_auc:.4f}"
)

print(
    f"  F1      : "
    f"{ensemble_f1:.4f}"
)

print(
    f"  Recall  : "
    f"{ensemble_recall:.4f}"
)

print(
    f"  Precision : "
    f"{ensemble_precision:.4f}"
)

print("\nDATA USAGE:")
print("  ✓ TRAIN used by both previously trained models")
print("  ✓ VALIDATION used for ensemble weight selection")
print("  ✓ TEST completely untouched")

print("\nNEXT STEP:")
print("  → Optimize decision threshold using VALIDATION only")
print("  → Freeze ensemble weights + threshold")
print("  → Perform ONE final evaluation on TEST")

print("\n✓ All STEP 13 assertions passed.")

STEP 13 — LIGHTGBM + FT-TRANSFORMER ENSEMBLE
✓ best_lgbm_model exists
✓ best_lgbm_iteration exists
✓ X_validation_final exists
✓ y_validation_final exists
✓ TEST will NOT be used

----------------------------------------------------------------------
[1] FINDING FT-TRANSFORMER VALIDATION PROBABILITIES
----------------------------------------------------------------------
✓ FT-Transformer validation probabilities found: validation_probability_ft
✓ FT-Transformer probabilities valid

----------------------------------------------------------------------
[2] GENERATING LIGHTGBM VALIDATION PROBABILITIES
----------------------------------------------------------------------
LightGBM validation probability shape : (308772,)
✓ LightGBM probabilities valid

----------------------------------------------------------------------
[3] VALIDATION ALIGNMENT
----------------------------------------------------------------------
Validation target rows       : 308,772
LightGBM probability rows    : 308

In [33]:
# ======================================================================
# STEP 14 — VALIDATION-BASED THRESHOLD OPTIMIZATION
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("STEP 14 — VALIDATION-BASED THRESHOLD OPTIMIZATION")
print("=" * 70)


# ======================================================================
# [0] REQUIRED OBJECT VERIFICATION
# ======================================================================

required_objects = [
    "final_ensemble_probability",
    "y_validation_final",
    "best_lgbm_weight",
    "best_ft_weight",
]

for obj in required_objects:

    assert obj in globals(), f"Missing required object: {obj}"

    print(f"✓ {obj} exists")


# ======================================================================
# [1] INPUT VERIFICATION
# ======================================================================

print("\n" + "-" * 70)
print("[1] INPUT VERIFICATION")
print("-" * 70)

ensemble_probability = np.asarray(
    final_ensemble_probability,
    dtype=np.float64
).reshape(-1)

y_val = np.asarray(
    y_validation_final,
    dtype=np.int8
).reshape(-1)

print(
    f"Validation target rows       : {len(y_val):,}"
)

print(
    f"Ensemble probability rows    : "
    f"{len(ensemble_probability):,}"
)

assert len(ensemble_probability) == len(y_val)

assert set(np.unique(y_val)) == {0, 1}

assert np.all(np.isfinite(ensemble_probability))

assert np.all(
    (ensemble_probability >= 0)
    &
    (ensemble_probability <= 1)
)

print("✓ Validation target valid")
print("✓ Ensemble probabilities valid")
print("✓ Lengths aligned")


# ======================================================================
# [2] FREEZE ENSEMBLE WEIGHTS
# ======================================================================

print("\n" + "-" * 70)
print("[2] FREEZING ENSEMBLE WEIGHTS")
print("-" * 70)

frozen_lgbm_weight = float(best_lgbm_weight)
frozen_ft_weight = float(best_ft_weight)

assert abs(
    frozen_lgbm_weight
    + frozen_ft_weight
    - 1.0
) < 1e-8

print(
    f"LightGBM weight       : "
    f"{frozen_lgbm_weight:.2f}"
)

print(
    f"FT-Transformer weight : "
    f"{frozen_ft_weight:.2f}"
)

print("✓ Ensemble weights frozen")


# ======================================================================
# [3] BASELINE AT THRESHOLD 0.50
# ======================================================================

print("\n" + "-" * 70)
print("[3] BASELINE THRESHOLD = 0.50")
print("-" * 70)

baseline_threshold = 0.50

baseline_prediction = (
    ensemble_probability >= baseline_threshold
).astype(np.int8)

baseline_accuracy = accuracy_score(
    y_val,
    baseline_prediction
)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_val,
    baseline_prediction
)

baseline_precision = precision_score(
    y_val,
    baseline_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_val,
    baseline_prediction,
    zero_division=0
)

baseline_f1 = f1_score(
    y_val,
    baseline_prediction,
    zero_division=0
)

baseline_cm = confusion_matrix(
    y_val,
    baseline_prediction
)

baseline_tn, baseline_fp, baseline_fn, baseline_tp = (
    baseline_cm.ravel()
)

baseline_specificity = (
    baseline_tn /
    (baseline_tn + baseline_fp)
)

print(f"Accuracy          : {baseline_accuracy:.4f}")
print(f"Balanced Accuracy : {baseline_balanced_accuracy:.4f}")
print(f"Precision         : {baseline_precision:.4f}")
print(f"Recall            : {baseline_recall:.4f}")
print(f"Specificity       : {baseline_specificity:.4f}")
print(f"F1                : {baseline_f1:.4f}")


# ======================================================================
# [4] THRESHOLD SEARCH
# ======================================================================

print("\n" + "-" * 70)
print("[4] VALIDATION THRESHOLD SEARCH")
print("-" * 70)

print("Optimization objective:")
print("  MAXIMIZE VALIDATION F1")

print()
print("Threshold search range:")
print("  0.05 → 0.95")
print("  increment = 0.01")

thresholds = np.round(
    np.arange(0.05, 0.951, 0.01),
    2
)

threshold_results = []

for threshold in thresholds:

    prediction = (
        ensemble_probability >= threshold
    ).astype(np.int8)

    accuracy = accuracy_score(
        y_val,
        prediction
    )

    balanced_accuracy = balanced_accuracy_score(
        y_val,
        prediction
    )

    precision = precision_score(
        y_val,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        prediction,
        zero_division=0
    )

    cm = confusion_matrix(
        y_val,
        prediction
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn /
        (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    threshold_results.append({

        "threshold": threshold,

        "accuracy": accuracy,

        "balanced_accuracy": balanced_accuracy,

        "precision": precision,

        "recall_sensitivity": recall,

        "specificity": specificity,

        "f1": f1
    })


threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results = threshold_results.sort_values(
    ["f1", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)


print("\nTop threshold candidates:")

print(
    threshold_results.head(10).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ======================================================================
# [5] SELECT BEST THRESHOLD
# ======================================================================

print("\n" + "-" * 70)
print("[5] SELECTING OPTIMAL THRESHOLD")
print("-" * 70)

best_threshold_row = threshold_results.iloc[0]

FINAL_THRESHOLD = float(
    best_threshold_row["threshold"]
)

print(
    f"Selected threshold : "
    f"{FINAL_THRESHOLD:.2f}"
)

print(
    f"Validation F1       : "
    f"{best_threshold_row['f1']:.4f}"
)

print(
    f"Validation Recall   : "
    f"{best_threshold_row['recall_sensitivity']:.4f}"
)

print(
    f"Validation Specificity : "
    f"{best_threshold_row['specificity']:.4f}"
)

print("✓ Threshold selected using VALIDATION only")


# ======================================================================
# [6] FROZEN-THRESHOLD VALIDATION EVALUATION
# ======================================================================

print("\n" + "-" * 70)
print("[6] FROZEN-THRESHOLD VALIDATION EVALUATION")
print("-" * 70)

ensemble_validation_prediction_frozen = (
    ensemble_probability >= FINAL_THRESHOLD
).astype(np.int8)

final_accuracy = accuracy_score(
    y_val,
    ensemble_validation_prediction_frozen
)

final_balanced_accuracy = balanced_accuracy_score(
    y_val,
    ensemble_validation_prediction_frozen
)

final_precision = precision_score(
    y_val,
    ensemble_validation_prediction_frozen,
    zero_division=0
)

final_recall = recall_score(
    y_val,
    ensemble_validation_prediction_frozen,
    zero_division=0
)

final_f1 = f1_score(
    y_val,
    ensemble_validation_prediction_frozen,
    zero_division=0
)

final_roc_auc = roc_auc_score(
    y_val,
    ensemble_probability
)

final_pr_auc = average_precision_score(
    y_val,
    ensemble_probability
)

cm = confusion_matrix(
    y_val,
    ensemble_validation_prediction_frozen
)

tn, fp, fn, tp = cm.ravel()

final_specificity = (
    tn /
    (tn + fp)
)


print(f"Threshold          : {FINAL_THRESHOLD:.2f}")
print(f"Accuracy           : {final_accuracy:.4f}")
print(f"Balanced Accuracy  : {final_balanced_accuracy:.4f}")
print(f"Precision          : {final_precision:.4f}")
print(f"Recall/Sensitivity : {final_recall:.4f}")
print(f"Specificity        : {final_specificity:.4f}")
print(f"F1                 : {final_f1:.4f}")
print(f"ROC-AUC            : {final_roc_auc:.4f}")
print(f"PR-AUC             : {final_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)


# ======================================================================
# [7] COMPARE THRESHOLD 0.50 VS OPTIMIZED
# ======================================================================

print("\n" + "-" * 70)
print("[7] THRESHOLD COMPARISON")
print("-" * 70)

threshold_comparison = pd.DataFrame({

    "Metric": [
        "Threshold",
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall/Sensitivity",
        "Specificity",
        "F1"
    ],

    "Threshold_0.50": [
        0.50,
        baseline_accuracy,
        baseline_balanced_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_specificity,
        baseline_f1
    ],

    "Optimized": [
        FINAL_THRESHOLD,
        final_accuracy,
        final_balanced_accuracy,
        final_precision,
        final_recall,
        final_specificity,
        final_f1
    ]
})

print(
    threshold_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ======================================================================
# [8] FREEZE FINAL DECISION RULE
# ======================================================================

print("\n" + "-" * 70)
print("[8] FREEZING FINAL DECISION RULE")
print("-" * 70)

FINAL_ENSEMBLE_LGBM_WEIGHT = frozen_lgbm_weight
FINAL_ENSEMBLE_FT_WEIGHT = frozen_ft_weight

FINAL_ENSEMBLE_THRESHOLD = FINAL_THRESHOLD

print(
    f"LightGBM weight       : "
    f"{FINAL_ENSEMBLE_LGBM_WEIGHT:.2f}"
)

print(
    f"FT-Transformer weight : "
    f"{FINAL_ENSEMBLE_FT_WEIGHT:.2f}"
)

print(
    f"Classification threshold : "
    f"{FINAL_ENSEMBLE_THRESHOLD:.2f}"
)

print("✓ Ensemble weights frozen")
print("✓ Classification threshold frozen")


# ======================================================================
# [9] TEST INTEGRITY
# ======================================================================

print("\n" + "-" * 70)
print("[9] TEST INTEGRITY")
print("-" * 70)

print(
    f"X_test_final : {X_test_final.shape}"
)

print(
    f"y_test_final : {y_test_final.shape}"
)

print("✓ TEST predictions NOT generated")
print("✓ TEST NOT used for threshold selection")
print("✓ TEST remains completely untouched")


# ======================================================================
# [10] FINAL ASSERTIONS
# ======================================================================

assert 0.0 <= FINAL_ENSEMBLE_LGBM_WEIGHT <= 1.0

assert 0.0 <= FINAL_ENSEMBLE_FT_WEIGHT <= 1.0

assert abs(
    FINAL_ENSEMBLE_LGBM_WEIGHT
    + FINAL_ENSEMBLE_FT_WEIGHT
    - 1.0
) < 1e-8

assert 0.05 <= FINAL_ENSEMBLE_THRESHOLD <= 0.95

assert len(
    ensemble_validation_prediction_frozen
) == len(y_val)

assert set(
    np.unique(
        ensemble_validation_prediction_frozen
    )
).issubset({0, 1})

assert X_test_final.shape[0] == len(y_test_final)

print("\n" + "=" * 70)
print("STEP 14 THRESHOLD OPTIMIZATION COMPLETE")
print("=" * 70)

print("\nFROZEN ENSEMBLE:")
print(
    f"  LightGBM       : "
    f"{FINAL_ENSEMBLE_LGBM_WEIGHT:.2f}"
)

print(
    f"  FT-Transformer : "
    f"{FINAL_ENSEMBLE_FT_WEIGHT:.2f}"
)

print("\nFROZEN THRESHOLD:")
print(
    f"  {FINAL_ENSEMBLE_THRESHOLD:.2f}"
)

print("\nVALIDATION:")
print(
    f"  PR-AUC            : "
    f"{final_pr_auc:.4f}"
)

print(
    f"  ROC-AUC           : "
    f"{final_roc_auc:.4f}"
)

print(
    f"  F1                : "
    f"{final_f1:.4f}"
)

print(
    f"  Sensitivity       : "
    f"{final_recall:.4f}"
)

print(
    f"  Specificity       : "
    f"{final_specificity:.4f}"
)

print(
    f"  Balanced Accuracy : "
    f"{final_balanced_accuracy:.4f}"
)

print("\nTEST STATUS:")
print("  ✓ Completely untouched")
print("  ✓ No predictions generated")
print("  ✓ No threshold optimization")
print("  ✓ No model modification")

print("\n✓ All STEP 14 assertions passed.")

STEP 14 — VALIDATION-BASED THRESHOLD OPTIMIZATION
✓ final_ensemble_probability exists
✓ y_validation_final exists
✓ best_lgbm_weight exists
✓ best_ft_weight exists

----------------------------------------------------------------------
[1] INPUT VERIFICATION
----------------------------------------------------------------------
Validation target rows       : 308,772
Ensemble probability rows    : 308,772
✓ Validation target valid
✓ Ensemble probabilities valid
✓ Lengths aligned

----------------------------------------------------------------------
[2] FREEZING ENSEMBLE WEIGHTS
----------------------------------------------------------------------
LightGBM weight       : 0.80
FT-Transformer weight : 0.20
✓ Ensemble weights frozen

----------------------------------------------------------------------
[3] BASELINE THRESHOLD = 0.50
----------------------------------------------------------------------
Accuracy          : 0.8045
Balanced Accuracy : 0.7663
Precision         : 0.4555
Recall

In [34]:
# ======================================================================
# STEP 15 — ONE-TIME FINAL TEST EVALUATION
# ======================================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 15 — ONE-TIME FINAL TEST EVALUATION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED-OBJECT VERIFICATION
# ----------------------------------------------------------------------

required_objects = [
    "best_lgbm_model",
    "X_test_final",
    "y_test_final",
    "best_lgbm_weight",
    "best_ft_weight",
    "FINAL_ENSEMBLE_THRESHOLD"
]

print("\n[1] REQUIRED-OBJECT VERIFICATION")
print("-" * 70)

missing_objects = []

for obj_name in required_objects:
    if obj_name in globals():
        print(f"✓ {obj_name} available")
    else:
        print(f"✗ {obj_name} MISSING")
        missing_objects.append(obj_name)

assert not missing_objects, (
    f"STEP 15 STOPPED — Missing required objects: {missing_objects}"
)


# ----------------------------------------------------------------------
# [2] VERIFY FROZEN DECISION RULE
# ----------------------------------------------------------------------

print("\n[2] FROZEN DECISION RULE VERIFICATION")
print("-" * 70)

FINAL_LGBM_WEIGHT = float(best_lgbm_weight)
FINAL_FT_WEIGHT = float(best_ft_weight)
FINAL_THRESHOLD = float(FINAL_ENSEMBLE_THRESHOLD)

print(f"LightGBM weight       : {FINAL_LGBM_WEIGHT:.2f}")
print(f"FT-Transformer weight : {FINAL_FT_WEIGHT:.2f}")
print(f"Classification threshold : {FINAL_THRESHOLD:.2f}")

assert np.isclose(FINAL_LGBM_WEIGHT, 0.80), \
    "LightGBM weight is not the frozen 0.80 value."

assert np.isclose(FINAL_FT_WEIGHT, 0.20), \
    "FT-Transformer weight is not the frozen 0.20 value."

assert np.isclose(FINAL_THRESHOLD, 0.57), \
    "Classification threshold is not the frozen 0.57 value."

assert np.isclose(
    FINAL_LGBM_WEIGHT + FINAL_FT_WEIGHT,
    1.0
), "Ensemble weights do not sum to 1."

print("✓ Ensemble weights frozen")
print("✓ Classification threshold frozen")


# ----------------------------------------------------------------------
# [3] VERIFY TEST DATA
# ----------------------------------------------------------------------

print("\n[3] TEST DATA VERIFICATION")
print("-" * 70)

assert len(X_test_final) == len(y_test_final), \
    "TEST feature/target lengths do not match."

assert len(X_test_final) > 0, \
    "TEST set is empty."

assert not pd.isna(y_test_final).any(), \
    "TEST target contains missing values."

unique_targets = np.unique(np.asarray(y_test_final))

assert set(unique_targets).issubset({0, 1}), \
    f"Unexpected TEST target values: {unique_targets}"

print(f"X_test_final shape : {X_test_final.shape}")
print(f"y_test_final shape : {y_test_final.shape}")
print(f"TEST rows          : {len(y_test_final):,}")
print(f"TEST target values : {unique_targets}")

print("✓ TEST data valid")


# ----------------------------------------------------------------------
# [4] LIGHTGBM TEST PROBABILITY
# ----------------------------------------------------------------------

print("\n[4] LIGHTGBM TEST PREDICTION")
print("-" * 70)

# Native categorical LightGBM model
test_probability_lgbm = best_lgbm_model.predict(
    X_test_final,
    num_iteration=best_lgbm_iteration
)

test_probability_lgbm = np.asarray(
    test_probability_lgbm,
    dtype=np.float64
).reshape(-1)

assert len(test_probability_lgbm) == len(y_test_final), \
    "LightGBM TEST probability length mismatch."

assert np.all(np.isfinite(test_probability_lgbm)), \
    "LightGBM TEST probabilities contain non-finite values."

assert np.all(
    (test_probability_lgbm >= 0) &
    (test_probability_lgbm <= 1)
), "LightGBM TEST probabilities outside [0,1]."

print(f"✓ LightGBM TEST probabilities generated")
print(f"  Rows : {len(test_probability_lgbm):,}")


# ----------------------------------------------------------------------
# [5] LOCATE FT-TRANSFORMER MODEL
# ----------------------------------------------------------------------

print("\n[5] FT-TRANSFORMER MODEL VERIFICATION")
print("-" * 70)

ft_model_candidates = [
    "ft_model",
    "ft_transformer_model",
    "best_ft_model",
    "best_ft_transformer",
    "ft_transformer"
]

available_ft_models = [
    name for name in ft_model_candidates
    if name in globals()
]

if len(available_ft_models) == 0:
    raise RuntimeError(
        "STEP 15 STOPPED — Could not identify the trained "
        "FT-Transformer model. Send the Step 12B output/object names "
        "before continuing. TEST has not been evaluated by FT-Transformer."
    )

if len(available_ft_models) > 1:
    print(
        "Multiple FT-Transformer candidates found:",
        available_ft_models
    )

FT_MODEL_NAME = available_ft_models[0]
ft_model = globals()[FT_MODEL_NAME]

print(f"✓ FT-Transformer model found: {FT_MODEL_NAME}")


# ----------------------------------------------------------------------
# [6] LOCATE FT TEST TENSORS
# ----------------------------------------------------------------------

print("\n[6] FT-TRANSFORMER TEST TENSOR VERIFICATION")
print("-" * 70)

test_num_candidates = [
    "test_num_tensor",
    "X_test_num_tensor",
    "test_numeric_tensor"
]

test_cat_candidates = [
    "test_cat_tensor",
    "X_test_cat_tensor",
    "test_categorical_tensor"
]

test_target_candidates = [
    "test_target_tensor",
    "y_test_tensor"
]

available_num = [
    name for name in test_num_candidates
    if name in globals()
]

available_cat = [
    name for name in test_cat_candidates
    if name in globals()
]

available_target = [
    name for name in test_target_candidates
    if name in globals()
]

if len(available_num) == 0 or len(available_cat) == 0:
    raise RuntimeError(
        "STEP 15 STOPPED — Could not identify the FT-Transformer "
        "TEST tensors. Send the Step 12B tensor object names before "
        "continuing. TEST has not been evaluated by FT-Transformer."
    )

FT_TEST_NUM_NAME = available_num[0]
FT_TEST_CAT_NAME = available_cat[0]

test_num_tensor = globals()[FT_TEST_NUM_NAME]
test_cat_tensor = globals()[FT_TEST_CAT_NAME]

print(f"✓ Numeric tensor : {FT_TEST_NUM_NAME}")
print(f"✓ Categorical tensor : {FT_TEST_CAT_NAME}")


# ----------------------------------------------------------------------
# [7] FT-TRANSFORMER TEST PROBABILITY
# ----------------------------------------------------------------------

print("\n[7] FT-TRANSFORMER TEST PREDICTION")
print("-" * 70)

ft_model.eval()

with torch.no_grad():
    ft_logits = ft_model(
        test_num_tensor,
        test_cat_tensor
    )

    # Handle common output shapes safely
    if isinstance(ft_logits, tuple):
        ft_logits = ft_logits[0]

    ft_logits = ft_logits.reshape(-1)

    test_probability_ft = torch.sigmoid(
        ft_logits
    ).detach().cpu().numpy().astype(np.float64)

assert len(test_probability_ft) == len(y_test_final), \
    "FT-Transformer TEST probability length mismatch."

assert np.all(np.isfinite(test_probability_ft)), \
    "FT-Transformer TEST probabilities contain non-finite values."

assert np.all(
    (test_probability_ft >= 0) &
    (test_probability_ft <= 1)
), "FT-Transformer TEST probabilities outside [0,1]."

print("✓ FT-Transformer TEST probabilities generated")
print(f"  Rows : {len(test_probability_ft):,}")


# ----------------------------------------------------------------------
# [8] ENSEMBLE TEST PROBABILITY
# ----------------------------------------------------------------------

print("\n[8] FROZEN ENSEMBLE TEST PROBABILITY")
print("-" * 70)

ensemble_test_probability = (
    FINAL_LGBM_WEIGHT * test_probability_lgbm
    +
    FINAL_FT_WEIGHT * test_probability_ft
)

assert len(ensemble_test_probability) == len(y_test_final), \
    "Ensemble TEST probability length mismatch."

assert np.all(np.isfinite(ensemble_test_probability)), \
    "Ensemble TEST probabilities contain non-finite values."

assert np.all(
    (ensemble_test_probability >= 0) &
    (ensemble_test_probability <= 1)
), "Ensemble TEST probabilities outside [0,1]."

print(
    f"Formula: "
    f"{FINAL_LGBM_WEIGHT:.2f} × LightGBM + "
    f"{FINAL_FT_WEIGHT:.2f} × FT-Transformer"
)

print("✓ Frozen ensemble applied")


# ----------------------------------------------------------------------
# [9] APPLY FROZEN THRESHOLD
# ----------------------------------------------------------------------

print("\n[9] APPLYING FROZEN THRESHOLD")
print("-" * 70)

y_test_array = np.asarray(y_test_final).astype(int)

test_prediction_final = (
    ensemble_test_probability >= FINAL_THRESHOLD
).astype(int)

assert len(test_prediction_final) == len(y_test_array), \
    "Final TEST prediction length mismatch."

print(f"Frozen threshold : {FINAL_THRESHOLD:.2f}")
print("✓ No threshold search performed")
print("✓ Frozen threshold applied")


# ----------------------------------------------------------------------
# [10] FINAL TEST METRICS
# ----------------------------------------------------------------------

print("\n[10] FINAL TEST EVALUATION")
print("-" * 70)

test_accuracy = accuracy_score(
    y_test_array,
    test_prediction_final
)

test_balanced_accuracy = balanced_accuracy_score(
    y_test_array,
    test_prediction_final
)

test_precision = precision_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_recall = recall_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_f1 = f1_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test_array,
    ensemble_test_probability
)

test_pr_auc = average_precision_score(
    y_test_array,
    ensemble_test_probability
)

test_cm = confusion_matrix(
    y_test_array,
    test_prediction_final
)

tn, fp, fn, tp = test_cm.ravel()

test_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

print(f"Accuracy          : {test_accuracy:.4f}")
print(f"Balanced Accuracy : {test_balanced_accuracy:.4f}")
print(f"Precision         : {test_precision:.4f}")
print(f"Recall/Sensitivity : {test_recall:.4f}")
print(f"Specificity       : {test_specificity:.4f}")
print(f"F1                : {test_f1:.4f}")
print(f"ROC-AUC           : {test_roc_auc:.4f}")
print(f"PR-AUC             : {test_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(test_cm)


# ----------------------------------------------------------------------
# [11] TEST CLASS DISTRIBUTION
# ----------------------------------------------------------------------

print("\n[11] TEST PREDICTION DISTRIBUTION")
print("-" * 70)

actual_positive = int(y_test_array.sum())
predicted_positive = int(test_prediction_final.sum())

print(f"Actual resistant cases    : {actual_positive:,}")
print(f"Actual susceptible cases  : {len(y_test_array) - actual_positive:,}")
print(f"Predicted resistant cases : {predicted_positive:,}")
print(
    f"Predicted susceptible cases : "
    f"{len(y_test_array) - predicted_positive:,}"
)


# ----------------------------------------------------------------------
# [12] FINAL TEST INTEGRITY ASSERTIONS
# ----------------------------------------------------------------------

print("\n[12] FINAL TEST INTEGRITY")
print("-" * 70)

assert np.isclose(
    FINAL_LGBM_WEIGHT,
    0.80
)

assert np.isclose(
    FINAL_FT_WEIGHT,
    0.20
)

assert np.isclose(
    FINAL_THRESHOLD,
    0.57
)

assert len(ensemble_test_probability) == 311057
assert len(test_prediction_final) == 311057

print("✓ Frozen ensemble weights unchanged")
print("✓ Frozen threshold unchanged")
print("✓ TEST rows preserved")
print("✓ No TEST-based tuning performed")
print("✓ No TEST-based model modification performed")


# ----------------------------------------------------------------------
# [13] SAVE FINAL TEST OBJECTS
# ----------------------------------------------------------------------

final_test_metrics = {
    "ensemble_lgbm_weight": FINAL_LGBM_WEIGHT,
    "ensemble_ft_weight": FINAL_FT_WEIGHT,
    "classification_threshold": FINAL_THRESHOLD,
    "accuracy": test_accuracy,
    "balanced_accuracy": test_balanced_accuracy,
    "precision": test_precision,
    "recall_sensitivity": test_recall,
    "specificity": test_specificity,
    "f1": test_f1,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc,
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp)
}

print("\n" + "=" * 70)
print("STEP 15 FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

print("\nFINAL FROZEN DECISION RULE:")
print(f"  LightGBM       : {FINAL_LGBM_WEIGHT:.2f}")
print(f"  FT-Transformer : {FINAL_FT_WEIGHT:.2f}")
print(f"  Threshold      : {FINAL_THRESHOLD:.2f}")

print("\nFINAL TEST RESULTS:")
print(f"  PR-AUC            : {test_pr_auc:.4f}")
print(f"  ROC-AUC           : {test_roc_auc:.4f}")
print(f"  F1                : {test_f1:.4f}")
print(f"  Sensitivity       : {test_recall:.4f}")
print(f"  Specificity       : {test_specificity:.4f}")
print(f"  Balanced Accuracy : {test_balanced_accuracy:.4f}")
print(f"  Accuracy          : {test_accuracy:.4f}")

print("\n✓ TEST evaluated using frozen decision rule")
print("✓ No TEST optimization performed")
print("✓ Final TEST evaluation completed")
print("=" * 70)

STEP 15 — ONE-TIME FINAL TEST EVALUATION

[1] REQUIRED-OBJECT VERIFICATION
----------------------------------------------------------------------
✓ best_lgbm_model available
✓ X_test_final available
✓ y_test_final available
✓ best_lgbm_weight available
✓ best_ft_weight available
✓ FINAL_ENSEMBLE_THRESHOLD available

[2] FROZEN DECISION RULE VERIFICATION
----------------------------------------------------------------------
LightGBM weight       : 0.80
FT-Transformer weight : 0.20
Classification threshold : 0.57
✓ Ensemble weights frozen
✓ Classification threshold frozen

[3] TEST DATA VERIFICATION
----------------------------------------------------------------------
X_test_final shape : (311057, 4)
y_test_final shape : (311057,)
TEST rows          : 311,057
TEST target values : [0 1]
✓ TEST data valid

[4] LIGHTGBM TEST PREDICTION
----------------------------------------------------------------------
✓ LightGBM TEST probabilities generated
  Rows : 311,057

[5] FT-TRANSFORMER MODEL VE

RuntimeError: STEP 15 STOPPED — Could not identify the FT-Transformer TEST tensors. Send the Step 12B tensor object names before continuing. TEST has not been evaluated by FT-Transformer.

In [35]:
# ======================================================================
# STEP 15A — IDENTIFY FT-TRANSFORMER TEST OBJECTS
# ======================================================================

print("=" * 70)
print("STEP 15A — FT-TRANSFORMER OBJECT IDENTIFICATION")
print("=" * 70)

print("\n[1] VARIABLES RELATED TO FT-TRANSFORMER / TEST")
print("-" * 70)

keywords = [
    "ft",
    "transformer",
    "test",
    "tensor",
    "cat",
    "num"
]

matching_names = sorted([
    name for name in globals()
    if any(k in name.lower() for k in keywords)
])

for name in matching_names:
    obj = globals()[name]

    try:
        shape = getattr(obj, "shape", None)
    except Exception:
        shape = None

    print(f"{name:<40} | type={type(obj).__name__:<25} | shape={shape}")


print("\n[2] POSSIBLE TRAINED MODEL OBJECTS")
print("-" * 70)

for name, obj in globals().items():
    if isinstance(obj, torch.nn.Module):
        print(
            f"{name:<40} | "
            f"class={obj.__class__.__name__}"
        )


print("\n[3] POSSIBLE TEST TENSORS")
print("-" * 70)

for name, obj in globals().items():

    if isinstance(obj, torch.Tensor):
        print(
            f"{name:<40} | "
            f"dtype={obj.dtype} | "
            f"shape={tuple(obj.shape)}"
        )


print("\n" + "=" * 70)
print("STEP 15A COMPLETE")
print("=" * 70)

print(
    "\nSTOP HERE.\n"
    "Do NOT run STEP 15 again yet.\n"
    "Send me the COMPLETE output from this cell."
)

STEP 15A — FT-TRANSFORMER OBJECT IDENTIFICATION

[1] VARIABLES RELATED TO FT-TRANSFORMER / TEST
----------------------------------------------------------------------
FINAL_ENSEMBLE_FT_WEIGHT                 | type=float                     | shape=None
FINAL_FT_WEIGHT                          | type=float                     | shape=None
FTTransformer                            | type=type                      | shape=None
FT_MODEL_NAME                            | type=str                       | shape=None
TensorDataset                            | type=type                      | shape=None
X_test_check                             | type=ndarray                   | shape=(311057, 4)
X_test_encoded                           | type=DataFrame                 | shape=(311057, 4)
X_test_encoded_temp                      | type=ndarray                   | shape=(311057, 4)
X_test_final                             | type=ndarray                   | shape=(311057, 4)
X_test_lgb            

In [37]:
# ======================================================================
# STEP 15C — VERIFY FT-TRANSFORMER FORWARD INPUT
# ======================================================================

import inspect
import torch

print("=" * 70)
print("STEP 15C — FT-TRANSFORMER FORWARD INPUT VERIFICATION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] MODEL VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] MODEL VERIFICATION")
print("-" * 70)

assert "ft_transformer" in globals(), \
    "ft_transformer is missing."

assert isinstance(ft_transformer, torch.nn.Module), \
    "ft_transformer is not a PyTorch model."

print(f"Model class : {ft_transformer.__class__.__name__}")


# ----------------------------------------------------------------------
# [2] FORWARD SIGNATURE
# ----------------------------------------------------------------------

print("\n[2] FORWARD SIGNATURE")
print("-" * 70)

forward_signature = inspect.signature(
    ft_transformer.forward
)

print(f"ft_transformer.forward{forward_signature}")


# ----------------------------------------------------------------------
# [3] TEST LOADER VERIFICATION
# ----------------------------------------------------------------------

print("\n[3] TEST LOADER VERIFICATION")
print("-" * 70)

assert "test_loader" in globals(), \
    "test_loader is missing."

first_batch = next(iter(test_loader))

print(f"Batch type : {type(first_batch).__name__}")

if isinstance(first_batch, (tuple, list)):

    print(f"Number of batch elements : {len(first_batch)}")

    for i, item in enumerate(first_batch):

        if isinstance(item, torch.Tensor):

            print(
                f"Element {i}: "
                f"type=Tensor, "
                f"shape={tuple(item.shape)}, "
                f"dtype={item.dtype}"
            )

        else:

            print(
                f"Element {i}: "
                f"type={type(item).__name__}"
            )

elif isinstance(first_batch, torch.Tensor):

    print(
        f"Single tensor batch: "
        f"shape={tuple(first_batch.shape)}, "
        f"dtype={first_batch.dtype}"
    )

else:

    print(
        f"Unexpected batch type: "
        f"{type(first_batch).__name__}"
    )


# ----------------------------------------------------------------------
# [4] DATASET STRUCTURE
# ----------------------------------------------------------------------

print("\n[4] TEST DATASET STRUCTURE")
print("-" * 70)

assert hasattr(test_loader, "dataset"), \
    "test_loader has no dataset."

print(
    f"Dataset class : "
    f"{test_loader.dataset.__class__.__name__}"
)

try:
    dataset_sample = test_loader.dataset[0]

    if isinstance(dataset_sample, (tuple, list)):

        print(
            f"Dataset sample elements : "
            f"{len(dataset_sample)}"
        )

        for i, item in enumerate(dataset_sample):

            if isinstance(item, torch.Tensor):

                print(
                    f"Element {i}: "
                    f"shape={tuple(item.shape)}, "
                    f"dtype={item.dtype}"
                )

            else:

                print(
                    f"Element {i}: "
                    f"type={type(item).__name__}"
                )

    elif isinstance(dataset_sample, torch.Tensor):

        print(
            f"Single tensor sample: "
            f"shape={tuple(dataset_sample.shape)}, "
            f"dtype={dataset_sample.dtype}"
        )

    else:

        print(
            f"Unexpected dataset sample type: "
            f"{type(dataset_sample).__name__}"
        )

except Exception as e:

    print(f"Could not inspect dataset sample: {e}")


# ----------------------------------------------------------------------
# [5] MODEL INPUT EXPECTATION
# ----------------------------------------------------------------------

print("\n[5] INTERPRETATION")
print("-" * 70)

parameters = list(
    inspect.signature(
        ft_transformer.forward
    ).parameters.values()
)

non_self_parameters = [
    p for p in parameters
    if p.name != "self"
]

print(
    f"Number of forward input parameters "
    f"(excluding self): {len(non_self_parameters)}"
)

for p in non_self_parameters:
    print(
        f"  {p.name}: "
        f"kind={p.kind}, "
        f"default={p.default}"
    )


print("\n" + "=" * 70)
print("STEP 15C COMPLETE")
print("=" * 70)

print(
    "\nSTOP HERE.\n"
    "No TEST predictions were generated in this step.\n"
    "Send me the COMPLETE output."
)
print("=" * 70)

STEP 15C — FT-TRANSFORMER FORWARD INPUT VERIFICATION

[1] MODEL VERIFICATION
----------------------------------------------------------------------
Model class : FTTransformer

[2] FORWARD SIGNATURE
----------------------------------------------------------------------
ft_transformer.forward(x)

[3] TEST LOADER VERIFICATION
----------------------------------------------------------------------
Batch type : list
Number of batch elements : 2
Element 0: type=Tensor, shape=(2048, 4), dtype=torch.int64
Element 1: type=Tensor, shape=(2048,), dtype=torch.float32

[4] TEST DATASET STRUCTURE
----------------------------------------------------------------------
Dataset class : TensorDataset
Dataset sample elements : 2
Element 0: shape=(4,), dtype=torch.int64
Element 1: shape=(), dtype=torch.float32

[5] INTERPRETATION
----------------------------------------------------------------------
Number of forward input parameters (excluding self): 1
  x: kind=POSITIONAL_OR_KEYWORD, default=<class 'insp

In [38]:
# ======================================================================
# STEP 15D — GENERATE FT-TRANSFORMER TEST PROBABILITIES
# ======================================================================

import numpy as np
import torch

print("=" * 70)
print("STEP 15D — FT-TRANSFORMER TEST PROBABILITY GENERATION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] REQUIRED OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "ft_transformer",
    "test_loader",
    "y_test_final"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"STEP 15D STOPPED — Missing objects: {missing_objects}"
)

print("✓ ft_transformer available")
print("✓ test_loader available")
print("✓ y_test_final available")


# ----------------------------------------------------------------------
# [2] VERIFY TEST LOADER
# ----------------------------------------------------------------------

print("\n[2] TEST LOADER VERIFICATION")
print("-" * 70)

assert len(test_loader.dataset) == len(y_test_final), (
    "TEST loader and y_test_final have different row counts."
)

print(f"TEST rows expected : {len(y_test_final):,}")
print(f"TEST loader rows   : {len(test_loader.dataset):,}")

print("✓ TEST loader aligned")


# ----------------------------------------------------------------------
# [3] VERIFY BATCH STRUCTURE
# ----------------------------------------------------------------------

print("\n[3] BATCH STRUCTURE VERIFICATION")
print("-" * 70)

first_batch = next(iter(test_loader))

assert isinstance(first_batch, (tuple, list)), (
    "Unexpected TEST batch type."
)

assert len(first_batch) == 2, (
    f"Expected 2 batch elements, found {len(first_batch)}."
)

batch_x, batch_y = first_batch

assert isinstance(batch_x, torch.Tensor), (
    "TEST model input is not a Tensor."
)

assert batch_x.ndim == 2, (
    f"Expected 2D categorical input, got {batch_x.ndim}D."
)

assert batch_x.shape[1] == 4, (
    f"Expected 4 categorical features, got {batch_x.shape[1]}."
)

assert batch_x.dtype == torch.int64, (
    f"Expected torch.int64 input, got {batch_x.dtype}."
)

print(f"Model input shape : {tuple(batch_x.shape)}")
print(f"Model input dtype : {batch_x.dtype}")
print("✓ Correct FT-Transformer input structure")


# ----------------------------------------------------------------------
# [4] EVALUATION MODE
# ----------------------------------------------------------------------

print("\n[4] MODEL EVALUATION MODE")
print("-" * 70)

ft_transformer.eval()

print("✓ FT-Transformer set to evaluation mode")


# ----------------------------------------------------------------------
# [5] GENERATE TEST PROBABILITIES
# ----------------------------------------------------------------------

print("\n[5] GENERATING TEST PROBABILITIES")
print("-" * 70)

ft_test_probability_parts = []

with torch.no_grad():

    for batch in test_loader:

        # test_loader structure:
        # batch[0] = categorical model input, shape (batch_size, 4)
        # batch[1] = target

        batch_x = batch[0]

        logits = ft_transformer(batch_x)

        # Safety handling for possible model output containers
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        logits = logits.reshape(-1)

        probabilities = torch.sigmoid(logits)

        ft_test_probability_parts.append(
            probabilities.cpu().numpy()
        )


test_probability_ft = np.concatenate(
    ft_test_probability_parts
).astype(np.float64)


# ----------------------------------------------------------------------
# [6] OUTPUT VALIDATION
# ----------------------------------------------------------------------

print("\n[6] OUTPUT VALIDATION")
print("-" * 70)

assert len(test_probability_ft) == len(y_test_final), (
    "FT-Transformer TEST probability length mismatch."
)

assert np.all(np.isfinite(test_probability_ft)), (
    "FT-Transformer TEST probabilities contain non-finite values."
)

assert np.all(
    (test_probability_ft >= 0.0) &
    (test_probability_ft <= 1.0)
), (
    "FT-Transformer TEST probabilities outside [0, 1]."
)

print(
    f"FT-Transformer TEST probability rows : "
    f"{len(test_probability_ft):,}"
)

print(
    f"Probability range : "
    f"{test_probability_ft.min():.6f} → "
    f"{test_probability_ft.max():.6f}"
)

print("✓ Correct number of probabilities")
print("✓ All probabilities finite")
print("✓ All probabilities within [0, 1]")


# ----------------------------------------------------------------------
# [7] TEST INTEGRITY
# ----------------------------------------------------------------------

print("\n[7] TEST INTEGRITY")
print("-" * 70)

assert len(test_probability_ft) == 311057

print("✓ TEST rows preserved: 311,057")
print("✓ No training performed")
print("✓ No validation tuning performed")
print("✓ No threshold optimization performed")
print("✓ No ensemble-weight modification performed")


# ----------------------------------------------------------------------
# [8] COMPLETION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 15D COMPLETE")
print("=" * 70)

print("\nOBJECT CREATED:")
print("  ✓ test_probability_ft")

print("\nTEST STATUS:")
print("  ✓ FT-Transformer TEST probabilities generated")
print("  ✓ No TEST metrics calculated yet")
print("  ✓ Frozen threshold unchanged")
print("  ✓ Frozen ensemble weights unchanged")

print("\nSTOP HERE.")
print("Send the COMPLETE output before proceeding.")

print("=" * 70)

STEP 15D — FT-TRANSFORMER TEST PROBABILITY GENERATION

[1] REQUIRED OBJECT VERIFICATION
----------------------------------------------------------------------
✓ ft_transformer available
✓ test_loader available
✓ y_test_final available

[2] TEST LOADER VERIFICATION
----------------------------------------------------------------------
TEST rows expected : 311,057
TEST loader rows   : 311,057
✓ TEST loader aligned

[3] BATCH STRUCTURE VERIFICATION
----------------------------------------------------------------------
Model input shape : (2048, 4)
Model input dtype : torch.int64
✓ Correct FT-Transformer input structure

[4] MODEL EVALUATION MODE
----------------------------------------------------------------------
✓ FT-Transformer set to evaluation mode

[5] GENERATING TEST PROBABILITIES
----------------------------------------------------------------------

[6] OUTPUT VALIDATION
----------------------------------------------------------------------
FT-Transformer TEST probability rows :

In [39]:
# ======================================================================
# STEP 15E — FINAL FROZEN ENSEMBLE TEST EVALUATION
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 15E — FINAL FROZEN ENSEMBLE TEST EVALUATION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] REQUIRED OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "test_probability_lgbm",
    "test_probability_ft",
    "y_test_final",
    "best_lgbm_weight",
    "best_ft_weight",
    "FINAL_ENSEMBLE_THRESHOLD"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"STEP 15E STOPPED — Missing objects: {missing_objects}"
)

print("✓ test_probability_lgbm available")
print("✓ test_probability_ft available")
print("✓ y_test_final available")
print("✓ best_lgbm_weight available")
print("✓ best_ft_weight available")
print("✓ FINAL_ENSEMBLE_THRESHOLD available")


# ----------------------------------------------------------------------
# [2] FREEZING VERIFICATION
# ----------------------------------------------------------------------

print("\n[2] FROZEN DECISION RULE VERIFICATION")
print("-" * 70)

FINAL_LGBM_WEIGHT = float(best_lgbm_weight)
FINAL_FT_WEIGHT = float(best_ft_weight)
FINAL_THRESHOLD = float(FINAL_ENSEMBLE_THRESHOLD)

print(f"LightGBM weight          : {FINAL_LGBM_WEIGHT:.2f}")
print(f"FT-Transformer weight    : {FINAL_FT_WEIGHT:.2f}")
print(f"Classification threshold : {FINAL_THRESHOLD:.2f}")

assert np.isclose(FINAL_LGBM_WEIGHT, 0.80)
assert np.isclose(FINAL_FT_WEIGHT, 0.20)
assert np.isclose(FINAL_THRESHOLD, 0.57)

assert np.isclose(
    FINAL_LGBM_WEIGHT + FINAL_FT_WEIGHT,
    1.0
)

print("✓ Frozen weights confirmed")
print("✓ Frozen threshold confirmed")
print("✓ Weights sum to 1.00")


# ----------------------------------------------------------------------
# [3] TEST INPUT VERIFICATION
# ----------------------------------------------------------------------

print("\n[3] TEST INPUT VERIFICATION")
print("-" * 70)

y_test_array = np.asarray(
    y_test_final
).astype(int).reshape(-1)

test_probability_lgbm = np.asarray(
    test_probability_lgbm,
    dtype=np.float64
).reshape(-1)

test_probability_ft = np.asarray(
    test_probability_ft,
    dtype=np.float64
).reshape(-1)

assert len(y_test_array) == 311057
assert len(test_probability_lgbm) == 311057
assert len(test_probability_ft) == 311057

assert len(y_test_array) == len(test_probability_lgbm)
assert len(y_test_array) == len(test_probability_ft)

assert np.all(np.isfinite(test_probability_lgbm))
assert np.all(np.isfinite(test_probability_ft))

assert np.all(
    (test_probability_lgbm >= 0) &
    (test_probability_lgbm <= 1)
)

assert np.all(
    (test_probability_ft >= 0) &
    (test_probability_ft <= 1)
)

assert set(np.unique(y_test_array)).issubset({0, 1})

print(f"TEST rows              : {len(y_test_array):,}")
print(f"LightGBM probabilities  : {len(test_probability_lgbm):,}")
print(f"FT probabilities        : {len(test_probability_ft):,}")

print("✓ All TEST arrays aligned")
print("✓ All probabilities valid")
print("✓ Binary TEST target verified")


# ----------------------------------------------------------------------
# [4] APPLY FROZEN ENSEMBLE
# ----------------------------------------------------------------------

print("\n[4] APPLYING FROZEN ENSEMBLE")
print("-" * 70)

ensemble_test_probability = (
    FINAL_LGBM_WEIGHT * test_probability_lgbm
    +
    FINAL_FT_WEIGHT * test_probability_ft
)

assert len(ensemble_test_probability) == len(y_test_array)

assert np.all(np.isfinite(ensemble_test_probability))

assert np.all(
    (ensemble_test_probability >= 0) &
    (ensemble_test_probability <= 1)
)

print(
    "Formula:"
)

print(
    f"  P = {FINAL_LGBM_WEIGHT:.2f} × P_LGBM "
    f"+ {FINAL_FT_WEIGHT:.2f} × P_FT"
)

print("✓ Frozen ensemble applied")


# ----------------------------------------------------------------------
# [5] APPLY FROZEN THRESHOLD
# ----------------------------------------------------------------------

print("\n[5] APPLYING FROZEN CLASSIFICATION THRESHOLD")
print("-" * 70)

test_prediction_final = (
    ensemble_test_probability >= FINAL_THRESHOLD
).astype(int)

assert len(test_prediction_final) == len(y_test_array)

print(f"Threshold : {FINAL_THRESHOLD:.2f}")
print("✓ Frozen threshold applied")
print("✓ No threshold search performed")


# ----------------------------------------------------------------------
# [6] FINAL TEST METRICS
# ----------------------------------------------------------------------

print("\n[6] FINAL TEST METRICS")
print("-" * 70)

test_accuracy = accuracy_score(
    y_test_array,
    test_prediction_final
)

test_balanced_accuracy = balanced_accuracy_score(
    y_test_array,
    test_prediction_final
)

test_precision = precision_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_recall = recall_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_f1 = f1_score(
    y_test_array,
    test_prediction_final,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test_array,
    ensemble_test_probability
)

test_pr_auc = average_precision_score(
    y_test_array,
    ensemble_test_probability
)

test_cm = confusion_matrix(
    y_test_array,
    test_prediction_final
)

tn, fp, fn, tp = test_cm.ravel()

test_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

print(f"Accuracy           : {test_accuracy:.4f}")
print(f"Balanced Accuracy  : {test_balanced_accuracy:.4f}")
print(f"Precision          : {test_precision:.4f}")
print(f"Recall/Sensitivity : {test_recall:.4f}")
print(f"Specificity        : {test_specificity:.4f}")
print(f"F1                 : {test_f1:.4f}")
print(f"ROC-AUC            : {test_roc_auc:.4f}")
print(f"PR-AUC             : {test_pr_auc:.4f}")


# ----------------------------------------------------------------------
# [7] CONFUSION MATRIX
# ----------------------------------------------------------------------

print("\n[7] CONFUSION MATRIX")
print("-" * 70)

print(
    "                 Predicted"
)

print(
    "                 0        1"
)

print(
    f"Actual 0     {tn:7d}  {fp:7d}"
)

print(
    f"Actual 1     {fn:7d}  {tp:7d}"
)

print("\nRaw matrix:")
print(test_cm)


# ----------------------------------------------------------------------
# [8] TEST CLASS DISTRIBUTION
# ----------------------------------------------------------------------

print("\n[8] TEST CLASS / PREDICTION DISTRIBUTION")
print("-" * 70)

actual_negative = int((y_test_array == 0).sum())
actual_positive = int((y_test_array == 1).sum())

predicted_negative = int((test_prediction_final == 0).sum())
predicted_positive = int((test_prediction_final == 1).sum())

print(f"Actual susceptible (0) : {actual_negative:,}")
print(f"Actual resistant (1)   : {actual_positive:,}")

print(f"Predicted susceptible (0) : {predicted_negative:,}")
print(f"Predicted resistant (1)   : {predicted_positive:,}")


# ----------------------------------------------------------------------
# [9] FINAL TEST OBJECTS
# ----------------------------------------------------------------------

print("\n[9] SAVING FINAL TEST RESULTS")
print("-" * 70)

final_test_metrics = {
    "lgbm_weight": FINAL_LGBM_WEIGHT,
    "ft_weight": FINAL_FT_WEIGHT,
    "threshold": FINAL_THRESHOLD,
    "accuracy": test_accuracy,
    "balanced_accuracy": test_balanced_accuracy,
    "precision": test_precision,
    "recall_sensitivity": test_recall,
    "specificity": test_specificity,
    "f1": test_f1,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc,
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

print("✓ ensemble_test_probability created")
print("✓ test_prediction_final created")
print("✓ test_cm created")
print("✓ final_test_metrics created")


# ----------------------------------------------------------------------
# [10] FINAL INTEGRITY ASSERTIONS
# ----------------------------------------------------------------------

print("\n[10] FINAL TEST INTEGRITY")
print("-" * 70)

assert np.isclose(FINAL_LGBM_WEIGHT, 0.80)
assert np.isclose(FINAL_FT_WEIGHT, 0.20)
assert np.isclose(FINAL_THRESHOLD, 0.57)

assert len(ensemble_test_probability) == 311057
assert len(test_prediction_final) == 311057

print("✓ Frozen LightGBM weight = 0.80")
print("✓ Frozen FT-Transformer weight = 0.20")
print("✓ Frozen threshold = 0.57")
print("✓ TEST rows = 311,057")
print("✓ No TEST-based optimization")
print("✓ No model modification")
print("✓ Final TEST evaluation completed")


# ----------------------------------------------------------------------
# [11] FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 15E — FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

print("\nFROZEN DECISION RULE")
print("-" * 70)
print(f"LightGBM       : {FINAL_LGBM_WEIGHT:.2f}")
print(f"FT-Transformer : {FINAL_FT_WEIGHT:.2f}")
print(f"Threshold      : {FINAL_THRESHOLD:.2f}")

print("\nFINAL TEST PERFORMANCE")
print("-" * 70)
print(f"PR-AUC             : {test_pr_auc:.4f}")
print(f"ROC-AUC            : {test_roc_auc:.4f}")
print(f"F1                 : {test_f1:.4f}")
print(f"Sensitivity        : {test_recall:.4f}")
print(f"Specificity        : {test_specificity:.4f}")
print(f"Balanced Accuracy  : {test_balanced_accuracy:.4f}")
print(f"Accuracy           : {test_accuracy:.4f}")

print("\nTEST STATUS")
print("-" * 70)
print("✓ Test evaluated once using frozen decision rule")
print("✓ No threshold optimization on TEST")
print("✓ No ensemble-weight optimization on TEST")
print("✓ No model retraining on TEST")

print("=" * 70)

STEP 15E — FINAL FROZEN ENSEMBLE TEST EVALUATION

[1] REQUIRED OBJECT VERIFICATION
----------------------------------------------------------------------
✓ test_probability_lgbm available
✓ test_probability_ft available
✓ y_test_final available
✓ best_lgbm_weight available
✓ best_ft_weight available
✓ FINAL_ENSEMBLE_THRESHOLD available

[2] FROZEN DECISION RULE VERIFICATION
----------------------------------------------------------------------
LightGBM weight          : 0.80
FT-Transformer weight    : 0.20
Classification threshold : 0.57
✓ Frozen weights confirmed
✓ Frozen threshold confirmed
✓ Weights sum to 1.00

[3] TEST INPUT VERIFICATION
----------------------------------------------------------------------
TEST rows              : 311,057
LightGBM probabilities  : 311,057
FT probabilities        : 311,057
✓ All TEST arrays aligned
✓ All probabilities valid
✓ Binary TEST target verified

[4] APPLYING FROZEN ENSEMBLE
-----------------------------------------------------------------

In [41]:
# ======================================================================
# STEP 16A — VERIFY ACTUAL TRAIN / VALIDATION COLUMNS
# ======================================================================

print("=" * 70)
print("STEP 16A — ACTUAL TRAIN / VALIDATION COLUMN VERIFICATION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "X_train_raw",
    "X_validation_raw",
    "X_train_final",
    "X_validation_final"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

for name in required_objects:
    obj = globals()[name]

    print(
        f"✓ {name:<22} "
        f"type={type(obj).__name__:<12} "
        f"shape={getattr(obj, 'shape', None)}"
    )


# ----------------------------------------------------------------------
# [2] RAW TRAIN COLUMNS
# ----------------------------------------------------------------------

print("\n[2] X_train_raw COLUMNS")
print("-" * 70)

train_raw_columns = list(X_train_raw.columns)

print(f"Number of columns: {len(train_raw_columns)}")

for i, column in enumerate(train_raw_columns, start=1):
    print(f"{i:2d}. {column}")


# ----------------------------------------------------------------------
# [3] RAW VALIDATION COLUMNS
# ----------------------------------------------------------------------

print("\n[3] X_validation_raw COLUMNS")
print("-" * 70)

validation_raw_columns = list(
    X_validation_raw.columns
)

print(f"Number of columns: {len(validation_raw_columns)}")

for i, column in enumerate(validation_raw_columns, start=1):
    print(f"{i:2d}. {column}")


# ----------------------------------------------------------------------
# [4] FINAL TRAIN ARRAY
# ----------------------------------------------------------------------

print("\n[4] X_train_final")
print("-" * 70)

print(f"Type  : {type(X_train_final).__name__}")
print(f"Shape : {X_train_final.shape}")

if hasattr(X_train_final, "columns"):
    print("Columns:")
    for i, column in enumerate(
        X_train_final.columns,
        start=1
    ):
        print(f"{i:2d}. {column}")
else:
    print(
        "X_train_final is an ndarray, so it has no column names."
    )


# ----------------------------------------------------------------------
# [5] FINAL VALIDATION ARRAY
# ----------------------------------------------------------------------

print("\n[5] X_validation_final")
print("-" * 70)

print(f"Type  : {type(X_validation_final).__name__}")
print(f"Shape : {X_validation_final.shape}")

if hasattr(X_validation_final, "columns"):
    print("Columns:")
    for i, column in enumerate(
        X_validation_final.columns,
        start=1
    ):
        print(f"{i:2d}. {column}")
else:
    print(
        "X_validation_final is an ndarray, so it has no column names."
    )


# ----------------------------------------------------------------------
# [6] COMMON / DIFFERENT COLUMNS
# ----------------------------------------------------------------------

print("\n[6] COLUMN CONSISTENCY")
print("-" * 70)

train_set = set(train_raw_columns)
validation_set = set(validation_raw_columns)

train_only = train_set - validation_set
validation_only = validation_set - train_set
common = train_set & validation_set

print(f"Common columns       : {len(common)}")
print(f"TRAIN-only columns   : {len(train_only)}")
print(f"VALIDATION-only cols : {len(validation_only)}")

if train_only:
    print("\nTRAIN-only:")
    for column in sorted(train_only):
        print(f"  - {column}")

if validation_only:
    print("\nVALIDATION-only:")
    for column in sorted(validation_only):
        print(f"  - {column}")


# ----------------------------------------------------------------------
# [7] PREVIOUS MODEL FEATURE LISTS
# ----------------------------------------------------------------------

print("\n[7] EXISTING FEATURE METADATA")
print("-" * 70)

for name in [
    "categorical_features",
    "categorical_feature_indices",
    "CURRENT_FEATURES"
]:

    if name in globals():
        print(f"\n{name}:")
        print(globals()[name])


# ----------------------------------------------------------------------
# [8] COMPLETION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 16A COMPLETE")
print("=" * 70)

print(
    "\nNo data was modified."
)

print(
    "No model was modified."
)

print(
    "No TEST data was used."
)

print(
    "\nSTOP HERE — SEND THE COMPLETE OUTPUT."
)

print("=" * 70)

STEP 16A — ACTUAL TRAIN / VALIDATION COLUMN VERIFICATION

[1] OBJECT VERIFICATION
----------------------------------------------------------------------
✓ X_train_raw            type=DataFrame    shape=(934500, 11)
✓ X_validation_raw       type=DataFrame    shape=(308772, 11)
✓ X_train_final          type=ndarray      shape=(1317556, 4)
✓ X_validation_final     type=ndarray      shape=(308772, 4)

[2] X_train_raw COLUMNS
----------------------------------------------------------------------
Number of columns: 11
 1. anon_id
 2. pat_enc_csn_id_coded
 3. order_proc_id_coded
 4. order_time_jittered_utc
 5. ordering_mode
 6. culture_description
 7. was_positive
 8. organism
 9. antibiotic
10. susceptibility
11. amr_target

[3] X_validation_raw COLUMNS
----------------------------------------------------------------------
Number of columns: 11
 1. anon_id
 2. pat_enc_csn_id_coded
 3. order_proc_id_coded
 4. order_time_jittered_utc
 5. ordering_mode
 6. culture_description
 7. was_positive
 

In [42]:
# ======================================================================
# STEP 16B — TRAIN/VALIDATION FEATURE ABLATION SETUP
# ======================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 16B — TRAIN/VALIDATION FEATURE ABLATION SETUP")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] REQUIRED OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "X_train_raw",
    "X_validation_raw",
    "y_train_raw",
    "y_validation_raw",
    "categorical_features"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"STEP 16B STOPPED — Missing objects: {missing_objects}"
)

print("✓ X_train_raw available")
print("✓ X_validation_raw available")
print("✓ y_train_raw available")
print("✓ y_validation_raw available")
print("✓ categorical_features available")


# ----------------------------------------------------------------------
# [2] DEFINE MODEL FEATURES
# ----------------------------------------------------------------------

print("\n[2] MODEL FEATURE DEFINITION")
print("-" * 70)

MODEL_FEATURES = [
    "ordering_mode",
    "culture_description",
    "organism",
    "antibiotic"
]

assert list(categorical_features) == MODEL_FEATURES, (
    "Existing categorical_features does not match the "
    "established four-feature model."
)

for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i}. {feature}")

print("✓ Four model predictors confirmed")


# ----------------------------------------------------------------------
# [3] VERIFY RAW DATA CONTAINS REQUIRED FEATURES
# ----------------------------------------------------------------------

print("\n[3] RAW DATA FEATURE VERIFICATION")
print("-" * 70)

for feature in MODEL_FEATURES:

    assert feature in X_train_raw.columns, (
        f"{feature} missing from X_train_raw."
    )

    assert feature in X_validation_raw.columns, (
        f"{feature} missing from X_validation_raw."
    )

    print(f"✓ {feature}")


# ----------------------------------------------------------------------
# [4] DEFINE ABLATION CANDIDATES
# ----------------------------------------------------------------------

print("\n[4] ABLATION CANDIDATES")
print("-" * 70)

feature_ablation_sets = {
    "all_4_features": MODEL_FEATURES.copy(),

    "without_ordering_mode": [
        "culture_description",
        "organism",
        "antibiotic"
    ],

    "without_culture_description": [
        "ordering_mode",
        "organism",
        "antibiotic"
    ],

    "without_organism": [
        "ordering_mode",
        "culture_description",
        "antibiotic"
    ],

    "without_antibiotic": [
        "ordering_mode",
        "culture_description",
        "organism"
    ]
}

for name, features in feature_ablation_sets.items():

    print(
        f"\n{name}:"
    )

    for feature in features:
        print(f"  ✓ {feature}")


# ----------------------------------------------------------------------
# [5] VERIFY EACH ABLATION SET
# ----------------------------------------------------------------------

print("\n[5] ABLATION SET VALIDATION")
print("-" * 70)

assert len(feature_ablation_sets) == 5

for name, features in feature_ablation_sets.items():

    assert len(features) >= 3, (
        f"{name} has fewer than 3 predictors."
    )

    assert len(features) == len(set(features)), (
        f"{name} contains duplicate features."
    )

    assert set(features).issubset(
        set(MODEL_FEATURES)
    ), (
        f"{name} contains a feature outside the "
        "established model feature set."
    )

    print(
        f"✓ {name}: {len(features)} features"
    )


# ----------------------------------------------------------------------
# [6] VERIFY TARGETS
# ----------------------------------------------------------------------

print("\n[6] TARGET VERIFICATION")
print("-" * 70)

y_train_array = np.asarray(
    y_train_raw
).astype(int).reshape(-1)

y_validation_array = np.asarray(
    y_validation_raw
).astype(int).reshape(-1)

assert len(y_train_array) == len(X_train_raw)
assert len(y_validation_array) == len(X_validation_raw)

assert set(np.unique(y_train_array)).issubset({0, 1})
assert set(np.unique(y_validation_array)).issubset({0, 1})

print(f"TRAIN rows      : {len(y_train_array):,}")
print(f"VALIDATION rows : {len(y_validation_array):,}")

print(
    f"TRAIN positive rate      : "
    f"{y_train_array.mean():.4f}"
)

print(
    f"VALIDATION positive rate : "
    f"{y_validation_array.mean():.4f}"
)

print("✓ Binary targets verified")
print("✓ Target lengths aligned")


# ----------------------------------------------------------------------
# [7] VERIFY NO TARGET / IDENTIFIER FEATURES ARE CANDIDATES
# ----------------------------------------------------------------------

print("\n[7] EXCLUDED FEATURE PROTECTION")
print("-" * 70)

for forbidden in [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded",
    "order_time_jittered_utc",
    "was_positive",
    "susceptibility",
    "amr_target"
]:

    for name, features in feature_ablation_sets.items():

        assert forbidden not in features, (
            f"Forbidden column {forbidden} found in {name}."
        )

    print(f"✓ {forbidden} excluded")


# ----------------------------------------------------------------------
# [8] TEST DATA PROTECTION
# ----------------------------------------------------------------------

print("\n[8] TEST DATA PROTECTION")
print("-" * 70)

print(
    "No TEST object is accessed by this ablation setup."
)

print(
    "✓ TEST remains untouched"
)


# ----------------------------------------------------------------------
# [9] CREATE FEATURE ABLATION SUMMARY
# ----------------------------------------------------------------------

print("\n[9] ABLATION SUMMARY")
print("-" * 70)

feature_ablation_summary = pd.DataFrame([
    {
        "experiment": name,
        "n_features": len(features),
        "features": ", ".join(features)
    }
    for name, features in feature_ablation_sets.items()
])

print(
    feature_ablation_summary.to_string(index=False)
)


# ----------------------------------------------------------------------
# [10] COMPLETION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 16B COMPLETE")
print("=" * 70)

print("\nOBJECTS CREATED:")
print("  ✓ MODEL_FEATURES")
print("  ✓ feature_ablation_sets")
print("  ✓ feature_ablation_summary")

print("\nABLATION EXPERIMENTS:")
print("  ✓ 4-feature baseline")
print("  ✓ Without ordering_mode")
print("  ✓ Without culture_description")
print("  ✓ Without organism")
print("  ✓ Without antibiotic")

print("\nDATA USAGE:")
print("  ✓ TRAIN used")
print("  ✓ VALIDATION used")
print("  ✗ TEST used")

print("\nMODEL STATUS:")
print("  ✓ Existing models unchanged")
print("  ✓ Existing frozen ensemble unchanged")
print("  ✓ Existing frozen threshold unchanged")

print("\nSTOP HERE.")
print("Send the COMPLETE output before proceeding.")

print("=" * 70)

STEP 16B — TRAIN/VALIDATION FEATURE ABLATION SETUP

[1] REQUIRED OBJECT VERIFICATION
----------------------------------------------------------------------
✓ X_train_raw available
✓ X_validation_raw available
✓ y_train_raw available
✓ y_validation_raw available
✓ categorical_features available

[2] MODEL FEATURE DEFINITION
----------------------------------------------------------------------
1. ordering_mode
2. culture_description
3. organism
4. antibiotic
✓ Four model predictors confirmed

[3] RAW DATA FEATURE VERIFICATION
----------------------------------------------------------------------
✓ ordering_mode
✓ culture_description
✓ organism
✓ antibiotic

[4] ABLATION CANDIDATES
----------------------------------------------------------------------

all_4_features:
  ✓ ordering_mode
  ✓ culture_description
  ✓ organism
  ✓ antibiotic

without_ordering_mode:
  ✓ culture_description
  ✓ organism
  ✓ antibiotic

without_culture_description:
  ✓ ordering_mode
  ✓ organism
  ✓ antibiotic



In [43]:
# ======================================================================
# STEP 16C — BUILD TRAIN/VALIDATION DATA FOR FEATURE ABLATION
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTEN

print("=" * 70)
print("STEP 16C — ABLATION TRAIN/VALIDATION DATA CONSTRUCTION")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] REQUIRED OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "X_train_raw",
    "X_validation_raw",
    "y_train_raw",
    "y_validation_raw",
    "feature_ablation_sets"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"STEP 16C STOPPED — Missing objects: {missing_objects}"
)

print("✓ Required objects available")


# ----------------------------------------------------------------------
# [2] TARGET PREPARATION
# ----------------------------------------------------------------------

print("\n[2] TARGET PREPARATION")
print("-" * 70)

y_train_ablation = (
    np.asarray(y_train_raw)
    .astype(int)
    .reshape(-1)
)

y_validation_ablation = (
    np.asarray(y_validation_raw)
    .astype(int)
    .reshape(-1)
)

assert len(y_train_ablation) == len(X_train_raw)
assert len(y_validation_ablation) == len(X_validation_raw)

assert set(np.unique(y_train_ablation)).issubset({0, 1})
assert set(np.unique(y_validation_ablation)).issubset({0, 1})

print(f"TRAIN rows      : {len(y_train_ablation):,}")
print(f"VALIDATION rows : {len(y_validation_ablation):,}")
print("✓ Targets verified")


# ----------------------------------------------------------------------
# [3] CONTAINERS
# ----------------------------------------------------------------------

print("\n[3] INITIALIZING ABLATION DATA CONTAINERS")
print("-" * 70)

ablation_encoders = {}
ablation_train_encoded = {}
ablation_validation_encoded = {}
ablation_train_resampled = {}
ablation_target_resampled = {}

print("✓ Containers initialized")


# ----------------------------------------------------------------------
# [4] PROCESS EACH FEATURE SET
# ----------------------------------------------------------------------

print("\n[4] BUILDING EACH ABLATION DATASET")
print("-" * 70)

for experiment_name, features in feature_ablation_sets.items():

    print("\n" + "-" * 70)
    print(f"EXPERIMENT: {experiment_name}")
    print("-" * 70)

    print(f"Features ({len(features)}): {features}")


    # ------------------------------------------------------------------
    # 4A. RAW FEATURE EXTRACTION
    # ------------------------------------------------------------------

    X_tr = X_train_raw[features].copy()
    X_va = X_validation_raw[features].copy()

    assert X_tr.shape[1] == len(features)
    assert X_va.shape[1] == len(features)

    assert list(X_tr.columns) == features
    assert list(X_va.columns) == features


    # ------------------------------------------------------------------
    # 4B. MISSING-VALUE HANDLING
    # ------------------------------------------------------------------
    # The original model data contained categorical variables.
    # Convert missing categorical values to a fixed explicit category.
    # This is performed separately inside each candidate pipeline.

    X_tr = X_tr.fillna("__MISSING__").astype(str)
    X_va = X_va.fillna("__MISSING__").astype(str)


    # ------------------------------------------------------------------
    # 4C. TRAIN-ONLY ORDINAL ENCODER
    # ------------------------------------------------------------------

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1
    )

    X_tr_encoded = encoder.fit_transform(X_tr)
    X_va_encoded = encoder.transform(X_va)

    # Convert:
    # known categories  → 1..N
    # unseen categories → 0
    X_tr_encoded = (
        X_tr_encoded.astype(np.int64) + 1
    )

    X_va_encoded = np.where(
        X_va_encoded < 0,
        0,
        X_va_encoded + 1
    ).astype(np.int64)


    # ------------------------------------------------------------------
    # 4D. ENCODING VALIDATION
    # ------------------------------------------------------------------

    assert X_tr_encoded.shape == (
        len(X_train_raw),
        len(features)
    )

    assert X_va_encoded.shape == (
        len(X_validation_raw),
        len(features)
    )

    assert np.all(X_tr_encoded >= 1)
    assert np.all(X_va_encoded >= 0)

    print(
        f"Encoded TRAIN shape      : "
        f"{X_tr_encoded.shape}"
    )

    print(
        f"Encoded VALIDATION shape : "
        f"{X_va_encoded.shape}"
    )


    # ------------------------------------------------------------------
    # 4E. STORE ENCODER + ENCODED DATA
    # ------------------------------------------------------------------

    ablation_encoders[experiment_name] = encoder
    ablation_train_encoded[experiment_name] = X_tr_encoded
    ablation_validation_encoded[experiment_name] = X_va_encoded


    # ------------------------------------------------------------------
    # 4F. TRAIN-ONLY SMOTEN
    # ------------------------------------------------------------------

    train_class_counts = pd.Series(
        y_train_ablation
    ).value_counts().sort_index()

    majority_count = int(train_class_counts.max())
    minority_count = int(train_class_counts.min())

    print(
        f"Original TRAIN classes: "
        f"0={int(train_class_counts.get(0, 0)):,}, "
        f"1={int(train_class_counts.get(1, 0)):,}"
    )

    # Memory-controlled SMOTEN strategy matching the original approach.
    minority_fit_count = min(5000, minority_count)

    rng = np.random.default_rng(42)

    minority_indices = np.flatnonzero(
        y_train_ablation == 1
    )

    selected_minority_indices = rng.choice(
        minority_indices,
        size=minority_fit_count,
        replace=False
    )

    majority_indices = np.flatnonzero(
        y_train_ablation == 0
    )

    smoten_fit_indices = np.concatenate([
        majority_indices,
        selected_minority_indices
    ])

    X_smoten_fit = X_tr_encoded[
        smoten_fit_indices
    ]

    y_smoten_fit = y_train_ablation[
        smoten_fit_indices
    ]

    smoten = SMOTEN(
        sampling_strategy=0.5,
        random_state=42,
        k_neighbors=5
    )

    X_smoten_resampled, y_smoten_resampled = (
        smoten.fit_resample(
            X_smoten_fit,
            y_smoten_fit
        )
    )

    synthetic_count = (
        len(X_smoten_resampled)
        - len(X_smoten_fit)
    )

    print(
        f"SMOTEN fitting rows     : "
        f"{len(X_smoten_fit):,}"
    )

    print(
        f"Synthetic minority rows: "
        f"{synthetic_count:,}"
    )


    # ------------------------------------------------------------------
    # 4G. APPEND SYNTHETIC DATA TO COMPLETE ORIGINAL TRAIN
    # ------------------------------------------------------------------

    synthetic_X = X_smoten_resampled[
        len(X_smoten_fit):
    ]

    synthetic_y = y_smoten_resampled[
        len(y_smoten_fit):
    ]

    X_final_train = np.vstack([
        X_tr_encoded,
        synthetic_X
    ])

    y_final_train = np.concatenate([
        y_train_ablation,
        synthetic_y
    ])


    # ------------------------------------------------------------------
    # 4H. RESAMPLED DATA VALIDATION
    # ------------------------------------------------------------------

    assert X_final_train.shape[0] == len(y_final_train)

    assert X_final_train.shape[1] == len(features)

    assert np.all(
        X_final_train.astype(np.int64) >= 1
    )

    assert set(
        np.unique(y_final_train)
    ).issubset({0, 1})

    final_counts = pd.Series(
        y_final_train
    ).value_counts().sort_index()

    print(
        f"Final TRAIN shape       : "
        f"{X_final_train.shape}"
    )

    print(
        f"Final TRAIN classes     : "
        f"0={int(final_counts.get(0, 0)):,}, "
        f"1={int(final_counts.get(1, 0)):,}"
    )

    print(
        f"Final minority/majority : "
        f"{final_counts.min() / final_counts.max():.4f}"
    )


    # ------------------------------------------------------------------
    # 4I. STORE FINAL TRAIN DATA
    # ------------------------------------------------------------------

    ablation_train_resampled[
        experiment_name
    ] = X_final_train

    ablation_target_resampled[
        experiment_name
    ] = y_final_train


# ----------------------------------------------------------------------
# [5] VALIDATION DATA REMAINS UNRESAMPLED
# ----------------------------------------------------------------------

print("\n" + "-" * 70)
print("[5] VALIDATION INTEGRITY")
print("-" * 70)

for experiment_name, features in feature_ablation_sets.items():

    X_va = ablation_validation_encoded[
        experiment_name
    ]

    assert X_va.shape == (
        len(X_validation_raw),
        len(features)
    )

    print(
        f"✓ {experiment_name}: "
        f"validation rows={len(X_va):,}, "
        f"features={X_va.shape[1]}"
    )


# ----------------------------------------------------------------------
# [6] TEST PROTECTION
# ----------------------------------------------------------------------

print("\n[6] TEST PROTECTION")
print("-" * 70)

print("✓ TEST data was not used")
print("✓ TEST probabilities were not accessed")
print("✓ TEST metrics were not accessed")
print("✓ TEST remains untouched")


# ----------------------------------------------------------------------
# [7] COMPLETION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 16C COMPLETE")
print("=" * 70)

print("\nOBJECTS CREATED:")
print("  ✓ ablation_encoders")
print("  ✓ ablation_train_encoded")
print("  ✓ ablation_validation_encoded")
print("  ✓ ablation_train_resampled")
print("  ✓ ablation_target_resampled")
print("  ✓ y_train_ablation")
print("  ✓ y_validation_ablation")

print("\nDATA INTEGRITY:")
print("  ✓ Encoders fitted TRAIN-only")
print("  ✓ SMOTEN applied TRAIN-only")
print("  ✓ VALIDATION untouched by SMOTEN")
print("  ✓ TEST untouched")

print("\nSTOP HERE.")
print("Send the COMPLETE output.")

print("=" * 70)

STEP 16C — ABLATION TRAIN/VALIDATION DATA CONSTRUCTION

[1] REQUIRED OBJECT VERIFICATION
----------------------------------------------------------------------
✓ Required objects available

[2] TARGET PREPARATION
----------------------------------------------------------------------
TRAIN rows      : 934,500
VALIDATION rows : 308,772
✓ Targets verified

[3] INITIALIZING ABLATION DATA CONTAINERS
----------------------------------------------------------------------
✓ Containers initialized

[4] BUILDING EACH ABLATION DATASET
----------------------------------------------------------------------

----------------------------------------------------------------------
EXPERIMENT: all_4_features
----------------------------------------------------------------------
Features (4): ['ordering_mode', 'culture_description', 'organism', 'antibiotic']
Encoded TRAIN shape      : (934500, 4)
Encoded VALIDATION shape : (308772, 4)
Original TRAIN classes: 0=776,112, 1=158,388
SMOTEN fitting rows     :

In [44]:
# ======================================================================
# STEP 16D — LIGHTGBM FEATURE ABLATION EXPERIMENT
# ======================================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("STEP 16D — LIGHTGBM FEATURE ABLATION EXPERIMENT")
print("=" * 70)


# ----------------------------------------------------------------------
# [1] REQUIRED OBJECT VERIFICATION
# ----------------------------------------------------------------------

print("\n[1] REQUIRED OBJECT VERIFICATION")
print("-" * 70)

required_objects = [
    "feature_ablation_sets",
    "ablation_train_resampled",
    "ablation_target_resampled",
    "ablation_validation_encoded",
    "y_validation_ablation"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"STEP 16D STOPPED — Missing objects: {missing_objects}"
)

print("✓ Ablation feature sets available")
print("✓ Resampled TRAIN datasets available")
print("✓ VALIDATION datasets available")
print("✓ VALIDATION target available")


# ----------------------------------------------------------------------
# [2] VERIFY FIVE EXPERIMENTS
# ----------------------------------------------------------------------

print("\n[2] ABLATION EXPERIMENT VERIFICATION")
print("-" * 70)

expected_experiments = [
    "all_4_features",
    "without_ordering_mode",
    "without_culture_description",
    "without_organism",
    "without_antibiotic"
]

assert set(feature_ablation_sets.keys()) == set(
    expected_experiments
)

print("✓ Five expected experiments present")


# ----------------------------------------------------------------------
# [3] COMMON LIGHTGBM CONFIGURATION
# ----------------------------------------------------------------------

print("\n[3] LIGHTGBM CONFIGURATION")
print("-" * 70)

# Use the previously selected LightGBM hyperparameters.
# Feature ablation is the only experimental variable.

ABLATION_LGBM_PARAMS = {
    "objective": "binary",
    "learning_rate": 0.01,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_split_gain": 0.0,
    "n_estimators": 3000,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

for key, value in ABLATION_LGBM_PARAMS.items():
    print(f"{key:<22}: {value}")


# ----------------------------------------------------------------------
# [4] INITIALIZE RESULT CONTAINERS
# ----------------------------------------------------------------------

print("\n[4] INITIALIZING RESULTS")
print("-" * 70)

ablation_lgbm_models = {}
ablation_lgbm_probabilities = {}
ablation_lgbm_predictions = {}
ablation_lgbm_results = []

print("✓ Result containers initialized")


# ----------------------------------------------------------------------
# [5] TRAIN EACH ABLATION MODEL
# ----------------------------------------------------------------------

print("\n[5] TRAINING ABLATION MODELS")
print("-" * 70)

for experiment_name in expected_experiments:

    print("\n" + "=" * 70)
    print(f"EXPERIMENT: {experiment_name}")
    print("=" * 70)

    features = feature_ablation_sets[
        experiment_name
    ]

    X_tr = ablation_train_resampled[
        experiment_name
    ]

    y_tr = ablation_target_resampled[
        experiment_name
    ]

    X_va = ablation_validation_encoded[
        experiment_name
    ]

    y_va = y_validation_ablation


    # --------------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------------

    assert X_tr.shape[1] == len(features)
    assert X_va.shape[1] == len(features)

    assert len(X_tr) == len(y_tr)
    assert len(X_va) == len(y_va)

    assert np.all(np.isfinite(X_tr))
    assert np.all(np.isfinite(X_va))

    print(f"Features             : {features}")
    print(f"TRAIN shape          : {X_tr.shape}")
    print(f"VALIDATION shape     : {X_va.shape}")


    # --------------------------------------------------------------
    # CATEGORICAL FEATURE INDICES
    # --------------------------------------------------------------

    categorical_indices = list(
        range(len(features))
    )

    print(
        f"Categorical indices  : "
        f"{categorical_indices}"
    )


    # --------------------------------------------------------------
    # MODEL
    # --------------------------------------------------------------

    model = lgb.LGBMClassifier(
        **ABLATION_LGBM_PARAMS
    )


    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    model.fit(
        X_tr,
        y_tr,

        eval_set=[
            (X_va, y_va)
        ],

        eval_names=[
            "validation"
        ],

        categorical_feature=categorical_indices,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False
            )
        ]
    )


    # --------------------------------------------------------------
    # VALIDATION PREDICTION
    # --------------------------------------------------------------

    validation_probability = model.predict_proba(
        X_va,
        num_iteration=model.best_iteration_
    )[:, 1]

    validation_prediction = (
        validation_probability >= 0.50
    ).astype(int)


    # --------------------------------------------------------------
    # METRICS
    # --------------------------------------------------------------

    accuracy = accuracy_score(
        y_va,
        validation_prediction
    )

    balanced_accuracy = balanced_accuracy_score(
        y_va,
        validation_prediction
    )

    precision = precision_score(
        y_va,
        validation_prediction,
        zero_division=0
    )

    recall = recall_score(
        y_va,
        validation_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_va,
        validation_prediction,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_va,
        validation_probability
    )

    pr_auc = average_precision_score(
        y_va,
        validation_probability
    )


    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    ablation_lgbm_models[
        experiment_name
    ] = model

    ablation_lgbm_probabilities[
        experiment_name
    ] = validation_probability

    ablation_lgbm_predictions[
        experiment_name
    ] = validation_prediction

    ablation_lgbm_results.append({
        "experiment": experiment_name,
        "n_features": len(features),
        "features": ", ".join(features),
        "best_iteration": model.best_iteration_,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    })


    print(
        f"Best iteration : "
        f"{model.best_iteration_}"
    )

    print(
        f"Validation PR-AUC : "
        f"{pr_auc:.4f}"
    )

    print(
        f"Validation ROC-AUC : "
        f"{roc_auc:.4f}"
    )

    print(
        f"Validation F1 : "
        f"{f1:.4f}"
    )


# ----------------------------------------------------------------------
# [6] COMPARISON TABLE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[6] ABLATION COMPARISON")
print("=" * 70)

ablation_lgbm_results_df = pd.DataFrame(
    ablation_lgbm_results
).sort_values(
    by="pr_auc",
    ascending=False
).reset_index(drop=True)

print(
    ablation_lgbm_results_df.to_string(
        index=False
    )
)


# ----------------------------------------------------------------------
# [7] FEATURE EFFECT RELATIVE TO BASELINE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[7] FEATURE EFFECT RELATIVE TO 4-FEATURE BASELINE")
print("=" * 70)

baseline_row = ablation_lgbm_results_df[
    ablation_lgbm_results_df["experiment"]
    == "all_4_features"
].iloc[0]

baseline_pr_auc = baseline_row["pr_auc"]
baseline_roc_auc = baseline_row["roc_auc"]

effect_rows = []

for _, row in ablation_lgbm_results_df.iterrows():

    effect_rows.append({
        "experiment": row["experiment"],
        "delta_pr_auc_vs_baseline":
            row["pr_auc"] - baseline_pr_auc,
        "delta_roc_auc_vs_baseline":
            row["roc_auc"] - baseline_roc_auc
    })

feature_ablation_effects = pd.DataFrame(
    effect_rows
).sort_values(
    by="delta_pr_auc_vs_baseline",
    ascending=False
)

print(
    feature_ablation_effects.to_string(
        index=False
    )
)


# ----------------------------------------------------------------------
# [8] TEST PROTECTION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("[8] TEST PROTECTION")
print("=" * 70)

print("✓ No TEST data accessed")
print("✓ No TEST probabilities accessed")
print("✓ No TEST metrics accessed")
print("✓ Existing Step 15E TEST results untouched")


# ----------------------------------------------------------------------
# [9] COMPLETION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 16D COMPLETE")
print("=" * 70)

print("\nOBJECTS CREATED:")
print("  ✓ ablation_lgbm_models")
print("  ✓ ablation_lgbm_probabilities")
print("  ✓ ablation_lgbm_predictions")
print("  ✓ ablation_lgbm_results")
print("  ✓ ablation_lgbm_results_df")
print("  ✓ feature_ablation_effects")

print("\nPRIMARY SELECTION METRIC:")
print("  Validation PR-AUC")

print("\nTEST STATUS:")
print("  ✓ Completely untouched")

print("\nSTOP HERE.")
print("Send the COMPLETE output.")

print("=" * 70)

STEP 16D — LIGHTGBM FEATURE ABLATION EXPERIMENT

[1] REQUIRED OBJECT VERIFICATION
----------------------------------------------------------------------
✓ Ablation feature sets available
✓ Resampled TRAIN datasets available
✓ VALIDATION datasets available
✓ VALIDATION target available

[2] ABLATION EXPERIMENT VERIFICATION
----------------------------------------------------------------------
✓ Five expected experiments present

[3] LIGHTGBM CONFIGURATION
----------------------------------------------------------------------
objective             : binary
learning_rate         : 0.01
num_leaves            : 31
max_depth             : -1
min_child_samples     : 100
subsample             : 0.9
colsample_bytree      : 0.9
reg_alpha             : 0.0
reg_lambda            : 1.0
min_split_gain        : 0.0
n_estimators          : 3000
random_state          : 42
n_jobs                : -1
verbosity             : -1

[4] INITIALIZING RESULTS
----------------------------------------------------

C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration : 2424
Validation PR-AUC : 0.6411
Validation ROC-AUC : 0.8597
Validation F1 : 0.5546

EXPERIMENT: without_ordering_mode
Features             : ['culture_description', 'organism', 'antibiotic']
TRAIN shape          : (1317556, 3)
VALIDATION shape     : (308772, 3)
Categorical indices  : [0, 1, 2]


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration : 2671
Validation PR-AUC : 0.6350
Validation ROC-AUC : 0.8582
Validation F1 : 0.5625

EXPERIMENT: without_culture_description
Features             : ['ordering_mode', 'organism', 'antibiotic']
TRAIN shape          : (1317556, 3)
VALIDATION shape     : (308772, 3)
Categorical indices  : [0, 1, 2]


...

Best iteration : 2529
Validation PR-AUC : 0.6361
Validation ROC-AUC : 0.8570
Validation F1 : 0.5530

EXPERIMENT: without_organism
Features             : ['ordering_mode', 'culture_description', 'antibiotic']
TRAIN shape          : (1317556, 3)
VALIDATION shape     : (308772, 3)
Categorical indices  : [0, 1, 2]


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration : 829
Validation PR-AUC : 0.4283
Validation ROC-AUC : 0.7869
Validation F1 : 0.4694

EXPERIMENT: without_antibiotic
Features             : ['ordering_mode', 'culture_description', 'organism']
TRAIN shape          : (1317556, 3)
VALIDATION shape     : (308772, 3)
Categorical indices  : [0, 1, 2]


C:\Users\HPCLAB\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration : 434
Validation PR-AUC : 0.2748
Validation ROC-AUC : 0.6207
Validation F1 : 0.2835

[6] ABLATION COMPARISON
                 experiment  n_features                                                 features  best_iteration  accuracy  balanced_accuracy  precision   recall       f1  roc_auc   pr_auc
             all_4_features           4 ordering_mode, culture_description, organism, antibiotic            2424  0.804950           0.766261   0.456174 0.707337 0.554647 0.859718 0.641135
without_culture_description           3                      ordering_mode, organism, antibiotic            2529  0.806925           0.762819   0.458961 0.695643 0.553043 0.857024 0.636066
      without_ordering_mode           3                culture_description, organism, antibiotic            2671  0.823533           0.759008   0.489739 0.660732 0.562528 0.858160 0.635015
           without_organism           3           ordering_mode, culture_description, antibiotic             829  0.7871